# Cross-Market Transfer Learning for Stock Return Prediction

Can a model trained on U.S. stocks help predict next-month stock returns in the United Kingdom and Japan, especially
when little local training data is available? The data are monthly stock-level observations from the
Jensen–Kelly–Pedersen (JKP) Global Factor dataset.

**Version 2.** Sections 1–40 are Version 1, complete and frozen (Git tag `v1.0`). Sections 41–51 add the exploratory Version 2
experiments, which were completed on Amarel (tag `v2.0-experiment`). Sections 52–60 are a read-only audit of Version 2 with its
own output folder. All execution switches are off by default.

Phase 1 — Start

## 1. Data Loading

The three country files were downloaded from WRDS with the JKP stock screens applied. Only identifiers, screens, size,
the target and the realized return (`ret_1_0`) are loaded here; the remaining characteristics are read in batches when needed.

In [ ]:
import hashlib  # fingerprints of experiment settings
import json  # saved experiment configurations
import os  # settings passed by the Amarel job scripts
import sys  # detect the operating system
import time  # elapsed training time
import warnings  # record model convergence warnings
from importlib.metadata import version  # installed package versions
from itertools import combinations, product  # model pairs and hyperparameter grids
from pathlib import Path  # portable file paths

import joblib  # save trained models
import matplotlib.pyplot as plt  # result figures
import numpy as np  # numerical arrays
import pandas as pd  # tables, dates and Parquet/Excel reading
import pyarrow as pa  # fixed column types for saved predictions
import pyarrow.parquet as pq  # Parquet schemas without loading data
from xgboost import XGBRegressor  # boosted decision trees; must be imported before torch on macOS
import torch  # neural networks, automatic differentiation and GPU support
import torch.nn.functional as F  # efficient attention
from IPython.display import display  # formatted tables in the notebook
from sklearn.exceptions import ConvergenceWarning  # raised when Elastic Net does not converge
from sklearn.linear_model import ElasticNet, LinearRegression  # linear benchmark; nonnegative ensemble stacking
from torch import nn  # neural network layers

In [ ]:
# Execution settings: the only place that controls training
#   "off"           define models and functions without training
#   "local_train"   train one Transformer on real data, as set in LOCAL_RUN
#   "full_research" run the experiments listed in RESEARCH_EXPERIMENTS
RUN_MODE = "off"

LOCAL_RUN = {
    "country": "GBR",      # "GBR", "JPN", or "USA" for U.S. pretraining
    "window": "train_1y",  # "train_1y", "train_3y" or "train_5y"; use "train_full" for "USA"
    "seed": 42,
    "max_epochs": 5,       # at most 100
}

RESEARCH_EPOCHS = 100  # maximum Transformer epochs in full research; use 5 for a short laptop run

# Full research, in dependency order; delete entries to run only some of them
RESEARCH_EXPERIMENTS = [
    "elastic_net", "xgboost", "local", "pretrained", "direct", "finetuned", "joint", "localmatched",
]
# To run one experiment: RUN_MODE = "full_research" and, for example, RESEARCH_EXPERIMENTS = ["xgboost"]

RUN_PHASE6_PREDICTIONS = False  # True generates test predictions from the saved models
RUN_FINAL_EVALUATION = False    # True evaluates the saved test predictions

DEVICE_CHOICE = "auto"   # "auto" selects cuda, then mps (Apple GPU), then cpu
RUN_SMOKE_TESTS = False  # True runs small code checks in any mode; they never save models
RUN_PREFLIGHT = False    # True checks that the largest cross-section fits in device memory

# Version 2 (Sections 41-51). Version 1 is complete and frozen; its switches above must stay off.
RUN_V2 = False                # True runs the stages below for the variants below
V2_STAGES = ["train"]         # any of "train", "predict", "evaluate", "compare"
V2_VARIANTS = ["control"]     # variants listed in Section 41
V2_STRATEGIES = []            # empty = all; e.g. ["localmatched"] for a short laptop run (RESEARCH_EPOCHS = 2)

# Read-only audit of the completed Version 2 results (Sections 52-60)
RUN_AUDIT = False
AUDIT_STAGES = ["reports", "data", "validation", "portfolio"]  # "reports" needs saved tables only

In [ ]:
ALL_EXPERIMENTS = [
    "elastic_net", "xgboost", "local", "pretrained", "direct", "finetuned", "joint", "localmatched",
]

# Amarel job scripts choose a run with environment variables, so the settings above need no editing there
RUN_MODE = os.environ.get("CMP_RUN_MODE", RUN_MODE)
RESEARCH_EPOCHS = int(os.environ.get("CMP_EPOCHS", RESEARCH_EPOCHS))
RESEARCH_EXPERIMENTS = os.environ.get("CMP_EXPERIMENTS", ",".join(RESEARCH_EXPERIMENTS)).split(",")
DEVICE_CHOICE = os.environ.get("CMP_DEVICE", DEVICE_CHOICE)


def switch(name, default):
    """Read an on/off environment variable ("1" or "0")."""
    return {"1": True, "0": False}[os.environ[name]] if name in os.environ else default


RUN_PHASE6_PREDICTIONS = switch("CMP_PREDICTIONS", RUN_PHASE6_PREDICTIONS)
RUN_FINAL_EVALUATION = switch("CMP_EVALUATION", RUN_FINAL_EVALUATION)
RUN_SMOKE_TESTS = switch("CMP_SMOKE_TESTS", RUN_SMOKE_TESTS)
RUN_PREFLIGHT = switch("CMP_PREFLIGHT", RUN_PREFLIGHT)
RUN_V2 = switch("CMP_V2", RUN_V2)
V2_STAGES = [name for name in os.environ.get("CMP_V2_STAGES", ",".join(V2_STAGES)).split(",") if name]
V2_VARIANTS = [name for name in os.environ.get("CMP_V2_VARIANTS", ",".join(V2_VARIANTS)).split(",") if name]
V2_STRATEGIES = [name for name in os.environ.get("CMP_V2_STRATEGIES", ",".join(V2_STRATEGIES)).split(",") if name]
RUN_AUDIT = switch("CMP_AUDIT", RUN_AUDIT)
AUDIT_STAGES = [name for name in os.environ.get("CMP_AUDIT_STAGES", ",".join(AUDIT_STAGES)).split(",") if name]
if not set(AUDIT_STAGES) <= {"reports", "data", "validation", "portfolio"}:
    raise ValueError(f"Unknown AUDIT_STAGES: {AUDIT_STAGES}")

if RUN_MODE not in ["off", "local_train", "full_research"]:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")
if not 1 <= RESEARCH_EPOCHS <= 100:
    raise ValueError("RESEARCH_EPOCHS must be between 1 and 100.")
if not set(RESEARCH_EXPERIMENTS) <= set(ALL_EXPERIMENTS):
    raise ValueError(f"Unknown experiments: {set(RESEARCH_EXPERIMENTS) - set(ALL_EXPERIMENTS)}")


if RUN_MODE != "off" or RUN_PHASE6_PREDICTIONS or RUN_FINAL_EVALUATION:
    raise ValueError(
        "Version 1 is frozen in this branch, so its results cannot be changed. "
        "Use the Git tag v1.0 in a separate folder to run Version 1."
    )


def should_run(experiment):
    """True if the experiment is part of the current full-research run."""
    if RUN_MODE == "full_research" and experiment in RESEARCH_EXPERIMENTS:
        return True
    reason = f"RUN_MODE = '{RUN_MODE}'" if RUN_MODE != "full_research" else "not in RESEARCH_EXPERIMENTS"
    print(f"{experiment}: not run ({reason})")
    return False


if sys.platform == "darwin":
    torch.set_num_threads(1)  # avoids a macOS crash between the PyTorch and XGBoost thread libraries (CPU only)

print(f"Run mode: {RUN_MODE} | maximum epochs: {RESEARCH_EPOCHS} | predictions: {RUN_PHASE6_PREDICTIONS} | "
      f"evaluation: {RUN_FINAL_EVALUATION} | smoke tests: {RUN_SMOKE_TESTS} | preflight: {RUN_PREFLIGHT}")
print(f"Version 2: {RUN_V2} | stages: {V2_STAGES} | variants: {V2_VARIANTS} | strategies: {V2_STRATEGIES or 'all'}")
print(f"Audit: {RUN_AUDIT} | stages: {AUDIT_STAGES}")

In [223]:
pd.set_option("display.float_format", "{:,.4f}".format)

# Paths are relative to the project folder; the environment variables are optional overrides
PROJECT_ROOT = Path(os.environ.get("CMP_PROJECT_ROOT", "."))
DATA_DIR = Path(os.environ.get("CMP_DATA_DIR", PROJECT_ROOT / "data"))

COUNTRIES = ["USA", "GBR", "JPN"]
FILES = {country: DATA_DIR / f"{country}.parquet" for country in COUNTRIES}

TARGET = "ret_exc_lead1m"
SCREENS = ["common", "exch_main", "primary_sec", "obs_main"]
METADATA_COLS = ["id", "eom", "excntry", "gvkey", "permno", "size_grp", "me", TARGET] + SCREENS

for path in FILES.values():
    if not path.exists():
        raise FileNotFoundError(f"Missing dataset: {path}. See data/README.md.")

In [ ]:
print("Loading JKP datasets...")

data = {}
for country, path in FILES.items():
    df = pd.read_parquet(path, columns=METADATA_COLS + ["ret_1_0"])
    df["eom"] = pd.to_datetime(df["eom"])
    for column in ["excntry", "gvkey", "size_grp"]:
        df[column] = df[column].astype("category")  # saves memory
    data[country] = df.sort_values(["id", "eom"], ignore_index=True)

display(pd.concat([df.head(2) for df in data.values()], ignore_index=True).drop(columns=SCREENS))

## 2. Dataset Structure

The model inputs come from the 153 characteristics JKP use for published factors, listed in the `abr_jkp` field of the
[official metadata](https://jkpfactors.com/jkp-wrds-guide).

In [225]:
FACTOR_URL = "https://github.com/bkelly-lab/jkp-data/raw/main/src/jkp/data/resources/factor_details.xlsx"

schemas = {country: pq.read_schema(path) for country, path in FILES.items()}
try:
    characteristics = pd.read_excel(FACTOR_URL)["abr_jkp"].dropna().drop_duplicates().tolist()
except OSError as error:  # includes network errors and timeouts
    characteristics = [c for c in schemas["USA"].names if c not in METADATA_COLS]
    print(f"WARNING: official metadata unavailable ({error}); characteristic names are provisional.")

if len(characteristics) != 153:
    raise ValueError(f"Expected 153 characteristics, found {len(characteristics)}.")
for country, schema in schemas.items():
    missing = [c for c in METADATA_COLS + characteristics if c not in schema.names]
    if missing:
        raise ValueError(f"{country}: missing columns {missing}")
    if not schema.equals(schemas["USA"]):
        raise ValueError(f"{country}: column names or types differ from USA.")

In [226]:
structure = pd.DataFrame({
    country: {
        "Observations": f"{len(df):,}", "Unique stocks": f"{df['id'].nunique():,}",
        "Columns": len(schemas[country].names),
        "First month": df["eom"].min().strftime("%Y-%m"), "Last month": df["eom"].max().strftime("%Y-%m"),
    } for country, df in data.items()
}).T
display(structure)
print(f"All {len(characteristics)} characteristics and {len(METADATA_COLS)} other columns are present, with identical schemas.")

,Observations,Unique stocks,Columns,First month,Last month
USA,"1,604,094","18,733",165,2000-01,2025-12
GBR,"534,295","5,182",165,2000-01,2025-12
JPN,"1,175,157","6,299",165,2000-01,2025-12


All 153 characteristics and 12 other columns are present, with identical schemas.


The proposal mentions 174 retained columns; the files contain 165, which include every field the proposal names.

## 3. Data Integrity Checks

The characteristics are scanned once, in batches, for missing and infinite values. All counts in the table should be zero.

In [227]:
print("Checking dataset integrity...")

missing_pct = pd.DataFrame(index=characteristics, columns=COUNTRIES, dtype=float)
infinite_values = dict.fromkeys(COUNTRIES, 0)

for country, path in FILES.items():
    for start in range(0, len(characteristics), 20):
        batch = pd.read_parquet(path, columns=characteristics[start:start + 20])
        missing_pct.loc[batch.columns, country] = 100 * batch.isna().mean()
        infinite_values[country] += int(np.isinf(batch.to_numpy()).sum())
    infinite_values[country] += int(np.isinf(data[country][["me", TARGET]].to_numpy()).sum())

Checking dataset integrity...


In [228]:
calendar = pd.date_range("2000-01-31", max(df["eom"].max() for df in data.values()), freq="ME")

integrity = pd.DataFrame({
    country: {
        "Wrong country code": (df["excntry"] != country).sum(),
        "Missing id or eom": df[["id", "eom"]].isna().any(axis=1).sum(),
        "Duplicate keys": df.duplicated(["excntry", "id", "eom"]).sum(),
        "Screen violations": (df[SCREENS] != 1).any(axis=1).sum(),
        "Missing months": len(calendar.difference(df["eom"])),
        "Infinite values": infinite_values[country],
    } for country, df in data.items()
})
display(integrity)

if integrity.to_numpy().sum() > 0:
    raise ValueError("Integrity check failed; see the nonzero counts above.")
print("All integrity checks passed.")

,USA,GBR,JPN
Wrong country code,0,0,0
Missing id or eom,0,0,0
Duplicate keys,0,0,0
Screen violations,0,0,0
Missing months,0,0,0
Infinite values,0,0,0


All integrity checks passed.


The target should equal the stock's realized return in the next month, `ret_1_0`, minus the T-bill rate. The share of
consecutive-month pairs agreeing within one percentage point checks this timing within the JKP data.

In [229]:
for country, df in data.items():
    by_stock = df.groupby("id")
    next_return = by_stock["ret_1_0"].shift(-1)
    consecutive = by_stock["eom"].shift(-1) == df["eom"] + pd.offsets.MonthEnd(1)
    difference = (next_return - df[TARGET])[consecutive].dropna()
    agreement = (difference.abs() <= 0.01).mean()
    print(f"{country}: {agreement:.2%} of {len(difference):,} targets match the next month's return")
    if agreement < 0.99:
        raise ValueError(f"{country}: the target is not aligned with the following month.")

USA: 100.00% of 1,545,608 targets match the next month's return
GBR: 100.00% of 523,283 targets match the next month's return
JPN: 100.00% of 1,168,162 targets match the next month's return


## 4. Missing Values and Target Availability

Values are reported as they are; nothing is filled, clipped or removed. Returns are in decimal units (1.0 = 100%).

In [230]:
print("Checking missing values...")

missing_summary = pd.DataFrame({
    country: {
        "Missing target (%)": 100 * df[TARGET].isna().mean(),
        "Missing size group (%)": 100 * df["size_grp"].isna().mean(),
        "Mean predictor missing (%)": missing_pct[country].mean(),
        "Median predictor missing (%)": missing_pct[country].median(),
        "Predictors over 50% missing": (missing_pct[country] > 50).sum(),
    } for country, df in data.items()
})
display(missing_summary)

print("Most incomplete characteristics (% missing, ranked by the three-country average):")
most_missing = missing_pct.mean(axis=1).nlargest(10).index
display(missing_pct.loc[most_missing])

Checking missing values...


,USA,GBR,JPN
Missing target (%),3.5689,1.8346,0.2596
Missing size group (%),0.4035,0.4469,0.2017
Mean predictor missing (%),14.7485,31.7971,26.1489
Median predictor missing (%),8.8561,29.4992,19.6708
Predictors over 50% missing,9.0000,20.0000,17.0000


Most incomplete characteristics (% missing, ranked by the three-country average):


,USA,GBR,JPN
seas_16_20na,72.8430,86.8644,64.4921
seas_16_20an,72.6464,85.4513,64.0935
rd5_at,61.4115,84.7575,62.7640
seas_11_15na,63.1121,76.4520,46.8341
seas_11_15an,62.9113,73.8515,46.3072
saleq_su,17.2693,64.6815,99.3574
rd_sale,53.0261,75.2195,51.7802
rd_me,50.3900,74.4259,51.7704
ocfq_saleq_std,19.5343,57.5884,99.3533
ni_inc8q,19.0711,48.3154,99.2096


In [231]:
print("Target distribution (decimal returns):")
display(pd.DataFrame({
    country: df[TARGET].describe(percentiles=[0.01, 0.5, 0.99, 0.999]) for country, df in data.items()
}))

print("Available targets near the dataset endpoint:")
endpoint = pd.DataFrame({
    country: df[df["eom"] >= "2025-09-30"].groupby("eom")[TARGET].count() for country, df in data.items()
})
endpoint.insert(0, "predicted month", (endpoint.index + pd.offsets.MonthEnd(1)).strftime("%Y-%m"))
display(endpoint)

Target distribution (decimal returns):


,USA,GBR,JPN
count,"1,546,846.0000","524,493.0000","1,172,106.0000"
mean,0.0253,0.0045,0.0066
std,13.3882,0.6662,0.1361
min,-1.0050,-1.0022,-0.9993
1%,-0.4670,-0.4440,-0.2721
50%,-0.0001,-0.0061,-0.0019
99%,0.6393,0.6303,0.4244
99.9%,1.8455,1.8495,1.1178
max,"15,348.9993",331.0520,11.9461


Available targets near the dataset endpoint:


,predicted month,USA,GBR,JPN
eom,,,,
2025-09-30,2025-10,4808,1185,3944
2025-10-31,2025-11,4806,1185,3944
2025-11-30,2025-12,4813,1187,3941
2025-12-31,2026-01,4,1181,3942


Some targets are extremely large, and squared-error models are sensitive to them. The data remain unchanged here;
Version 1 kept them, and Section 43 tests robust alternatives. The U.S. has almost no January 2026 targets, so the common test
period ends in December 2025.

## 5. Modeling Data Requirements

The target `ret_exc_lead1m` is the following month's USD stock return in excess of the U.S. T-bill rate. Periods are
defined by the **predicted-return month**, which is one month after the observation month `eom`.

| Period | Observation months (`eom`) | Predicted-return months |
|---|---|---|
| U.S. training | Jan 2000 – Nov 2017 | Feb 2000 – Dec 2017 |
| Local training (1, 3, 5 years) | from Dec 2016, Dec 2014, Dec 2012 – Nov 2017 | from Jan 2017, Jan 2015, Jan 2013 – Dec 2017 |
| Validation | Dec 2017 – Nov 2019 | Jan 2018 – Dec 2019 |
| Testing | Dec 2019 – Nov 2025 | Jan 2020 – Dec 2025 (72 months) |

Phase 1 — End

Phase 2 — Start

## 6. Temporal Data Splitting

Periods are defined by the predicted-return month, which is one month after `eom`. Local training windows of 1, 3 and
5 years all end in December 2017. Observations are assigned by date only, never at random.

In [232]:
# First and last predicted-return month of each period
PERIODS = {
    "train_full": ("2000-02", "2017-12"),
    "train_5y": ("2013-01", "2017-12"),
    "train_3y": ("2015-01", "2017-12"),
    "train_1y": ("2017-01", "2017-12"),
    "validation": ("2018-01", "2019-12"),
    "test": ("2020-01", "2025-12"),
}
PERIODS = {
    name: (pd.Timestamp(first) + pd.offsets.MonthEnd(0), pd.Timestamp(last) + pd.offsets.MonthEnd(0))
    for name, (first, last) in PERIODS.items()
}
TRAIN_WINDOWS = {
    "USA": ["train_full"],
    "GBR": ["train_1y", "train_3y", "train_5y"],
    "JPN": ["train_1y", "train_3y", "train_5y"],
}

for df in data.values():
    df["predicted_month"] = df["eom"] + pd.offsets.MonthEnd(1)


def in_period(predicted_month, period):
    """True where the predicted-return month belongs to the period."""
    return predicted_month.between(*PERIODS[period])

In [233]:
print("Preparing chronological data splits...")

rows = []
for country, df in data.items():
    for period in TRAIN_WINDOWS[country] + ["validation", "test"]:
        part = df[in_period(df["predicted_month"], period)]
        rows.append({
            "Country": country, "Period": period, "Observations": len(part),
            "Labeled": part[TARGET].notna().sum(), "Months": part["eom"].nunique(),
            "First eom": part["eom"].min(), "Last eom": part["eom"].max(),
        })
split_summary = pd.DataFrame(rows)
display(split_summary)

Preparing chronological data splits...


,Country,Period,Observations,Labeled,Months,First eom,Last eom
0,USA,train_full,1129349,1091387,215,2000-01-31,2017-11-30
1,USA,validation,106502,101787,24,2017-12-31,2019-11-30
2,USA,test,363367,353668,72,2019-12-31,2025-11-30
3,GBR,train_1y,18606,18239,12,2016-12-31,2017-11-30
4,GBR,train_3y,57465,56515,36,2014-12-31,2017-11-30
5,GBR,train_5y,96737,95227,60,2012-12-31,2017-11-30
6,GBR,validation,36808,36378,24,2017-12-31,2019-11-30
7,GBR,test,100297,99377,72,2019-12-31,2025-11-30
8,JPN,train_1y,44705,44650,12,2016-12-31,2017-11-30
9,JPN,train_3y,132899,132686,36,2014-12-31,2017-11-30


In [234]:
expected_months = {name: len(pd.date_range(*dates, freq="ME")) for name, dates in PERIODS.items()}
assert (split_summary["Months"] == split_summary["Period"].map(expected_months)).all(), "A period has missing months."

# Training ends before validation, and validation ends before testing
train_names = [name for name in PERIODS if name.startswith("train")]
assert {PERIODS[name][1] for name in train_names} == {pd.Timestamp("2017-12-31")}
assert PERIODS["train_full"][1] < PERIODS["validation"][0]
assert PERIODS["validation"][1] < PERIODS["test"][0]
print("All periods are complete and do not overlap; all training windows end in December 2017.")

All periods are complete and do not overlap; all training windows end in December 2017.


## 7. Predictor Selection

One common predictor set is used by all models. A characteristic is kept if, in **each** of the seven training windows
(U.S. full history; UK and Japan 1, 3 and 5 years), it is available for at least 50% of observations and takes at least
two distinct values. The denominator is the number of **labeled** training observations, because only these enter
supervised training. Infinite values count as unavailable.

In [235]:
MIN_COVERAGE = 0.50


def read_observations(country, period, columns):
    """Read selected columns for one period, ordered by month and stock."""
    first, last = [date - pd.offsets.MonthEnd(1) for date in PERIODS[period]]  # predicted month to eom
    return read_months(country, first, last, columns)


def read_months(country, first_eom, last_eom, columns):
    """Read selected columns for a range of observation months, ordered by month and stock."""
    dates = [("eom", ">=", first_eom.date()), ("eom", "<=", last_eom.date())]
    df = pd.read_parquet(FILES[country], columns=["id", "eom"] + columns, filters=dates)
    df["eom"] = pd.to_datetime(df["eom"])
    return df.sort_values(["eom", "id"], ignore_index=True)

In [236]:
print("Measuring training coverage and variation...")

coverage, variation, selection_last_eom = {}, {}, {}
for country, windows in TRAIN_WINDOWS.items():
    longest = windows[-1]  # the shorter windows are subsets of this one
    for start in range(0, len(characteristics), 20):
        names = characteristics[start:start + 20]
        df = read_observations(country, longest, [TARGET] + names)
        df = df[df[TARGET].notna()]
        selection_last_eom[country] = df["eom"].max()
        for window in windows:
            part = df.loc[in_period(df["eom"] + pd.offsets.MonthEnd(1), window), names]
            part = part.where(np.isfinite(part))
            for name in names:
                coverage.setdefault(f"{country} {window}", {})[name] = part[name].notna().mean()
                variation.setdefault(f"{country} {window}", {})[name] = part[name].min() < part[name].max()

coverage = pd.DataFrame(coverage).loc[characteristics]
variation = pd.DataFrame(variation).loc[characteristics]

Measuring training coverage and variation...


In [237]:
enough_coverage = (coverage >= MIN_COVERAGE).all(axis=1)
enough_variation = variation.all(axis=1)

# A characteristic failing both rules is counted once, under coverage
feature_table = coverage.assign(status="selected")
feature_table.loc[~enough_variation, "status"] = "insufficient variation"
feature_table.loc[~enough_coverage, "status"] = "insufficient coverage"

selected_features = feature_table.index[feature_table["status"] == "selected"].tolist()
assert not set(selected_features) & set(METADATA_COLS), "Metadata columns cannot be predictors."

print(f"Initial characteristics:          {len(characteristics)}")
print(f"Removed, insufficient coverage:   {(feature_table['status'] == 'insufficient coverage').sum()}")
print(f"Removed, insufficient variation:  {(feature_table['status'] == 'insufficient variation').sum()}")
print(f"Selected predictors:              {len(selected_features)}")
print()
for start in range(0, len(selected_features), 10):
    print(", ".join(selected_features[start:start + 10]))

Initial characteristics:          153
Removed, insufficient coverage:   27
Removed, insufficient variation:  0
Selected predictors:              126

ret_6_1, ret_12_1, tax_gr1a, prc_highprc_252d, resff3_6_1, resff3_12_1, be_me, debt_me, at_me, ret_60_12
ni_me, fcf_me, sale_gr3, sale_gr1, ebitda_mev, sale_me, ocf_me, bev_mev, netdebt_me, eq_dur
capex_abn, at_gr1, ppeinv_gr1a, noa_at, noa_gr1a, lnoa_gr1a, capx_gr1, capx_gr2, capx_gr3, chcsho_12m
eqnpo_12m, debt_gr3, inv_gr1, inv_gr1a, oaccruals_at, taccruals_at, cowc_gr1a, coa_gr1a, col_gr1a, nncoa_gr1a
ncoa_gr1a, ncol_gr1a, nfna_gr1a, sti_gr1a, lti_gr1a, fnl_gr1a, be_gr1a, oaccruals_ni, taccruals_ni, netis_at
eqnetis_at, dbnetis_at, ebit_bev, ebit_sale, sale_bev, at_turnover, gp_at, gp_atl1, ope_be, ope_bel1
op_at, op_atl1, cop_at, cop_atl1, f_score, o_score, z_score, pi_nix, at_be, opex_at
emp_gr1, age, dsale_dinv, dsale_drec, dgp_dsale, dsale_dsga, sale_emp_gr1, tangibility, cash_at, ni_ar1
ni_ivol, earnings_variability, aliq_at, ali

In [238]:
excluded = feature_table[feature_table["status"] != "selected"].copy()
excluded.insert(0, "lowest coverage", excluded.drop(columns="status").min(axis=1))
excluded.insert(1, "lowest in", excluded.drop(columns=["status", "lowest coverage"]).idxmin(axis=1))
print("Excluded characteristics (coverage as a share of labeled training observations):")
display(excluded[["status", "lowest coverage", "lowest in"]].sort_values("lowest coverage"))

Excluded characteristics (coverage as a share of labeled training observations):


,status,lowest coverage,lowest in
ivol_hxz4_21d,insufficient coverage,0.0000,JPN train_1y
iskew_hxz4_21d,insufficient coverage,0.0000,JPN train_1y
ocfq_saleq_std,insufficient coverage,0.0081,JPN train_1y
saleq_su,insufficient coverage,0.0083,JPN train_3y
saleq_gr1,insufficient coverage,0.0092,JPN train_3y
ni_inc8q,insufficient coverage,0.0110,JPN train_1y
niq_su,insufficient coverage,0.0116,JPN train_5y
niq_be_chg1,insufficient coverage,0.0121,JPN train_1y
niq_at_chg1,insufficient coverage,0.0122,JPN train_1y
niq_be,insufficient coverage,0.0130,JPN train_5y


## 8. Cross-Sectional Normalization

Within each country-month, every predictor is ranked across all screened stocks, including stocks without a return
label, and mapped to [−0.5, 0.5] by `(rank − 1) / (n − 1) − 0.5`. Ties receive average ranks. A single available
observation receives zero, and missing values stay missing until the next section.

In [239]:
def rank_within_month(features, months):
    """Rank each column within each month and map the ranks to [-0.5, 0.5]."""
    values = features.where(np.isfinite(features))
    by_month = values.groupby(months)
    ranks = by_month.rank(method="average")
    n = by_month.transform("count")
    scaled = (ranks - 1) / (n - 1) - 0.5
    return scaled.mask((n == 1) & values.notna(), 0.0)

In [ ]:
example_month = pd.Timestamp("2017-06-30")
rows = []
for country in COUNTRIES:
    raw = read_months(country, example_month, example_month, selected_features)
    ranked = rank_within_month(raw[selected_features], raw["eom"])

    # Show the smallest, middle and largest value, and one missing value, of one incomplete predictor
    feature = raw[selected_features].isna().mean().idxmax()
    order = raw[feature].dropna().sort_values().index
    picks = [order[0], order[len(order) // 2], order[-1], raw.index[raw[feature].isna()][0]]
    for i in picks:
        rows.append({
            "Country": country, "id": raw.loc[i, "id"], "Characteristic": feature,
            "Original": raw.loc[i, feature], "Ranked": ranked.loc[i, feature],
        })
print(f"Ranking example, {example_month:%B %Y}:")
display(pd.DataFrame(rows))

## 9. Missing Data Handling

Missing ranked predictors receive zero, the neutral point of the scale. Missing returns are never filled; a label mask
marks the stocks that can be used in losses and metrics. Finite target values are kept unchanged.

In [241]:
feature_types = {str(schemas["USA"].field(name).type) for name in selected_features}
print(f"Data types of the selected predictors: {feature_types}")
assert feature_types == {"double"}, "A selected predictor is not numeric and needs categorical handling."


def prepare_features(features, months):
    """Rank within month, then replace missing ranks with the neutral value zero."""
    return rank_within_month(features, months).fillna(0.0).astype("float32")

Data types of the selected predictors: {'double'}


All selected predictors are numeric, so no categorical treatment is needed.

## 10. Model Input Preparation

`prepare_data` returns two aligned objects for one country and one period: `info` (identifiers, months, target and
label mask) and `X` (a `float32` matrix with columns ordered as `selected_features`). Data are read one year at a time,
so raw characteristics for a full history are never in memory together. `month_labels`, the number of labeled stocks in
the month, allows equal monthly weighting during training.

In [ ]:
def prepare_data(country, period, features, missing_flags=False):
    """Prepare one country and period; returns (info, X) ordered by month and stock."""
    first, last = [date - pd.offsets.MonthEnd(1) for date in PERIODS[period]]  # predicted month to eom
    return prepare_months(country, first, last, features, missing_flags)


def prepare_months(country, first_eom, last_eom, features, missing_flags=False):
    """Prepare a range of observation months; returns (info, X) ordered by month and stock.

    With `missing_flags`, one indicator per feature (1 = value missing) follows the ranked features.
    """
    n_rows = data[country]["eom"].between(first_eom, last_eom).sum()
    n_features = len(features)
    X = np.empty((n_rows, n_features * (2 if missing_flags else 1)), dtype="float32")
    info, position = [], 0

    for year in range(first_eom.year, last_eom.year + 1):
        start = max(first_eom, pd.Timestamp(year, 1, 31))
        end = min(last_eom, pd.Timestamp(year, 12, 31))
        df = read_months(country, start, end, [TARGET] + features)
        rows = slice(position, position + len(df))
        X[rows, :n_features] = prepare_features(df[features], df["eom"]).to_numpy()
        if missing_flags:
            X[rows, n_features:] = ~np.isfinite(df[features].to_numpy())
        position += len(df)
        info.append(df[["id", "eom", TARGET]])

    assert position == n_rows, "Prepared rows do not match the dataset."
    info = pd.concat(info, ignore_index=True)
    info.insert(0, "country", country)
    info["predicted_month"] = info["eom"] + pd.offsets.MonthEnd(1)
    info["has_label"] = np.isfinite(info[TARGET])
    info["month_labels"] = info.groupby("eom")["has_label"].transform("sum")
    return info, X


def get_cross_section(info, X, eom):
    """Return all stocks of one observation month."""
    rows = (info["eom"] == eom).to_numpy()
    return info[rows], X[rows]

In [ ]:
print("Preparing example cross-sections...")

EXAMPLE_MONTHS = {"training": "2017-11-30", "validation": "2018-12-31", "testing": "2025-11-30"}
examples, rows = {}, []
for country in COUNTRIES:
    for period, eom in EXAMPLE_MONTHS.items():
        eom = pd.Timestamp(eom)
        info, X = prepare_months(country, eom - pd.offsets.MonthEnd(1), eom, selected_features)  # two months
        examples[country, period] = (info, X)
        month_info, month_X = get_cross_section(info, X, eom)
        rows.append({
            "Country": country, "Period": period, "eom": eom, "Stocks": len(month_info),
            "Predictors": month_X.shape[1], "Labels": month_info["has_label"].sum(),
            "Missing labels": (~month_info["has_label"]).sum(), "X shape": month_X.shape,
        })
display(pd.DataFrame(rows))
display(info.head(3))

## 11. Preprocessing Verification

The checks below use the prepared examples and a small artificial table for ranking edge cases.

In [244]:
# Selection used training months only
assert all(last == pd.Timestamp("2017-11-30") for last in selection_last_eom.values())

# Ranking rules: ties, missing and infinite values, single observation, empty month, independent months
test_months = pd.Series([1, 1, 1, 1, 2, 2, 3])
test_values = pd.DataFrame({"x": [5.0, 1.0, 5.0, np.inf, 9.0, np.nan, np.nan]})
expected = pd.DataFrame({"x": [0.25, -0.5, 0.25, np.nan, 0.0, np.nan, np.nan]})
pd.testing.assert_frame_equal(rank_within_month(test_values, test_months), expected)
assert (prepare_features(test_values, test_months)["x"].iloc[[3, 5, 6]] == 0).all()
print("Selection dates and ranking rules verified.")

Selection dates and ranking rules verified.


In [245]:
for (country, period), (info, X) in examples.items():
    raw = read_months(country, info["eom"].min(), info["eom"].max(), [TARGET] + selected_features)
    original = data[country].set_index(["id", "eom"])[TARGET].loc[pd.MultiIndex.from_frame(info[["id", "eom"]])]

    assert X.dtype == "float32" and X.shape == (len(info), len(selected_features))
    assert np.isfinite(X).all() and X.min() >= -0.5 and X.max() <= 0.5
    assert (X[raw[selected_features].isna().to_numpy()] == 0).all()  # missing predictors are zero
    assert raw[["id", "eom"]].equals(info[["id", "eom"]])  # identifiers preserved
    np.testing.assert_array_equal(info[TARGET], original)  # targets unchanged, missing stays missing
    assert (info["has_label"] == info[TARGET].notna()).all()
    assert (info["predicted_month"] == info["eom"] + pd.offsets.MonthEnd(1)).all()

assert all(schema.equals(schemas["USA"]) for schema in (pq.read_schema(path) for path in FILES.values()))
assert all(pq.read_metadata(FILES[c]).num_rows == len(data[c]) for c in COUNTRIES)
print(f"Verified {len(examples)} prepared samples: predictors, identifiers, targets and label masks are correct.")
print("Raw Parquet files keep the same schema and row counts.")

Verified 9 prepared samples: predictors, identifiers, targets and label masks are correct.
Raw Parquet files keep the same schema and row counts.


**Return persistence.** Pooled correlation between a stock's excess returns in consecutive months, using only returns
through December 2017.

In [246]:
rows = []
for country, df in data.items():
    by_stock = df.groupby("id")  # data are sorted by id and eom
    pairs = pd.DataFrame({"previous": by_stock[TARGET].shift(1), "current": df[TARGET]})
    consecutive = df["eom"] == by_stock["eom"].shift(1) + pd.offsets.MonthEnd(1)
    pairs = pairs[consecutive & in_period(df["predicted_month"], "train_full")].dropna()
    rows.append({
        "Country": country, "Pairs": len(pairs), "Pearson": pairs.corr().iloc[0, 1],
        "Spearman": pairs.rank().corr().iloc[0, 1],  # correlation of ranks
    })
display(pd.DataFrame(rows))

,Country,Pairs,Pearson,Spearman
0,USA,1078361,-0.0001,0.0094
1,GBR,379562,0.0041,0.0723
2,JPN,788268,0.0081,0.0029


Phase 2 — End

Phase 3 — Start

## 12. Baseline Model Configuration

Elastic Net and XGBoost predict a stock's next-month excess return from its own characteristics. Each is trained
separately for the UK and Japan on 1, 3 and 5 years of local data, giving six configurations per model family.

Training is controlled by `RUN_MODE` at the top of the notebook. The optional smoke test only checks that the code works
on a few months; its numbers are not research results.

In [247]:
SEED = 42
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))  # CPU threads for XGBoost
MAX_SETTINGS = 20  # proposal limit per model family and configuration

LOCAL_COUNTRIES = ["GBR", "JPN"]
# Shortened runs have their own folder, so they are never mistaken for the 100-epoch results
RESULTS_DIR = PROJECT_ROOT / ("results" if RESEARCH_EPOCHS == 100 else f"results_{RESEARCH_EPOCHS}epochs")
MODELS_DIR = RESULTS_DIR / "models"
VALIDATION_FILE = RESULTS_DIR / "baseline_validation.csv"


def save_safely(path, save):
    """Write to a temporary file first, so an interrupted job never leaves a partial file."""
    temporary = path.with_name(path.name + ".tmp")
    save(temporary)
    temporary.replace(path)

Every training month has the same total weight: a stock's weight is proportional to one over the number of labeled
stocks in its month, scaled to average one. Validation MSE is the average of the monthly MSE values.

In [248]:
def keep_labeled(info, X):
    """Keep observations with an available return; they are the only ones used in losses."""
    labeled = info["has_label"].to_numpy()
    return {"info": info[labeled].reset_index(drop=True), "X": X[labeled], "y": info.loc[labeled, TARGET].to_numpy()}


def load_labeled(country, period):
    """Load labeled training or validation observations."""
    if period == "test":
        raise ValueError("Test data cannot be used for training or model selection.")
    return keep_labeled(*prepare_data(country, period, selected_features))


def month_weights(info):
    """Weights that give each month equal total weight and average one."""
    month_size = info.groupby("eom")["id"].transform("size").to_numpy()
    return len(info) / (info["eom"].nunique() * month_size)


def monthly_mse(info, y, prediction):
    """Average of monthly mean squared errors."""
    squared_error = pd.Series((y - prediction) ** 2)
    return squared_error.groupby(info["eom"].to_numpy()).mean().mean()

In [249]:
def fit_model(model, train):
    """Fit with monthly-equal weights; returns True if no convergence warning was raised."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(train["X"], train["y"], sample_weight=month_weights(train["info"]))
    for warning in caught:
        if not issubclass(warning.category, ConvergenceWarning):
            warnings.warn_explicit(warning.message, warning.category, warning.filename, warning.lineno)
    return not any(issubclass(warning.category, ConvergenceWarning) for warning in caught)


def predict_model(model, sample):
    """Predictions aligned with stock identifiers and months."""
    predictions = sample["info"][["country", "id", "eom", "predicted_month"]].copy()
    predictions["prediction"] = model.predict(sample["X"])
    return predictions

In [250]:
if RUN_SMOKE_TESTS:
    # Last three UK training months, and the first three validation months to check the evaluation code
    month_end = pd.offsets.MonthEnd
    train_end, validation_end = [PERIODS[p][1] - month_end(1) for p in ("train_1y", "validation")]
    validation_start = PERIODS["validation"][0] - month_end(1)
    smoke_train = keep_labeled(*prepare_months("GBR", train_end - month_end(2), train_end, selected_features))
    smoke_valid = keep_labeled(
        *prepare_months("GBR", validation_start, validation_start + month_end(2), selected_features)
    )
    print(f"Smoke-test training sample:   {smoke_train['X'].shape}")
    print(f"Smoke-test validation sample: {smoke_valid['X'].shape}")

## 13. Elastic Net Regression

Squared error with L1 and L2 coefficient penalties. `alpha` sets the overall penalty and `l1_ratio` the share of L1.
The intercept is not penalized. Predictors already share a common scale, so no further scaling is applied.

In [251]:
ELASTIC_NET_GRID = [
    {"alpha": alpha, "l1_ratio": l1_ratio}
    for alpha, l1_ratio in product([1e-5, 1e-4, 3e-4, 1e-3, 1e-2], [0.1, 0.4, 0.7, 0.95])
]


def make_elastic_net(params):
    # precompute=True uses the 126 x 126 Gram matrix, which is faster for many rows
    return ElasticNet(**params, precompute=True, max_iter=100_000, tol=1e-4, random_state=SEED)


print(f"Elastic Net settings: {len(ELASTIC_NET_GRID)}")

Elastic Net settings: 20


In [252]:
if RUN_SMOKE_TESTS:
    smoke_models = {"Elastic Net": make_elastic_net({"alpha": 1e-3, "l1_ratio": 0.5})}
    converged = fit_model(smoke_models["Elastic Net"], smoke_train)
    print(f"Elastic Net smoke test: converged = {converged}, "
          f"nonzero coefficients = {(smoke_models['Elastic Net'].coef_ != 0).sum()}")

## 14. XGBoost Regression

Boosted trees with squared-error loss. Tree depth, learning rate and number of trees are set by an explicit grid, so
the number of trees is chosen by the same monthly-equal validation MSE as every other setting.

In [253]:
XGBOOST_GRID = [
    {"max_depth": depth, "learning_rate": rate, "n_estimators": trees}
    for depth, rate, trees in product([2, 4, 6], [0.01, 0.03, 0.1], [100, 300])
]


def make_xgboost(params):
    return XGBRegressor(**params, objective="reg:squarederror", tree_method="hist", n_jobs=N_JOBS, random_state=SEED)


print(f"XGBoost settings: {len(XGBOOST_GRID)}")

XGBoost settings: 18


In [254]:
if RUN_SMOKE_TESTS:
    smoke_models["XGBoost"] = make_xgboost({"max_depth": 2, "learning_rate": 0.1, "n_estimators": 20})
    fit_model(smoke_models["XGBoost"], smoke_train)

    rows = []
    for family, model in smoke_models.items():
        prediction = predict_model(model, smoke_valid)["prediction"].to_numpy()
        rows.append({
            "Model": family, "Training shape": smoke_train["X"].shape, "Prediction shape": prediction.shape,
            "All finite": np.isfinite(prediction).all(),
            "Monthly MSE (code check only)": monthly_mse(smoke_valid["info"], smoke_valid["y"], prediction),
        })
    display(pd.DataFrame(rows))

## 15. Validation and Model Selection

For each country, training history and model family, every setting is fitted on the local training window and scored on
the 2018–2019 validation period. The setting with the lowest monthly-equal MSE is kept, and is not refitted on
validation data. The zero-forecast MSE and the largest absolute prediction are recorded to reveal distortion from
extreme returns. A prediction standard deviation of zero means every stock received the same forecast.

In [255]:
MODEL_FAMILIES = {
    "Elastic Net": (make_elastic_net, ELASTIC_NET_GRID),
    "XGBoost": (make_xgboost, XGBOOST_GRID),
}


def search(family, train, valid):
    """Evaluate every setting; returns the best model and a table of all settings."""
    make_model, grid = MODEL_FAMILIES[family]
    best_model, best_mse, trials = None, np.inf, []
    started = time.time()
    for number, params in enumerate(grid, start=1):
        model = make_model(params)
        converged = fit_model(model, train)
        prediction = model.predict(valid["X"])
        mse = monthly_mse(valid["info"], valid["y"], prediction)
        trials.append({
            **params, "validation_mse": mse, "max_abs_prediction": np.abs(prediction).max(),
            "prediction_std": prediction.std(), "converged": converged,
        })
        if np.isfinite(mse) and mse < best_mse:
            best_model, best_mse = model, mse
        print(f"  setting {number}/{len(grid)} {params} | validation {mse:.6f} | best {best_mse:.6f} | "
              f"{time.time() - started:.0f}s", flush=True)
    if best_model is None:
        raise ValueError(f"{family}: no setting produced a finite validation MSE.")

    trials = pd.DataFrame(trials)
    trials["selected"] = trials.index == trials["validation_mse"].idxmin()
    return best_model, trials

In [256]:
BASELINE_EXPERIMENTS = {"elastic_net": "Elastic Net", "xgboost": "XGBoost"}


def baseline_path(country, window, experiment):
    return MODELS_DIR / f"{country}_{window}_{experiment}.joblib"


def baseline_complete(country, window, experiment):
    """True if the saved model uses the current predictors and its results are recorded."""
    path = baseline_path(country, window, experiment)
    if not (path.exists() and VALIDATION_FILE.exists()):
        return False
    saved = pd.read_csv(VALIDATION_FILE)
    family = BASELINE_EXPERIMENTS[experiment]
    recorded = ((saved["country"] == country) & (saved["window"] == window) & (saved["family"] == family)).any()
    if not recorded or joblib.load(path)["features"] != selected_features:
        raise ValueError(f"{path.name} exists but is incomplete or outdated. Move it before rerunning.")
    return True


def run_baseline(experiment, country, window, train, valid):
    """Search, save and record one Elastic Net or XGBoost configuration."""
    family = BASELINE_EXPERIMENTS[experiment]
    print(f"\n{family} | {country} | {window} | cpu")
    print(f"  Training: {train['info']['eom'].nunique()} months, {train['info']['id'].nunique():,} stocks, "
          f"{len(train['y']):,} labeled rows | Validation: {valid['info']['eom'].nunique()} months | "
          f"{len(selected_features)} features")
    model, trials = search(family, train, valid)
    trials = trials.assign(
        country=country, window=window, family=family, train_rows=len(train["y"]), validation_rows=len(valid["y"]),
        predictors=len(selected_features), settings=len(trials),
        zero_mse=monthly_mse(valid["info"], valid["y"], 0.0),
    )
    if VALIDATION_FILE.exists():
        saved = pd.read_csv(VALIDATION_FILE)
        same = (saved["country"] == country) & (saved["window"] == window) & (saved["family"] == family)
        trials = pd.concat([saved[~same], trials], ignore_index=True)
    save_safely(VALIDATION_FILE, lambda path: trials.to_csv(path, index=False))  # results first, model last
    saved_model = {"model": model, "features": selected_features}
    save_safely(baseline_path(country, window, experiment), lambda path: joblib.dump(saved_model, path))

In [257]:
for experiment in BASELINE_EXPERIMENTS:
    if not should_run(experiment):
        continue
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for country in LOCAL_COUNTRIES:
        valid = load_labeled(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if baseline_complete(country, window, experiment):
                print(f"{experiment} | {country} | {window}: skipped (saved model and results are complete)")
                continue
            train = load_labeled(country, window)
            assert train["info"]["predicted_month"].max() < valid["info"]["predicted_month"].min()
            run_baseline(experiment, country, window, train, valid)

elastic_net: not run (RUN_MODE = 'off')
xgboost: not run (RUN_MODE = 'off')


In [258]:
if VALIDATION_FILE.exists():
    validation_results = pd.read_csv(VALIDATION_FILE)
    best = validation_results[validation_results["selected"]]
    hyperparameters = ["alpha", "l1_ratio", "max_depth", "learning_rate", "n_estimators"]
    comparison = best[["country", "window", "family", "train_rows", "validation_rows", "predictors", "settings"]].assign(
        best_settings=[
            ", ".join(f"{name}={row[name]:g}" for name in hyperparameters if pd.notna(row[name]))
            for _, row in best.iterrows()
        ],
        validation_mse=best["validation_mse"], zero_mse=best["zero_mse"],
        max_abs_prediction=best["max_abs_prediction"], prediction_std=best["prediction_std"],
        converged=best["converged"],
    )
    display(comparison)

    not_converged = (~validation_results["converged"]).sum()
    print(f"Settings with convergence warnings: {not_converged} of {len(validation_results)}")
else:
    print("No validation results yet: the baseline experiments have not been run.")

No validation results yet: the baseline experiments have not been run.


## 16. Baseline Model Verification

The checks use the smoke-test fits, the configuration and, when available, the saved full-training results.

In [259]:
assert all(0 < len(grid) <= MAX_SETTINGS for _, grid in MODEL_FAMILIES.values())

# Every supervised period is complete, precedes the test period, and training precedes validation
for country in LOCAL_COUNTRIES:
    assert TRAIN_WINDOWS[country] == ["train_1y", "train_3y", "train_5y"]
    for window in TRAIN_WINDOWS[country]:
        assert PERIODS[window][1] < PERIODS["validation"][0] <= PERIODS["validation"][1] < PERIODS["test"][0]

if VALIDATION_FILE.exists():
    configuration = ["country", "window", "family"]
    assert (validation_results.groupby(configuration).size() <= MAX_SETTINGS).all()
    assert (validation_results.groupby(configuration)["selected"].sum() == 1).all()
    assert (validation_results["predictors"] == len(selected_features)).all()
print("Configuration and search limits verified.")

Configuration and search limits verified.


In [260]:
if RUN_SMOKE_TESTS:
    # Correct periods: no validation month in training, no test month anywhere
    assert in_period(smoke_train["info"]["predicted_month"], "train_1y").all()
    assert in_period(smoke_valid["info"]["predicted_month"], "validation").all()

    # Only labeled, finite targets, and equal total weight per month
    assert smoke_train["info"]["has_label"].all() and np.isfinite(smoke_train["y"]).all()
    weights = pd.Series(month_weights(smoke_train["info"]))
    month_totals = weights.groupby(smoke_train["info"]["eom"]).sum()
    assert np.allclose(month_totals, month_totals.iloc[0]) and np.isclose(weights.mean(), 1)

    for family, model in smoke_models.items():
        assert model.n_features_in_ == len(selected_features) == smoke_train["X"].shape[1]
        predictions = predict_model(model, smoke_valid)
        assert np.isfinite(predictions["prediction"]).all()

        # Each prediction is matched to the same stock and month in the source data
        matched = predictions.merge(data["GBR"][["id", "eom", TARGET]], on=["id", "eom"], validate="one_to_one")
        assert len(matched) == len(predictions) and np.array_equal(matched[TARGET], smoke_valid["y"])
    print("Periods, labels, weights, predictions and stock alignment verified.")

In [261]:
if RUN_SMOKE_TESTS:
    # Weights reach the models: putting all weight on one month must change the fit
    info = smoke_train["info"]
    one_month = (info["eom"] == info["eom"].max()).to_numpy()
    for family, model in smoke_models.items():
        make_model = MODEL_FAMILIES[family][0]
        params = {name: model.get_params()[name] for name in MODEL_FAMILIES[family][1][0]}
        weighted = make_model(params).fit(smoke_train["X"], smoke_train["y"], sample_weight=one_month.astype(float))
        unweighted = make_model(params).fit(smoke_train["X"], smoke_train["y"])
        assert not np.allclose(weighted.predict(smoke_valid["X"]), unweighted.predict(smoke_valid["X"]))
    print("Both models respond to sample weights.")

Phase 3 — End

Phase 4 — Start

## 17. Cross-Sectional Transformer Architecture

The Transformer receives all stocks of one country-month as one sequence and predicts each stock's next-month excess
return. Attention lets a prediction depend on the characteristics of other stocks in the same month. There are no
positional embeddings or stock identifiers, so the order of stocks carries no meaning.

In [262]:
def select_device(choice):
    """Use the requested device, or the first available of cuda, mps and cpu."""
    available = {"cuda": torch.cuda.is_available(), "mps": torch.backends.mps.is_available(), "cpu": True}
    if choice == "auto":
        choice = next(name for name, usable in available.items() if usable)
    if not available.get(choice, False):
        raise ValueError(f"Device '{choice}' is not available on this computer.")
    return torch.device(choice)


DEVICE = select_device(DEVICE_CHOICE)
print(f"Device: {DEVICE}")

N_HEADS = 4
FEEDFORWARD_DIM = 128
MAX_EPOCHS = RESEARCH_EPOCHS  # the proposal maximum is 100; LOCAL_RUN sets its own limit
PATIENCE = 10  # epochs without validation improvement before stopping
TRANSFORMER_SEEDS = [SEED, SEED + 1, SEED + 2]  # the first seed is also used for the hyperparameter search

DEFAULT_CONFIG = {
    "embedding_dim": 64, "n_blocks": 2, "dropout": 0.10, "learning_rate": 3e-4, "weight_decay": 1e-4,
}

Device: mps


In [263]:
class TransformerBlock(nn.Module):
    """Attention across stocks followed by a feed-forward layer, each with a residual connection."""

    def __init__(self, embedding_dim, dropout):
        super().__init__()
        self.dropout_rate = dropout
        self.attention_norm = nn.LayerNorm(embedding_dim)
        self.query_key_value = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.attention_output = nn.Linear(embedding_dim, embedding_dim)
        self.feedforward_norm = nn.LayerNorm(embedding_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(embedding_dim, FEEDFORWARD_DIM), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(FEEDFORWARD_DIM, embedding_dim),
        )
        self.dropout = nn.Dropout(dropout)

    def attend(self, x):
        n_stocks, embedding_dim = x.shape
        # Shape of query, key and value: (1 month, heads, stocks, dimension per head)
        query, key, value = [
            part.view(1, n_stocks, N_HEADS, embedding_dim // N_HEADS).transpose(1, 2)
            for part in self.query_key_value(x).chunk(3, dim=-1)
        ]
        attended = F.scaled_dot_product_attention(
            query, key, value, dropout_p=self.dropout_rate if self.training else 0.0
        )
        return self.attention_output(attended.transpose(1, 2).reshape(n_stocks, embedding_dim))

    def forward(self, x):
        x = x + self.dropout(self.attend(self.attention_norm(x)))
        return x + self.dropout(self.feedforward(self.feedforward_norm(x)))

In [264]:
class CrossSectionalTransformer(nn.Module):
    """Predict one return per stock from a (stocks, features) matrix of a single country-month."""

    def __init__(self, n_features, embedding_dim, n_blocks, dropout):
        super().__init__()
        self.input_layer = nn.Linear(n_features, embedding_dim)
        self.input_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([TransformerBlock(embedding_dim, dropout) for _ in range(n_blocks)])
        self.output_norm = nn.LayerNorm(embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, 1)

    def forward(self, x):
        x = self.input_dropout(self.input_layer(x))
        for block in self.blocks:
            x = block(x)
        return self.output_layer(self.output_norm(x)).squeeze(-1)


def make_transformer(config):
    """Build a new model on the selected device."""
    return CrossSectionalTransformer(
        len(selected_features), config["embedding_dim"], config["n_blocks"], config["dropout"]
    ).to(DEVICE)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 18. Monthly Training and Validation

Each country-month is one training step, so every month has equal importance whatever its number of stocks. The order
of training months is shuffled each epoch with a fixed seed; stocks and months are never mixed. All stocks enter the
attention, but losses use labeled stocks only. Months without labels are skipped.

In [265]:
def make_months(info, X):
    """Split prepared data into one set of CPU tensors per country-month."""
    months = []
    for eom in info["eom"].unique():
        month_info, month_X = get_cross_section(info, X, eom)
        months.append({
            "info": month_info.reset_index(drop=True),
            "X": torch.from_numpy(month_X),
            "y": torch.tensor(month_info[TARGET].to_numpy(), dtype=torch.float32),
            "labeled": torch.tensor(month_info["has_label"].to_numpy(), dtype=torch.bool),
        })
    return months


def load_months(country, period):
    """Load training or validation months for one country."""
    if period == "test":
        raise ValueError("Test data cannot be used for training or model selection.")
    return make_months(*prepare_data(country, period, selected_features))


def month_mse(model, month):
    """Squared error averaged over the labeled stocks of one month."""
    labeled = month["labeled"].to(DEVICE)
    prediction = model(month["X"].to(DEVICE))
    return ((prediction[labeled] - month["y"].to(DEVICE)[labeled]) ** 2).mean()


def describe_month(month):
    first = month["info"].iloc[0]
    return f"{first['country']} {first['eom']:%Y-%m} ({len(month['info'])} stocks)"


def device_error(month, error):
    """Explain out-of-memory and unsupported-operation errors; other errors are returned unchanged."""
    if isinstance(error, torch.cuda.OutOfMemoryError) or "out of memory" in str(error).lower():
        return RuntimeError(
            f"{DEVICE} memory is too small for full attention on {describe_month(month)}. "
            "Use a device with more memory, or DEVICE_CHOICE = 'cpu'; the cross-section must not be split."
        )
    if isinstance(error, NotImplementedError):
        return RuntimeError(f"An operation is not supported on {DEVICE}. Set DEVICE_CHOICE = 'cpu'.")
    return error


In [266]:
def validate_transformer(model, months):
    """Monthly-equal MSE without gradients."""
    model.eval()
    losses = []
    with torch.inference_mode():
        for month in months:
            if month["labeled"].any():
                try:
                    losses.append(month_mse(model, month).item())
                except RuntimeError as error:
                    raise device_error(month, error) from error
    if not losses:
        raise ValueError("No labeled months are available for validation.")
    return float(np.mean(losses))


def predict_transformer(model, months):
    """Predictions for all stocks, aligned with identifiers and months."""
    model.eval()
    predictions = []
    with torch.inference_mode():
        for month in months:
            month_predictions = month["info"][["country", "id", "eom", "predicted_month"]].copy()
            try:
                month_predictions["prediction"] = model(month["X"].to(DEVICE)).cpu().numpy()
            except RuntimeError as error:
                raise device_error(month, error) from error
            predictions.append(month_predictions)
    return pd.concat(predictions, ignore_index=True)

In [267]:
def train_one_epoch(model, optimizer, months, order):
    """One optimizer step per month; returns the average monthly training MSE."""
    model.train()
    losses, last_update = [], time.time()
    for step, i in enumerate(order, start=1):
        month = months[i]
        optimizer.zero_grad(set_to_none=True)
        try:
            loss = month_mse(model, month)
            loss.backward()
        except RuntimeError as error:
            raise device_error(month, error) from error

        if not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss in {describe_month(month)}.")
        if not all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None):
            raise FloatingPointError(f"Nonfinite gradient in {describe_month(month)}.")
        optimizer.step()
        losses.append(loss.item())
        if time.time() - last_update > 30:  # progress within long epochs
            print(f"    month {step}/{len(order)}, average training MSE {np.mean(losses):.6f}", flush=True)
            last_update = time.time()
    return float(np.mean(losses))

In [268]:
def print_run(label, train_months, valid_months, config, seed, max_epochs):
    """Describe a training run before it starts."""
    info = pd.concat([month["info"][["country", "id", "predicted_month"]] for month in train_months])
    labeled = sum(int(month["labeled"].sum()) for month in train_months)
    print(f"\nTransformer | {label} | seed {seed} | {DEVICE}")
    print(f"  Countries: {', '.join(info['country'].unique())} | Training: {len(train_months)} months, "
          f"{info['predicted_month'].min():%Y-%m} to {info['predicted_month'].max():%Y-%m}, "
          f"{info[['country', 'id']].drop_duplicates().shape[0]:,} stocks, {len(info):,} rows ({labeled:,} labeled)")
    print(f"  Validation: {len(valid_months)} months | Features: {len(selected_features)} | Maximum epochs: {max_epochs}")
    print("  Settings: " + ", ".join(f"{name}={value:g}" for name, value in config.items()), flush=True)

Training stops after `PATIENCE` epochs without a lower validation MSE, or after 100 epochs, and the best state is
restored. `initial_state` allows training to continue from existing weights.

In [269]:
def train_transformer(train_months, valid_months, config, seed, label, initial_state=None,
                      max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Train with early stopping; returns the best model and a record of the run."""
    if not 1 <= max_epochs <= 100:
        raise ValueError("max_epochs must be between 1 and 100.")
    torch.manual_seed(seed)  # also seeds CUDA and MPS
    shuffler = np.random.default_rng(seed)
    model = make_transformer(config)
    if initial_state is not None:
        model.load_state_dict(initial_state)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"]
    )
    train_months = [month for month in train_months if month["labeled"].any()]
    if not train_months:
        raise ValueError(f"{label}: no labeled training months.")

    print_run(label, train_months, valid_months, config, seed, max_epochs)

    started = time.time()
    best_mse, best_epoch, best_state, history = np.inf, 0, None, []
    for epoch in range(1, max_epochs + 1):
        train_mse = train_one_epoch(model, optimizer, train_months, shuffler.permutation(len(train_months)))
        valid_mse = validate_transformer(model, valid_months)
        if not np.isfinite(valid_mse):
            raise FloatingPointError(f"{label}: nonfinite validation MSE at epoch {epoch}.")
        history.append({"epoch": epoch, "train_mse": train_mse, "validation_mse": valid_mse})

        if valid_mse < best_mse:
            best_mse, best_epoch = valid_mse, epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

        stop = epoch - best_epoch >= patience
        print(f"  epoch {epoch}/{max_epochs} | train MSE {train_mse:.6f} | validation MSE {valid_mse:.6f} | "
              f"best {best_mse:.6f} | {time.time() - started:.0f}s", flush=True)
        if stop:
            print(f"  stopped early: no improvement for {patience} epochs", flush=True)
            break

    model.load_state_dict(best_state)
    run = {
        "config": dict(config), "seed": seed, "best_epoch": best_epoch, "validation_mse": best_mse,
        "n_parameters": count_parameters(model), "history": history, "max_epochs": max_epochs, "state": best_state,
    }
    return model, run

In [ ]:
if RUN_PREFLIGHT:
    # One training step of the largest model on the largest cross-section, with artificial inputs
    sizes = pd.Series({country: df.groupby("eom").size().max() for country, df in data.items()})
    print(f"Device: {DEVICE} | features: {len(selected_features)} | largest cross-sections: {sizes.to_dict()}")
    n_stocks = int(sizes.max())
    generator = np.random.default_rng(SEED)
    largest_month = {
        "info": pd.DataFrame({"country": sizes.idxmax(), "id": np.arange(n_stocks), "eom": pd.Timestamp("2000-01-31")}),
        "X": torch.from_numpy(generator.uniform(-0.5, 0.5, (n_stocks, len(selected_features))).astype("float32")),
        "y": torch.zeros(n_stocks), "labeled": torch.ones(n_stocks, dtype=torch.bool),
    }
    largest_model = make_transformer({**DEFAULT_CONFIG, "embedding_dim": 128, "n_blocks": 2})
    started = time.time()
    train_one_epoch(largest_model, torch.optim.AdamW(largest_model.parameters()), [largest_month], [0])
    print(f"Largest training step: {n_stocks:,} stocks in {time.time() - started:.1f} seconds")
    if DEVICE.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name()} | peak memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB of "
              f"{torch.cuda.get_device_properties(DEVICE).total_memory / 1e9:.1f} GB")
    del largest_model, largest_month

## 19. Transformer Hyperparameter Selection

The 18 settings cover all eight architectures (embedding 64/128, 1/2 blocks, dropout 10%/20%) at two learning rates,
plus a stronger weight decay for the default architecture at both learning rates. Selection uses only the 2018–2019
validation MSE.

In [270]:
TRANSFORMER_GRID = [
    {"embedding_dim": dim, "n_blocks": blocks, "dropout": dropout, "learning_rate": rate, "weight_decay": 1e-4}
    for dim, blocks, dropout, rate in product([64, 128], [2, 1], [0.10, 0.20], [3e-4, 1e-3])
]
TRANSFORMER_GRID += [{**DEFAULT_CONFIG, "learning_rate": rate, "weight_decay": 1e-2} for rate in [3e-4, 1e-3]]

print(f"Transformer settings: {len(TRANSFORMER_GRID)}")

Transformer settings: 18


In [271]:
def search_transformer(train_months, valid_months, label, grid=TRANSFORMER_GRID, initial_state=None):
    """Evaluate every setting with one seed; returns the best run and a table of all settings."""
    best_run, trials = None, []
    for number, config in enumerate(grid, start=1):
        trial_label = f"{label} | setting {number}/{len(grid)}"
        trial = {**config, "seed": SEED, "validation_mse": np.nan, "best_epoch": np.nan, "failure": ""}
        try:
            _, run = train_transformer(train_months, valid_months, config, SEED, trial_label, initial_state)
            trial.update(
                validation_mse=run["validation_mse"], best_epoch=run["best_epoch"], n_parameters=run["n_parameters"]
            )
            if best_run is None or run["validation_mse"] < best_run["validation_mse"]:
                best_run = run
        except FloatingPointError as error:  # unstable setting: record it and continue
            trial["failure"] = str(error)
            print(f"FAILED {trial_label}: {error}", flush=True)
        trials.append(trial)

    if best_run is None:
        raise RuntimeError(f"{label}: every setting became numerically unstable.")
    return best_run, pd.DataFrame(trials)


def train_seeds(train_months, valid_months, best_run, label, initial_state=None):
    """Train the selected setting from three seeds; the search run already provides the first."""
    runs = [best_run]
    for seed in TRANSFORMER_SEEDS:
        if seed != best_run["seed"]:
            _, run = train_transformer(train_months, valid_months, best_run["config"], seed, label, initial_state)
            runs.append(run)
    return runs

## 20. Local Transformer Experiments

`local_train` mode trains the default Transformer once on real data, using the country and history in `LOCAL_RUN`. Its
checkpoint is stored in the `development/` results folder and is never used as a research result.

`full_research` mode applies the complete search and three seeds to the UK and Japan with 1, 3 and 5 years of local data,
all ending in December 2017. Only the three selected runs per configuration are saved, each with everything needed to
rebuild the model. Configurations verified as complete are skipped, so interrupted training can be resumed.

In [272]:
TRANSFORMER_FILE = RESULTS_DIR / "transformer_local_validation.csv"
DEV_DIR = RESULTS_DIR / "development"


def checkpoint_path(experiment, country, window, seed):
    if experiment == "development":
        return DEV_DIR / f"DEV_{country}_{window}_transformer_seed{seed}.pt"
    return MODELS_DIR / f"{country}_{window}_transformer_{experiment}_seed{seed}.pt"


def save_checkpoint(run, experiment, country, window):
    checkpoint = {
        **run, "experiment": experiment, "country": country, "window": window, "n_heads": N_HEADS,
        "feedforward_dim": FEEDFORWARD_DIM, "n_features": len(selected_features), "features": list(selected_features),
    }
    save_safely(checkpoint_path(experiment, country, window, run["seed"]), lambda path: torch.save(checkpoint, path))


def load_checkpoint(path):
    """Rebuild a saved model; returns (model, checkpoint)."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    if checkpoint["features"] != selected_features:
        raise ValueError(f"{path.name}: saved predictors differ from selected_features.")
    if (checkpoint["n_heads"], checkpoint["feedforward_dim"]) != (N_HEADS, FEEDFORWARD_DIM):
        raise ValueError(f"{path.name}: saved architecture differs from the notebook settings.")
    model = make_transformer(checkpoint["config"])
    model.load_state_dict(checkpoint["state"])
    return model.eval(), checkpoint

In [273]:
def check_training_periods(train_months, valid_months, window):
    """Confirm that months belong to the permitted training and validation periods."""
    train_dates = pd.concat([month["info"]["predicted_month"] for month in train_months])
    valid_dates = pd.concat([month["info"]["predicted_month"] for month in valid_months])
    assert in_period(train_dates, window).all() and in_period(valid_dates, "validation").all()
    assert train_dates.max() < valid_dates.min() and valid_dates.max() < PERIODS["test"][0]


def run_local_experiment(country, window, valid_months):
    """Search and train three seeds for one country and history; returns the results table and the runs."""
    label = f"local | {country} | {window}"
    train_months = load_months(country, window)
    check_training_periods(train_months, valid_months, window)

    best_run, trials = search_transformer(train_months, valid_months, label)
    runs = train_seeds(train_months, valid_months, best_run, label)
    seed_rows = []
    for run in runs:
        seed_rows.append({
            **run["config"], "seed": run["seed"], "validation_mse": run["validation_mse"],
            "best_epoch": run["best_epoch"], "n_parameters": run["n_parameters"], "failure": "",
        })
    results = pd.concat([trials.assign(stage="search"), pd.DataFrame(seed_rows).assign(stage="seed")])
    return results.assign(country=country, window=window), runs

In [274]:
def all_seeds_saved(experiment, country, window):
    """True if all three seeds are saved as complete research runs."""
    paths = [checkpoint_path(experiment, country, window, seed) for seed in TRANSFORMER_SEEDS]
    if not all(path.exists() for path in paths):
        return False
    for seed, path in zip(TRANSFORMER_SEEDS, paths):
        _, checkpoint = load_checkpoint(path)
        identity = (checkpoint["experiment"], checkpoint["country"], checkpoint["window"], checkpoint["seed"])
        if identity != (experiment, country, window, seed) or checkpoint["max_epochs"] != MAX_EPOCHS:
            raise ValueError(f"{path.name} is not a complete research run. Move it before rerunning.")
    return True

In [275]:
if RUN_MODE == "local_train":
    country, window, seed = LOCAL_RUN["country"], LOCAL_RUN["window"], LOCAL_RUN["seed"]
    if window not in TRAIN_WINDOWS.get(country, []):
        raise ValueError(f"LOCAL_RUN: choose a window from {TRAIN_WINDOWS} (country first).")

    print("Loading real data for the development run...")
    dev_train = load_months(country, window)
    dev_valid = load_months(country, "validation")
    check_training_periods(dev_train, dev_valid, window)
    dev_model, dev_run = train_transformer(
        dev_train, dev_valid, DEFAULT_CONFIG, seed, f"DEVELOPMENT | {country} | {window}",
        max_epochs=LOCAL_RUN["max_epochs"],
    )

    DEV_DIR.mkdir(parents=True, exist_ok=True)
    save_checkpoint(dev_run, "development", country, window)
    print(f"\nBest epoch {dev_run['best_epoch']}, validation MSE {dev_run['validation_mse']:.6f}, "
          f"{dev_run['n_parameters']:,} parameters")
    print(f"Development checkpoint: {checkpoint_path('development', country, window, seed)}")
    print("This is a limited one-seed run, not a research result.")
else:
    print(f"Development run: not run (RUN_MODE = '{RUN_MODE}')")

Development run: not run (RUN_MODE = 'off')


In [276]:
if RUN_MODE == "local_train":
    # Compare with forecasting zero, and reload the saved checkpoint
    zero_mse = np.mean([(m["y"][m["labeled"]] ** 2).mean().item() for m in dev_valid if m["labeled"].any()])
    reloaded, _ = load_checkpoint(checkpoint_path("development", country, window, seed))
    assert np.isclose(validate_transformer(reloaded, dev_valid), dev_run["validation_mse"], rtol=1e-4)

    display(pd.DataFrame(dev_run["history"]).set_index("epoch"))
    print(f"Zero-forecast validation MSE: {zero_mse:.6f}")
    display(predict_transformer(reloaded, dev_valid[:1]).head())

In [277]:
if should_run("local"):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            saved = pd.DataFrame(columns=["country", "window"])
            if TRANSFORMER_FILE.exists():
                saved = pd.read_csv(TRANSFORMER_FILE)
            this_run = (saved["country"] == country) & (saved["window"] == window)
            if all_seeds_saved("local", country, window):
                if not this_run.any():
                    raise ValueError(f"local | {country} | {window}: models exist but results are missing. Move them first.")
                print(f"local | {country} | {window}: skipped (three verified checkpoints and results exist)")
                continue

            results, runs = run_local_experiment(country, window, valid_months)
            table = pd.concat([saved[~this_run], results], ignore_index=True)
            save_safely(TRANSFORMER_FILE, lambda path: table.to_csv(path, index=False))  # results first, models last
            for run in runs:
                save_checkpoint(run, "local", country, window)

local: not run (RUN_MODE = 'off')


In [278]:
if TRANSFORMER_FILE.exists():
    transformer_results = pd.read_csv(TRANSFORMER_FILE)
    seed_runs = transformer_results[transformer_results["stage"] == "seed"]
    display(seed_runs[[
        "country", "window", "seed", *DEFAULT_CONFIG, "best_epoch", "validation_mse", "n_parameters"
    ]])
else:
    print("No Transformer results yet: the local experiments have not been run.")

No Transformer results yet: the local experiments have not been run.


## 21. Transformer Verification

Configuration checks always run. The smoke test uses small artificial cross-sections and saves nothing.

In [279]:
assert len(TRANSFORMER_GRID) <= MAX_SETTINGS and DEFAULT_CONFIG in TRANSFORMER_GRID
assert all(TRANSFORMER_GRID.count(config) == 1 for config in TRANSFORMER_GRID)
assert all(config["embedding_dim"] % N_HEADS == 0 for config in TRANSFORMER_GRID)
assert len(set(TRANSFORMER_SEEDS)) == 3 and SEED in TRANSFORMER_SEEDS
assert 1 <= MAX_EPOCHS <= 100

for country in LOCAL_COUNTRIES:
    for window in TRAIN_WINDOWS[country]:
        assert PERIODS[window][1] < PERIODS["validation"][0] <= PERIODS["validation"][1] < PERIODS["test"][0]
print("Transformer configuration verified.")

Transformer configuration verified.


In [280]:
def make_smoke_months(sizes, first_eom):
    """Artificial cross-sections with the same structure as prepared data; two stocks per month lack labels."""
    generator = np.random.default_rng(SEED)
    info = pd.concat([
        pd.DataFrame({"country": "GBR", "id": np.arange(size), "eom": eom})
        for size, eom in zip(sizes, pd.date_range(first_eom, periods=len(sizes), freq="ME"))
    ], ignore_index=True)
    info["predicted_month"] = info["eom"] + pd.offsets.MonthEnd(1)
    info[TARGET] = generator.normal(0, 0.1, len(info))
    info.loc[info["id"] < 2, TARGET] = np.nan
    info["has_label"] = info[TARGET].notna()
    X = generator.uniform(-0.5, 0.5, (len(info), len(selected_features))).astype("float32")
    return info, X

In [281]:
if RUN_SMOKE_TESTS:
    print("Checking shapes, alignment, masking and gradients...")
    smoke_info, smoke_X = make_smoke_months([20, 35, 27], "2017-09-30")
    smoke_train_months = make_months(smoke_info, smoke_X)
    smoke_valid_months = make_months(*make_smoke_months([25, 30], "2017-12-31"))
    check_training_periods(smoke_train_months, smoke_valid_months, "train_1y")

    torch.manual_seed(SEED)
    model = make_transformer(DEFAULT_CONFIG).eval()
    month = smoke_train_months[1]
    assert model.input_layer.in_features == len(selected_features)

    # Variable stock counts, finite outputs and identifier alignment
    predictions = predict_transformer(model, smoke_train_months)
    assert np.isfinite(predictions["prediction"]).all()
    keys = ["country", "id", "eom", "predicted_month"]
    assert predictions[keys].equals(smoke_info[keys].reset_index(drop=True))

    # Reordering stocks reorders predictions in the same way
    order = torch.randperm(len(month["X"]))
    with torch.inference_mode():
        original = model(month["X"].to(DEVICE)).cpu()
        reordered = model(month["X"][order].to(DEVICE)).cpu()
    assert original.shape == (len(month["X"]),)
    assert torch.allclose(original[order], reordered, atol=1e-5)

    # Predictions depend on the other stocks in the month
    with torch.inference_mode():
        alone = model(month["X"][:1].to(DEVICE)).cpu()
    assert not torch.allclose(alone, original[:1])

In [282]:
if RUN_SMOKE_TESTS:
    # Unlabeled targets never affect the loss, and months are averaged equally
    changed = {**month, "y": torch.where(month["labeled"], month["y"], torch.tensor(1e6))}
    with torch.inference_mode():
        assert torch.equal(month_mse(model, month), month_mse(model, changed))
        monthly_losses = [month_mse(model, m).item() for m in smoke_train_months]
    assert np.isclose(validate_transformer(model, smoke_train_months), np.mean(monthly_losses))

    # A month without labels is skipped
    unlabeled = {**month, "labeled": torch.zeros_like(month["labeled"])}
    assert validate_transformer(model, [month, unlabeled]) == validate_transformer(model, [month])

    # Finite gradients and a real parameter update
    before = [p.detach().clone() for p in model.parameters()]
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    train_one_epoch(model, optimizer, smoke_train_months, range(len(smoke_train_months)))
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
    assert any(not torch.equal(old, new) for old, new in zip(before, model.parameters()))
    print("Loss masking, monthly averaging, gradients and updates verified.")

In [283]:
if RUN_SMOKE_TESTS:
    # Same seed gives the same run; a different seed gives a different model
    _, first = train_transformer(smoke_train_months, smoke_valid_months, DEFAULT_CONFIG, SEED, "Smoke", max_epochs=3)
    _, second = train_transformer(smoke_train_months, smoke_valid_months, DEFAULT_CONFIG, SEED, "Smoke", max_epochs=3)
    _, other = train_transformer(smoke_train_months, smoke_valid_months, DEFAULT_CONFIG, SEED + 1, "Smoke", max_epochs=3)
    assert np.isclose(first["validation_mse"], second["validation_mse"], rtol=1e-4)
    assert any(not torch.equal(first["state"][name], other["state"][name]) for name in first["state"])

    # The returned model is the best validation state
    best_model = make_transformer(DEFAULT_CONFIG)
    best_model.load_state_dict(first["state"])
    assert np.isclose(validate_transformer(best_model, smoke_valid_months), first["validation_mse"])

    # With no learning, validation never improves, so training stops after one epoch plus the patience
    frozen = {**DEFAULT_CONFIG, "learning_rate": 0.0}
    _, stopped = train_transformer(smoke_train_months, smoke_valid_months, frozen, SEED, "Smoke", patience=2)
    assert stopped["best_epoch"] == 1 and len(stopped["history"]) == 3
    print("Reproducibility, best-state restoration and early stopping verified.")

Phase 4 — End

Phase 5 — Start

## 22. U.S. Transformer Pretraining

The source model is trained on U.S. predicted returns from February 2000 to December 2017 and selected on U.S.
2018–2019 validation data only. Its three seeds are the starting point for direct transfer and fine-tuning.

In [284]:
TRANSFER_FILE = RESULTS_DIR / "transfer_validation.csv"
ARCHITECTURE = ["embedding_dim", "n_blocks", "dropout"]


def architecture(config):
    """The settings that must match between compared models."""
    return {name: config[name] for name in ARCHITECTURE}

In [285]:
def rows_of(results, experiment, country, window):
    """Rows of one experiment in the results table."""
    return (
        (results["experiment"] == experiment) & (results["country"] == country) & (results["window"] == window)
    )


def read_transfer_results():
    if TRANSFER_FILE.exists():
        return pd.read_csv(TRANSFER_FILE)
    return pd.DataFrame(columns=["experiment", "country", "window"])


def record_results(table, experiment, country, window):
    """Add one experiment to the results file, replacing earlier rows of the same experiment."""
    saved = read_transfer_results()
    saved = saved[~rows_of(saved, experiment, country, window)]
    table = table.assign(experiment=experiment, country=country, window=window)
    combined = pd.concat([saved, table], ignore_index=True)
    save_safely(TRANSFER_FILE, lambda path: combined.to_csv(path, index=False))


def is_complete(experiment, country, window, sources=None):
    """True if all seeds are saved and recorded; saved models must match the current U.S. source."""
    paths = [checkpoint_path(experiment, country, window, seed) for seed in TRANSFORMER_SEEDS]
    recorded = rows_of(read_transfer_results(), experiment, country, window).any()
    if not all_seeds_saved(experiment, country, window):
        return False
    if not recorded:
        raise ValueError(f"{paths[0].name} and its other seeds exist but are not in {TRANSFER_FILE}. Move them first.")

    for seed, path in zip(TRANSFORMER_SEEDS, paths):
        _, checkpoint = load_checkpoint(path)
        if sources is not None and architecture(checkpoint["config"]) != architecture(sources[seed]["config"]):
            raise ValueError(f"{path.name} has a different architecture from the U.S. model. Move it before rerunning.")
        if experiment == "finetuned" and checkpoint["source_mse"] != sources[seed]["validation_mse"]:
            raise ValueError(f"{path.name} came from a different U.S. checkpoint. Move it before rerunning.")
    print(f"{experiment} | {country} | {window}: skipped (three verified checkpoints and results exist)")
    return True

In [286]:
def train_experiment(experiment, country, window, train_months, valid_months, grid, sources=None):
    """Search, train three seeds, save and record one experiment.

    With `sources`, each seed starts from the U.S. checkpoint of the same seed; otherwise weights are random.
    """
    label = f"{experiment} | {country} | {window}"
    if sources is None:
        best_run, trials = search_transformer(train_months, valid_months, label, grid)
        runs = train_seeds(train_months, valid_months, best_run, label)
    else:
        best_run, trials = search_transformer(train_months, valid_months, label, grid, sources[SEED]["state"])
        runs = [best_run]
        for seed in TRANSFORMER_SEEDS:
            if seed != SEED:
                _, run = train_transformer(
                    train_months, valid_months, best_run["config"], seed, label, sources[seed]["state"]
                )
                runs.append(run)

    rows = []
    for run in runs:
        seed = run["seed"]
        source_name = "" if sources is None else checkpoint_path("pretrained", "USA", "train_full", seed).name
        run["source_checkpoint"] = source_name
        run["source_mse"] = float("nan") if sources is None else sources[seed]["validation_mse"]
        rows.append({
            **run["config"], "seed": seed, "best_epoch": run["best_epoch"], "validation_mse": run["validation_mse"],
            "n_parameters": run["n_parameters"], "checkpoint": checkpoint_path(experiment, country, window, seed).name,
            "source_checkpoint": source_name,
        })
    table = pd.concat([trials.assign(stage="search"), pd.DataFrame(rows).assign(stage="seed")])
    record_results(table, experiment, country, window)  # results first, models last
    for run in runs:
        save_checkpoint(run, experiment, country, window)

In [287]:
if should_run("pretrained"):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    if not is_complete("pretrained", "USA", "train_full"):
        us_valid = load_months("USA", "validation")
        us_train = load_months("USA", "train_full")
        check_training_periods(us_train, us_valid, "train_full")
        train_experiment("pretrained", "USA", "train_full", us_train, us_valid, TRANSFORMER_GRID)
        del us_train, us_valid

pretrained: not run (RUN_MODE = 'off')


In [288]:
def load_us_sources():
    """Load the three U.S. checkpoints, which must share one configuration."""
    sources = {}
    for seed in TRANSFORMER_SEEDS:
        path = checkpoint_path("pretrained", "USA", "train_full", seed)
        if not path.exists():
            raise FileNotFoundError(f"U.S. checkpoint {path} not found. Run U.S. pretraining first.")
        _, checkpoint = load_checkpoint(path)
        if (checkpoint["experiment"], checkpoint["country"], checkpoint["seed"]) != ("pretrained", "USA", seed):
            raise ValueError(f"{path.name} is not the expected U.S. pretrained model.")
        sources[seed] = checkpoint

    if any(source["config"] != sources[SEED]["config"] for source in sources.values()):
        raise ValueError("The U.S. checkpoints have different configurations.")
    return sources

## 23. Direct Cross-Market Transfer

Each U.S. model is applied to UK and Japanese cross-sections without any weight update. Direct transfer uses no local
training data, so one set of U.S. models serves all three local-history comparisons and nothing new is saved.

In [289]:
def predict_direct_transfer(months, seed):
    """Forecasts of the U.S. model for another market's cross-sections, with weights unchanged."""
    model, _ = load_checkpoint(checkpoint_path("pretrained", "USA", "train_full", seed))
    return predict_transformer(model, months)


def validate_direct_transfer(country):
    """Validation MSE of each U.S. model in another market; returns a results table."""
    valid_months = load_months(country, "validation")
    rows = []
    for seed, source in load_us_sources().items():
        path = checkpoint_path("pretrained", "USA", "train_full", seed)
        model, _ = load_checkpoint(path)
        mse = validate_transformer(model, valid_months)
        assert all(torch.equal(value.cpu(), source["state"][name]) for name, value in model.state_dict().items())
        rows.append({
            **source["config"], "seed": seed, "best_epoch": source["best_epoch"], "validation_mse": mse,
            "n_parameters": source["n_parameters"], "checkpoint": path.name, "source_checkpoint": path.name,
        })
    return pd.DataFrame(rows).assign(stage="seed")

In [290]:
if should_run("direct"):
    for country in LOCAL_COUNTRIES:
        print(f"Validating direct transfer: USA to {country}...")
        record_results(validate_direct_transfer(country), "direct", country, "all")

direct: not run (RUN_MODE = 'off')


## 24. Local Fine-Tuning

A model with the U.S. architecture starts from the U.S. weights of the same seed and continues training on local data.
All parameters are updated. Only the learning rate and weight decay are tuned, and every learning rate is smaller than
in U.S. pretraining. Weights are loaded strictly, so incompatible architectures raise an error.

In [291]:
def fine_tuning_grid(source_config):
    """Smaller learning rates and two weight decays, with the source architecture fixed."""
    return [
        {**source_config, "learning_rate": factor * source_config["learning_rate"], "weight_decay": decay}
        for factor, decay in product([0.3, 0.1, 0.03], [1e-4, 1e-2])
    ]


if should_run("finetuned"):
    sources = load_us_sources()
    grid = fine_tuning_grid(sources[SEED]["config"])
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if is_complete("finetuned", country, window, sources):
                continue
            train_months = load_months(country, window)
            check_training_periods(train_months, valid_months, window)
            train_experiment("finetuned", country, window, train_months, valid_months, grid, sources)

finetuned: not run (RUN_MODE = 'off')


## 25. Joint U.S.–Local Training

A randomly initialized model is trained on U.S. (2000–2017) and local country-months together. Each country-month
remains its own cross-section, so U.S. and local stocks never attend to each other. Selection uses the destination
country's validation data.

Every country-month has equal weight. With 215 U.S. months, local months are about 5% (1 year), 14% (3 years) and 22%
(5 years) of the training objective; no extra weighting is applied.

For controlled comparisons, joint models use the U.S.-selected architecture, and only the learning rate and weight
decay are tuned.

In [292]:
def matched_grid(source_config):
    """Fixed source architecture with a small learning-rate and weight-decay search."""
    return [
        {**architecture(source_config), "learning_rate": rate, "weight_decay": decay}
        for rate, decay in product([1e-4, 3e-4, 1e-3], [1e-4, 1e-2])
    ]


def join_months(us_months, local_months):
    """Combine country-months without merging any cross-sections."""
    joined = us_months + local_months
    assert all(month["info"][["country", "eom"]].drop_duplicates().shape[0] == 1 for month in joined)
    return joined

In [293]:
if should_run("joint"):
    sources = load_us_sources()  # provides the architecture only, never the weights
    grid = matched_grid(sources[SEED]["config"])
    us_months = None
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if is_complete("joint", country, window, sources):
                continue
            if us_months is None:  # load the U.S. history once, and only if needed
                us_months = load_months("USA", "train_full")
            local_months = load_months(country, window)
            check_training_periods(us_months, valid_months, "train_full")
            check_training_periods(local_months, valid_months, window)
            train_experiment("joint", country, window, join_months(us_months, local_months), valid_months, grid)
    del us_months

joint: not run (RUN_MODE = 'off')


## 26. Transfer Experiment Configuration

Section 20 tunes the local-only architecture freely, so it may differ from the U.S. architecture. The primary comparison
therefore uses matched local-only models: random initialization, local data only, the U.S. architecture and the same
search as joint training. They are saved separately and the Section 20 models are kept.

| Strategy | Experiment | Initial weights | Training data | Search |
|---|---|---|---|---|
| Local only, freely tuned | `local` | Random | Local | 18 settings |
| Local only, matched | `localmatched` | Random | Local | 6 settings |
| Direct transfer | `direct` | U.S. model | None | None |
| Fine-tuning | `finetuned` | U.S. model | Local | 6 settings |
| Joint training | `joint` | Random | U.S. + local | 6 settings |

The order of execution is U.S. pretraining, direct transfer, fine-tuning, joint training and matched local models.
Run one notebook at a time, because all experiments share one results file.

In [294]:
if should_run("localmatched"):
    sources = load_us_sources()  # provides the architecture only, never the weights
    grid = matched_grid(sources[SEED]["config"])
    for country in LOCAL_COUNTRIES:
        valid_months = load_months(country, "validation")
        for window in TRAIN_WINDOWS[country]:
            if is_complete("localmatched", country, window, sources):
                continue
            train_months = load_months(country, window)
            check_training_periods(train_months, valid_months, window)
            train_experiment("localmatched", country, window, train_months, valid_months, grid)

localmatched: not run (RUN_MODE = 'off')


In [295]:
if TRANSFER_FILE.exists():
    transfer_results = read_transfer_results()
    seed_runs = transfer_results[transfer_results["stage"] == "seed"]
    print("Validation results by seed:")
    display(seed_runs[[
        "experiment", "country", "window", "seed", *DEFAULT_CONFIG, "best_epoch", "validation_mse", "n_parameters",
        "checkpoint", "source_checkpoint",
    ]])

    # Compare the freely tuned local architectures with the U.S. architecture
    us_rows = seed_runs[rows_of(seed_runs, "pretrained", "USA", "train_full")]
    if TRANSFORMER_FILE.exists() and not us_rows.empty:
        local_runs = pd.read_csv(TRANSFORMER_FILE).query("stage == 'seed'")
        local_runs = local_runs.drop_duplicates(["country", "window"])[["country", "window", *ARCHITECTURE]]
        local_runs["matches_us"] = (local_runs[ARCHITECTURE] == us_rows[ARCHITECTURE].iloc[0]).all(axis=1)
        print("Freely tuned local-only architectures:")
        display(local_runs)
else:
    print("No transfer results yet: the transfer experiments have not been run.")

No transfer results yet: the transfer experiments have not been run.


## 27. Transfer Learning Verification

Configuration checks always run. Checkpoint checks apply to any saved models, and the smoke test uses the artificial
months of Section 21 when `RUN_SMOKE_TESTS` is on.

In [296]:
fine_grid, joint_grid = fine_tuning_grid(DEFAULT_CONFIG), matched_grid(DEFAULT_CONFIG)

assert max(len(TRANSFORMER_GRID), len(fine_grid), len(joint_grid)) <= MAX_SETTINGS
assert all(architecture(config) == architecture(DEFAULT_CONFIG) for config in fine_grid + joint_grid)
assert all(config["learning_rate"] < DEFAULT_CONFIG["learning_rate"] for config in fine_grid)
assert len(set(TRANSFORMER_SEEDS)) == 3 and SEED in TRANSFORMER_SEEDS

# U.S. and local training end before validation, which ends before testing
assert TRAIN_WINDOWS["USA"] == ["train_full"]
assert all(PERIODS[window][1] < PERIODS["validation"][0] for windows in TRAIN_WINDOWS.values() for window in windows)
assert PERIODS["validation"][1] < PERIODS["test"][0]
print("Transfer configuration verified.")

Transfer configuration verified.


In [297]:
EXPERIMENTS = [("pretrained", "USA", "train_full")] + [
    (experiment, country, window)
    for experiment in ["local", "localmatched", "finetuned", "joint"]
    for country in LOCAL_COUNTRIES for window in TRAIN_WINDOWS[country]
]
paths = [checkpoint_path(*names, seed) for names in EXPERIMENTS for seed in TRANSFORMER_SEEDS]
assert len(set(paths)) == len(paths)  # every model has its own file

n_checked = 0
for names in EXPERIMENTS:
    for seed in TRANSFORMER_SEEDS:
        path = checkpoint_path(*names, seed)
        if not path.exists():
            continue
        _, checkpoint = load_checkpoint(path)  # also checks the predictor order and strict weight loading
        assert (checkpoint["experiment"], checkpoint["country"], checkpoint["window"]) == names
        assert checkpoint["seed"] == seed
        if names[0] in ["localmatched", "finetuned", "joint"]:
            _, source = load_checkpoint(checkpoint_path("pretrained", "USA", "train_full", seed))
            assert architecture(checkpoint["config"]) == architecture(source["config"])
            if names[0] == "finetuned":
                assert checkpoint["source_mse"] == source["validation_mse"]
                assert checkpoint["config"]["learning_rate"] < source["config"]["learning_rate"]
            else:
                assert checkpoint["source_checkpoint"] == ""  # random initialization
        n_checked += 1
print(f"Saved Transformer checkpoints verified: {n_checked} of {len(paths)}")

Saved Transformer checkpoints verified: 0 of 75


In [298]:
if RUN_SMOKE_TESTS:
    us_info, us_X = make_smoke_months([40, 32], "2017-10-31")
    us_smoke = make_months(us_info.assign(country="USA"), us_X)
    local_smoke = make_months(*make_smoke_months([20, 25], "2017-10-31"))
    valid_smoke = make_months(*make_smoke_months([22, 28], "2017-12-31"))
    check_training_periods(us_smoke, valid_smoke, "train_full")
    check_training_periods(local_smoke, valid_smoke, "train_1y")

    model, source = train_transformer(us_smoke, us_smoke, DEFAULT_CONFIG, SEED, "Smoke source", max_epochs=2)

    # Direct transfer: predictions are aligned and weights do not change
    predictions = predict_transformer(model, local_smoke)
    local_keys = pd.concat([month["info"][["id", "eom"]] for month in local_smoke], ignore_index=True)
    assert predictions[["id", "eom"]].equals(local_keys)
    assert all(torch.equal(value.cpu(), source["state"][name]) for name, value in model.state_dict().items())

In [299]:
if RUN_SMOKE_TESTS:
    def same_weights(first, second):
        return all(torch.equal(first[name], second[name]) for name in first)

    # Fine-tuning starts exactly from the source weights (a zero learning rate leaves them unchanged), then updates them
    frozen = {**DEFAULT_CONFIG, "learning_rate": 0.0}
    _, start = train_transformer(
        local_smoke, valid_smoke, frozen, SEED, "Smoke start", source["state"], max_epochs=1
    )
    _, tuned = train_transformer(
        local_smoke, valid_smoke, fine_grid[0], SEED, "Smoke fine-tuning", source["state"], max_epochs=1
    )
    assert same_weights(start["state"], source["state"])
    assert not same_weights(tuned["state"], source["state"])

    # Joint training starts from random weights and keeps the two markets in separate cross-sections
    joint_smoke = join_months(us_smoke, local_smoke)
    assert len(joint_smoke) == len(us_smoke) + len(local_smoke)
    _, joint_start = train_transformer(joint_smoke, valid_smoke, frozen, SEED, "Smoke joint", max_epochs=1)
    assert not same_weights(joint_start["state"], source["state"])

    # Different architectures cannot be loaded silently
    try:
        make_transformer({**DEFAULT_CONFIG, "embedding_dim": 128}).load_state_dict(source["state"])
        mismatch_rejected = False
    except RuntimeError:
        mismatch_rejected = True
    assert mismatch_rejected
    print("Direct transfer, fine-tuning and joint-training behavior verified.")

Phase 5 — End

Phase 6 — Start

## 28. Model Checkpoint Readiness

Predictions use only models saved by earlier sections; nothing is trained here. The primary comparison has six strategies.
The independently tuned local-only Transformer (`local`) is predicted separately and is not part of the ensemble.

In [ ]:
BASELINE_STRATEGIES = ["elastic_net", "xgboost"]
PRIMARY_STRATEGIES = BASELINE_STRATEGIES + ["localmatched", "direct", "finetuned", "joint"]
ALL_STRATEGIES = PRIMARY_STRATEGIES + ["local"]
TRANSFORMER_STRATEGIES = [s for s in ALL_STRATEGIES if s not in BASELINE_STRATEGIES]


def us_checkpoint_path(seed):
    return checkpoint_path("pretrained", "USA", "train_full", seed)


def prediction_files(strategy, country, window):
    """Saved model file of each seed; direct transfer uses the same U.S. files for every country and window."""
    if strategy in BASELINE_STRATEGIES:
        return {"single": baseline_path(country, window, strategy)}
    if strategy == "direct":
        return {str(seed): us_checkpoint_path(seed) for seed in TRANSFORMER_SEEDS}
    return {str(seed): checkpoint_path(strategy, country, window, seed) for seed in TRANSFORMER_SEEDS}


def checkpoint_readiness():
    """List every required model file and whether it exists."""
    return pd.DataFrame([
        {"country": country, "window": window, "strategy": strategy, "seed": seed, "file": path.name,
         "available": path.exists()}
        for country in LOCAL_COUNTRIES for window in TRAIN_WINDOWS[country] for strategy in ALL_STRATEGIES
        for seed, path in prediction_files(strategy, country, window).items()
    ])

In [ ]:
def load_baseline(country, window, strategy):
    """Load a saved Elastic Net or XGBoost model and check its predictors."""
    path = baseline_path(country, window, strategy)
    saved = joblib.load(path)
    if saved["features"] != selected_features or saved["model"].n_features_in_ != len(selected_features):
        raise ValueError(f"{path.name}: saved predictors differ from selected_features.")
    return saved["model"]


def load_transformer(country, window, strategy, seed):
    """Load a saved Transformer and check its identity, training length and architecture."""
    if strategy == "direct":
        path, identity = us_checkpoint_path(seed), ("pretrained", "USA", "train_full", seed)
    else:
        path, identity = checkpoint_path(strategy, country, window, seed), (strategy, country, window, seed)

    model, checkpoint = load_checkpoint(path)  # checks predictor names and order
    if tuple(checkpoint[name] for name in ["experiment", "country", "window", "seed"]) != identity:
        raise ValueError(f"{path.name}: the checkpoint belongs to a different experiment.")
    if checkpoint["max_epochs"] != MAX_EPOCHS:
        raise ValueError(f"{path.name}: trained with a maximum of {checkpoint['max_epochs']} epochs, not {MAX_EPOCHS}.")

    if strategy in ["localmatched", "finetuned", "joint"]:
        _, source = load_checkpoint(us_checkpoint_path(seed))
        if architecture(checkpoint["config"]) != architecture(source["config"]):
            raise ValueError(f"{path.name}: the architecture differs from the U.S. model.")
        if strategy == "finetuned" and checkpoint["source_mse"] != source["validation_mse"]:
            raise ValueError(f"{path.name}: fine-tuned from a different U.S. checkpoint.")
    return model

In [ ]:
if RUN_PHASE6_PREDICTIONS:
    readiness = checkpoint_readiness()
    missing_models = readiness[~readiness["available"]]
    print(f"Model files available: {readiness['available'].sum()} of {len(readiness)}")
    if not missing_models.empty:
        display(missing_models)
        raise FileNotFoundError(f"{len(missing_models)} required model files are missing; see the table above.")
else:
    print("Checkpoint readiness: not checked (RUN_PHASE6_PREDICTIONS = False)")

## 29. Out-of-Sample Predictions

Test data are loaded by a separate inference-only function; the training loaders still reject the test period. Every
available stock receives a prediction, including those without an actual return. Transformers see complete monthly
cross-sections in evaluation mode, without gradients.

In [ ]:
KEYS = ["country", "id", "eom", "predicted_month"]


def load_test_for_inference(country):
    """Test-period data for inference only; never used for training or model selection."""
    info, X = prepare_data(country, "test", selected_features)
    assert in_period(info["predicted_month"], "test").all()
    assert info["predicted_month"].nunique() == expected_months["test"]
    return info, X


def predict_baseline(model, info, X):
    """Baseline predictions for every stock, with identifiers."""
    predictions = info[KEYS].copy()
    predictions["prediction"] = model.predict(X)
    return predictions


def check_predictions(predictions, info, label):
    """Require one finite prediction for every stock-month."""
    if predictions.duplicated(KEYS).any():
        raise ValueError(f"{label}: duplicate stock-months in the predictions.")
    if len(predictions) != len(info):
        raise ValueError(f"{label}: {len(predictions):,} predictions for {len(info):,} stock-months.")
    if not np.isfinite(predictions["prediction"]).all():
        raise ValueError(f"{label}: missing or infinite predictions.")

In [ ]:
def predict_strategy(country, window, strategy, info, X, months):
    """Predictions of one strategy for each of its saved models: {seed: predictions}."""
    results = {}
    for seed in prediction_files(strategy, country, window):
        if strategy in BASELINE_STRATEGIES:
            predictions = predict_baseline(load_baseline(country, window, strategy), info, X)
        else:
            predictions = predict_transformer(load_transformer(country, window, strategy, int(seed)), months)
        check_predictions(predictions, info, f"{country} {window} {strategy} {seed}")
        results[seed] = predictions
    return results

## 30. Ensemble Construction

The rule is fixed before any test outcome is examined. Transformer forecasts are first averaged over the three seeds.
The ensemble is then the equal-weight average of the six primary strategies within each country and training history,
so a strategy with three seeds does not receive more weight. A partial ensemble is never produced.

In [ ]:
ENSEMBLE_WEIGHTS = {strategy: 1 / len(PRIMARY_STRATEGIES) for strategy in PRIMARY_STRATEGIES}


def final_column(strategy):
    """Column holding the strategy's final forecast: one model, or the average of three seeds."""
    return f"{strategy}|single" if strategy in BASELINE_STRATEGIES else f"{strategy}|average"


def add_seed_averages(wide):
    """Average the three seeds of every Transformer strategy."""
    for strategy in TRANSFORMER_STRATEGIES:
        wide[f"{strategy}|average"] = wide[[f"{strategy}|{seed}" for seed in TRANSFORMER_SEEDS]].mean(axis=1, skipna=False)
    return wide


def add_ensemble(wide):
    """Equal-weight average of the primary strategies; all of them must be present and complete."""
    columns = {strategy: final_column(strategy) for strategy in PRIMARY_STRATEGIES}
    absent = [strategy for strategy, column in columns.items() if column not in wide or wide[column].isna().any()]
    if absent:
        raise ValueError(f"The ensemble cannot be built; incomplete strategies: {absent}")
    wide["ensemble|ensemble"] = sum(ENSEMBLE_WEIGHTS[strategy] * wide[column] for strategy, column in columns.items())
    return wide

## 31. Prediction Alignment and Storage

Forecasts are joined by country, stock and month, never by row position. `in_common` marks stock-months predicted by all
primary strategies, and `has_actual` marks those with an actual return. All forecasts go to one compressed Parquet file
with one row per stock-month, training history, strategy and seed (`single`, a seed number, `average` or `ensemble`).
Direct transfer does not depend on local history, so its forecasts are repeated for each history.

In [ ]:
PREDICTIONS_FILE = RESULTS_DIR / "test_predictions.parquet"
OVERWRITE_PREDICTIONS = False  # set to True only to replace completed predictions

PREDICTION_SCHEMA = pa.schema([
    ("country", pa.string()), ("id", pa.int64()), ("eom", pa.timestamp("ns")), ("predicted_month", pa.timestamp("ns")),
    ("window", pa.string()), ("strategy", pa.string()), ("seed", pa.string()), ("prediction", pa.float64()),
    ("actual_return", pa.float64()), ("has_actual", pa.bool_()), ("in_common", pa.bool_()),
])


def build_wide_table(info, strategy_predictions):
    """One row per stock-month, with actual returns and one column per strategy and seed."""
    wide = info[KEYS].copy()
    wide["actual_return"] = info[TARGET]
    wide["has_actual"] = info["has_label"]
    for strategy, seeds in strategy_predictions.items():
        for seed, predictions in seeds.items():
            column = predictions.rename(columns={"prediction": f"{strategy}|{seed}"})
            wide = wide.merge(column, on=KEYS, how="left", validate="one_to_one")
            if wide[f"{strategy}|{seed}"].isna().any():
                raise ValueError(f"{strategy} {seed}: some stock-months could not be matched.")

    wide = add_ensemble(add_seed_averages(wide))
    primary_columns = [final_column(strategy) for strategy in PRIMARY_STRATEGIES]
    wide["in_common"] = np.isfinite(wide[primary_columns]).all(axis=1)
    return wide

In [ ]:
def write_wide_table(writer, wide, window):
    """Append each strategy and seed as rows of the prediction schema."""
    for column in [name for name in wide.columns if "|" in name]:
        strategy, seed = column.split("|")
        rows = wide[KEYS + ["actual_return", "has_actual", "in_common"]].assign(
            window=window, strategy=strategy, seed=seed, prediction=wide[column].astype("float64")
        )
        table = pa.Table.from_pandas(rows[PREDICTION_SCHEMA.names], schema=PREDICTION_SCHEMA, preserve_index=False)
        writer.write_table(table)


def read_predictions(country, window, strategy, seed):
    """Read one saved forecast, ordered by month and stock."""
    selection = [("country", "==", country), ("window", "==", window), ("strategy", "==", strategy), ("seed", "==", seed)]
    df = pd.read_parquet(PREDICTIONS_FILE, filters=selection)
    return df.sort_values(["eom", "id"], ignore_index=True)

In [ ]:
def generate_predictions():
    """Predict the test period for every experiment and save one file; nothing is saved if any step fails."""
    temporary_file = PREDICTIONS_FILE.with_suffix(".tmp")
    try:
        with pq.ParquetWriter(temporary_file, PREDICTION_SCHEMA, compression="snappy") as writer:
            for country in LOCAL_COUNTRIES:
                info, X = load_test_for_inference(country)
                months = make_months(info, X)
                direct_predictions = None
                for window in TRAIN_WINDOWS[country]:
                    strategy_predictions = {}
                    for strategy in ALL_STRATEGIES:
                        if strategy == "direct" and direct_predictions is not None:
                            strategy_predictions[strategy] = direct_predictions  # same for every window
                            continue
                        print(f"Predicting: {country} | {window} | {strategy}", flush=True)
                        strategy_predictions[strategy] = predict_strategy(country, window, strategy, info, X, months)
                    direct_predictions = strategy_predictions["direct"]
                    write_wide_table(writer, build_wide_table(info, strategy_predictions), window)
        temporary_file.replace(PREDICTIONS_FILE)
    finally:
        temporary_file.unlink(missing_ok=True)

In [ ]:
if RUN_PHASE6_PREDICTIONS:
    if PREDICTIONS_FILE.exists() and not OVERWRITE_PREDICTIONS:
        print(f"Predictions: skipped ({PREDICTIONS_FILE} exists and OVERWRITE_PREDICTIONS = False)")
    else:
        print(f"Generating test predictions on {DEVICE}...")
        generate_predictions()
        print(f"Saved: {PREDICTIONS_FILE}")
else:
    print("Test predictions: not run (RUN_PHASE6_PREDICTIONS = False)")

## 32. Prediction Verification

Configuration checks always run. The saved file is checked only when predictions are enabled. No performance is
measured here.

In [ ]:
assert set(ENSEMBLE_WEIGHTS) == set(PRIMARY_STRATEGIES) and len(set(PRIMARY_STRATEGIES)) == 6
assert np.isclose(sum(ENSEMBLE_WEIGHTS.values()), 1) and len(set(ENSEMBLE_WEIGHTS.values())) == 1
assert "local" not in PRIMARY_STRATEGIES and "local" in ALL_STRATEGIES
assert PERIODS["test"] == (pd.Timestamp("2020-01-31"), pd.Timestamp("2025-12-31")) and expected_months["test"] == 72

# The training loaders must refuse the test period (they stop before reading any data)
for loader in [load_labeled, load_months]:
    try:
        loader("GBR", "test")
        test_rejected = False
    except ValueError:
        test_rejected = True
    assert test_rejected
print("Prediction configuration verified.")

In [ ]:
def verify_forecast(saved, country, window, strategy, seed):
    """Check the identifiers, dates and values of one saved forecast."""
    label = f"{country} {window} {strategy} {seed}"
    source = data[country]
    source = source[in_period(source["predicted_month"], "test")].sort_values(["eom", "id"], ignore_index=True)

    assert saved.columns.tolist() == PREDICTION_SCHEMA.names, label
    assert not saved.duplicated(KEYS).any() and len(saved) == len(source), label
    assert (saved["country"] == country).all() and (saved["id"] == source["id"]).all(), label
    for month in ["eom", "predicted_month"]:  # every stock-month is present and correctly dated
        assert (saved[month] == source[month]).all(), label
    assert np.isfinite(saved["prediction"]).all(), label
    np.testing.assert_array_equal(saved["actual_return"], source[TARGET], label)  # actual returns are unchanged
    assert (saved["has_actual"] == saved["actual_return"].notna()).all() and saved["in_common"].all(), label

In [ ]:
if RUN_PHASE6_PREDICTIONS:
    print("Verifying saved predictions...")
    for country in LOCAL_COUNTRIES:
        for window in TRAIN_WINDOWS[country]:
            values = {}
            for strategy in ALL_STRATEGIES + ["ensemble"]:
                if strategy == "ensemble":
                    seeds = ["ensemble"]
                else:
                    seeds = list(prediction_files(strategy, country, window))
                    seeds += ["average"] if strategy in TRANSFORMER_STRATEGIES else []
                for seed in seeds:
                    saved = read_predictions(country, window, strategy, seed)
                    verify_forecast(saved, country, window, strategy, seed)
                    values[strategy, seed] = saved["prediction"].to_numpy()

            # Seed averages and the ensemble are reproduced from the saved forecasts
            for strategy in TRANSFORMER_STRATEGIES:
                seed_mean = np.mean([values[strategy, str(seed)] for seed in TRANSFORMER_SEEDS], axis=0)
                assert np.allclose(values[strategy, "average"], seed_mean)
            finals = [values[tuple(final_column(strategy).split("|"))] for strategy in PRIMARY_STRATEGIES]
            assert np.allclose(values["ensemble", "ensemble"], np.mean(finals, axis=0))
            print(f"{country} | {window}: {len(values)} forecasts verified, {len(finals[0]):,} stock-months each")

    n_forecasts = len(LOCAL_COUNTRIES) * 3 * len(values)
    expected_rows = sum(in_period(data[c]["predicted_month"], "test").sum() for c in LOCAL_COUNTRIES) * 3 * len(values)
    assert pq.read_metadata(PREDICTIONS_FILE).num_rows == expected_rows
    print(f"All {n_forecasts} saved forecasts verified.")
else:
    print("Saved predictions: not verified (RUN_PHASE6_PREDICTIONS = False)")

Phase 6 — End

Phase 7 — Start

## 33. Out-of-Sample Prediction Metrics

All models are evaluated on the same stock-months of January 2020 – December 2025: those predicted by every primary
model (`in_common`) with an available actual return. Returns are never clipped or removed.

MSE is the average of 72 monthly MSE values. `OOS R² = 1 − model MSE / zero-forecast MSE`, where both use the same
stock-months and monthly averaging. Coverage is the share of stock-months with an actual return that are evaluated.

In [ ]:
MODEL_NAMES = {
    "ensemble": "Ensemble", "finetuned": "Fine-tuned Transformer", "direct": "Direct U.S. transfer",
    "joint": "Joint U.S.-local Transformer", "localmatched": "Local Transformer (matched)",
    "xgboost": "XGBoost", "elastic_net": "Elastic Net", "local": "Local Transformer (freely tuned)",
}
PRIMARY_MODELS = [name for name in MODEL_NAMES if name != "local"]
EVALUATION_DIR = RESULTS_DIR / "evaluation"

DECILE = 0.10
COST_BPS = 10  # illustrative assumption: basis points per dollar traded (the proposal gives no rate)
BLOCK_MONTHS = 6
BOOTSTRAP_SAMPLES = 2000
SIGNIFICANCE_LEVEL = 0.05


def main_seed(strategy):
    """Identifier of a model's main forecast: single model, seed average or ensemble."""
    return "ensemble" if strategy == "ensemble" else final_column(strategy).split("|")[1]


def forecast_list():
    """Every forecast to evaluate: main forecasts and individual Transformer seeds."""
    forecasts = [(strategy, main_seed(strategy)) for strategy in MODEL_NAMES]
    forecasts += [(strategy, str(seed)) for strategy in TRANSFORMER_STRATEGIES for seed in TRANSFORMER_SEEDS]
    return forecasts

In [ ]:
def load_forecast(country, window, strategy, seed):
    """Load one saved forecast and check its dates, keys and values."""
    label = f"{country} {window} {strategy} {seed}"
    stocks = read_predictions(country, window, strategy, seed)
    if stocks.empty:
        raise ValueError(f"{label}: not found in {PREDICTIONS_FILE}.")
    if stocks.duplicated(KEYS).any():
        raise ValueError(f"{label}: duplicate stock-months.")
    if not in_period(stocks["predicted_month"], "test").all() or stocks["predicted_month"].nunique() != 72:
        raise ValueError(f"{label}: dates differ from January 2020 - December 2025.")
    if not np.isfinite(stocks["prediction"]).all():
        raise ValueError(f"{label}: missing or infinite predictions.")
    if np.isinf(stocks["actual_return"]).any():
        raise ValueError(f"{label}: infinite actual returns.")
    return stocks


def squared_error_metrics(evaluated):
    """MSE of the model and of the zero forecast for one month."""
    actual = evaluated["actual_return"]
    return {"mse": ((actual - evaluated["prediction"]) ** 2).mean(), "zero_mse": (actual ** 2).mean()}

## 34. Cross-Sectional Rank Information Coefficient

The rank IC is the Spearman correlation between predicted and actual returns across stocks, calculated separately for
each month. Ties receive average ranks. A month is undefined when all predictions or all returns are equal.

In [ ]:
def rank_ic(evaluated):
    """Spearman correlation across the stocks of one month."""
    predicted, actual = evaluated["prediction"].rank(), evaluated["actual_return"].rank()
    if predicted.nunique() < 2 or actual.nunique() < 2:
        return np.nan
    return predicted.corr(actual)

## 35. Long-Short Portfolio Construction

Each month, all stocks predicted by every primary model are sorted by predicted return, with ties ordered by stock ID.
The top and bottom `floor(10% × N)` stocks form equally weighted long and short sides of $1 each.

- Stocks are selected without using actual returns. A selected stock with no actual return is left out of that side's
  average return, and the number of such stocks is recorded.
- If a model gives every stock the same forecast, it cannot rank stocks; it then takes no position and earns zero.

In [ ]:
def long_short_weights(universe, decile=DECILE):
    """Signed weights by stock ID: +1/n for the highest share `decile` of stocks and -1/n for the lowest."""
    if universe["prediction"].nunique() < 2:
        return pd.Series(dtype="float64", index=pd.Index([], dtype="int64"))
    ids = universe.sort_values(["prediction", "id"])["id"].to_numpy()
    n = max(1, int(np.floor(decile * len(ids))))
    return pd.Series([-1 / n] * n + [1 / n] * n, index=np.concatenate([ids[:n], ids[-n:]]))


def portfolio_returns(universe, weights):
    """Long, short and long-short return of one month."""
    if weights.empty:
        return {"long_return": 0.0, "short_return": 0.0, "gross_return": 0.0, "n_side": 0, "missing_selected": 0}
    returns = universe.set_index("id")["actual_return"].reindex(weights.index)
    long_return, short_return = returns[weights > 0].mean(), returns[weights < 0].mean()
    if not np.isfinite([long_return, short_return]).all():
        raise ValueError("A portfolio side has no available returns.")
    return {
        "long_return": long_return, "short_return": short_return, "gross_return": long_return - short_return,
        "n_side": int((weights > 0).sum()), "missing_selected": int(returns.isna().sum()),
    }

## 36. Portfolio Performance and Transaction Costs

- **Turnover** is the traded notional per $1 long and $1 short: the sum of absolute changes in signed weights from the
  previous month. The first month includes opening the positions. Drift of weights between rebalancing dates is ignored.
- **Cost** is turnover × `COST_BPS` / 10,000. The 10 basis points are an illustrative assumption.
- **Annualization**: mean × 12, volatility and Sharpe ratio × √12.
- **Cumulative performance** is the sum of monthly returns with constant $1 sides. Returns are not compounded, because
  a long-short month can lose more than 100%. Drawdown is measured on this sum.

In [ ]:
def traded_notional(weights, previous_weights):
    """Sum of absolute weight changes, including stocks entering and leaving."""
    return weights.sub(previous_weights, fill_value=0).abs().sum()


def evaluate_forecast(stocks, decile=DECILE, cost_bps=COST_BPS):
    """Monthly metrics and portfolio results of one forecast."""
    rows = []
    previous_weights = pd.Series(dtype="float64", index=pd.Index([], dtype="int64"))
    for month, month_stocks in stocks.groupby("predicted_month"):
        universe = month_stocks[month_stocks["in_common"]]
        evaluated = universe[universe["has_actual"]]
        weights = long_short_weights(universe, decile)
        turnover = traded_notional(weights, previous_weights)
        row = {
            "predicted_month": month, "n_labeled": int(month_stocks["has_actual"].sum()), "n_evaluated": len(evaluated),
            **squared_error_metrics(evaluated), "rank_ic": rank_ic(evaluated),
            **portfolio_returns(universe, weights), "turnover": turnover, "cost": turnover * cost_bps / 10_000,
        }
        row["net_return"] = row["gross_return"] - row["cost"]
        rows.append(row)
        previous_weights = weights
    return pd.DataFrame(rows)

In [ ]:
def sharpe_ratio(returns, axis=-1):
    """Annualized Sharpe ratio; undefined when returns do not vary."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.mean(returns, axis=axis) / np.std(returns, axis=axis, ddof=1) * np.sqrt(12)


def performance(returns):
    """Summary of a monthly return series."""
    cumulative = returns.cumsum()
    drawdown = cumulative - cumulative.cummax().clip(lower=0)
    return {
        "mean_monthly": returns.mean(), "annual_return": 12 * returns.mean(), "monthly_volatility": returns.std(),
        "annual_volatility": np.sqrt(12) * returns.std(), "sharpe": sharpe_ratio(returns.to_numpy()),
        "cumulative": cumulative.iloc[-1], "max_drawdown": drawdown.min(),
    }


def summarize(months):
    """Summary of one forecast from its monthly results."""
    months = months.sort_values("predicted_month")
    gross, net = performance(months["gross_return"]), performance(months["net_return"])
    return pd.Series({
        "mse": months["mse"].mean(), "oos_r2": 1 - months["mse"].mean() / months["zero_mse"].mean(),
        "stock_months": months["n_evaluated"].sum(), "months": len(months),
        "coverage": months["n_evaluated"].sum() / months["n_labeled"].sum(),
        "mean_ic": months["rank_ic"].mean(), "ic_std": months["rank_ic"].std(), "ic_months": months["rank_ic"].count(),
        **{f"gross_{name}": value for name, value in gross.items()},
        "mean_turnover": months["turnover"].mean(), "mean_cost": months["cost"].mean(),
        **{f"net_{name}": value for name, value in net.items()},
        "months_without_position": (months["n_side"] == 0).sum(), "missing_selected": months["missing_selected"].sum(),
        "months_above_100pct": (months["gross_return"].abs() > 1).sum(),
    })

In [ ]:
GROUP = ["country", "window", "strategy", "seed"]

if RUN_FINAL_EVALUATION:
    if not PREDICTIONS_FILE.exists():
        raise FileNotFoundError(f"{PREDICTIONS_FILE} is required. Run Section 31 with RUN_PHASE6_PREDICTIONS = True first.")

    results = []
    for country in LOCAL_COUNTRIES:
        for window in TRAIN_WINDOWS[country]:
            print(f"Evaluating: {country} | {window}", flush=True)
            for strategy, seed in forecast_list():
                months = evaluate_forecast(load_forecast(country, window, strategy, seed))
                results.append(months.assign(country=country, window=window, strategy=strategy, seed=seed))
    monthly = pd.concat(results, ignore_index=True)
    summary = pd.DataFrame(
        [{**dict(zip(GROUP, names)), **summarize(months)} for names, months in monthly.groupby(GROUP)]
    )
    summary["is_main"] = summary["seed"] == summary["strategy"].map(main_seed)
    summary["model"] = summary["strategy"].map(MODEL_NAMES)
else:
    print("Final evaluation: not run (RUN_FINAL_EVALUATION = False)")

In [ ]:
def main_table(columns):
    """Main forecasts in a fixed model order."""
    table = summary[summary["is_main"]].copy()
    table["order"] = table["strategy"].map(list(MODEL_NAMES).index)
    return table.sort_values(["country", "window", "order"])[["country", "window", "model"] + columns]


if RUN_FINAL_EVALUATION:
    print("Prediction accuracy and rank IC:")
    display(main_table(["mse", "oos_r2", "stock_months", "months", "coverage", "mean_ic", "ic_std", "ic_months"]))
    print("Gross long-short performance:")
    display(main_table([
        "gross_mean_monthly", "gross_annual_return", "gross_monthly_volatility", "gross_annual_volatility",
        "gross_sharpe", "gross_cumulative", "gross_max_drawdown",
    ]))
    print(f"Net of {COST_BPS} basis points per dollar traded (assumption):")
    display(main_table([
        "mean_turnover", "mean_cost", "net_mean_monthly", "net_annual_return", "net_sharpe", "net_cumulative",
        "net_max_drawdown",
    ]))

    flags = main_table(["months_without_position", "missing_selected", "months_above_100pct"])
    flags = flags[(flags[["months_without_position", "months_above_100pct"]] > 0).any(axis=1)]
    if not flags.empty:
        print("WARNING: months without a ranking, or with a long-short return beyond +/-100%. Interpret with care:")
        display(flags)

In [ ]:
if RUN_FINAL_EVALUATION:
    print("Variation across the three Transformer seeds (individual seeds, not the seed average):")
    seed_rows = summary[summary["seed"].isin([str(seed) for seed in TRANSFORMER_SEEDS])]
    seed_variation = seed_rows.groupby(["country", "window", "model"])[["oos_r2", "mean_ic", "gross_sharpe"]]
    display(seed_variation.agg(["mean", "std", "min", "max"]))

## 37. Statistical Significance Testing

Every pair of models is compared with a paired moving-block bootstrap: blocks of six consecutive months are resampled,
and the same months are used for both models. Countries and training histories are never mixed.

Differences are expressed as the **improvement of model A over model B**, so a positive value always favors A (for MSE,
the MSE of B minus the MSE of A). P-values are two-sided and use the bootstrap distribution centered at zero. Holm's
procedure is applied to the 21 primary pairs of each country, history and metric; comparisons with the freely tuned
local Transformer form a separate family. A metric undefined in any month for either model is not tested.

In [ ]:
TEST_METRICS = {  # label: (monthly column, sign of improvement, statistic)
    "MSE": ("mse", -1, np.mean), "Rank IC": ("rank_ic", 1, np.mean),
    "Gross return": ("gross_return", 1, np.mean), "Net return": ("net_return", 1, np.mean),
    "Gross Sharpe": ("gross_return", 1, sharpe_ratio), "Net Sharpe": ("net_return", 1, sharpe_ratio),
}


def block_indices(n_months, rng):
    """Month positions of every bootstrap sample, built from blocks of consecutive months."""
    n_blocks = int(np.ceil(n_months / BLOCK_MONTHS))
    starts = rng.integers(0, n_months - BLOCK_MONTHS + 1, size=(BOOTSTRAP_SAMPLES, n_blocks))
    blocks = starts[:, :, None] + np.arange(BLOCK_MONTHS)
    return blocks.reshape(BOOTSTRAP_SAMPLES, -1)[:, :n_months]


def bootstrap_difference(a, b, sign, statistic, indices):
    """Improvement of A over B with a confidence interval and two-sided p-value."""
    undefined = {"improvement": np.nan, "ci_low": np.nan, "ci_high": np.nan, "p_value": np.nan}
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        return undefined
    observed = sign * (statistic(a, axis=-1) - statistic(b, axis=-1))
    samples = sign * (statistic(a[indices], axis=-1) - statistic(b[indices], axis=-1))  # same months for both
    if not (np.isfinite(observed) and np.isfinite(samples).all()):
        return undefined

    tail = (1 - 0.95) / 2
    extreme = np.abs(samples - samples.mean()) >= np.abs(observed)  # distribution centered at zero
    return {
        "improvement": observed, "ci_low": np.quantile(samples, tail), "ci_high": np.quantile(samples, 1 - tail),
        "p_value": (1 + extreme.sum()) / (1 + len(samples)),
    }


def holm_adjust(p_values):
    """Holm-adjusted p-values; undefined tests stay undefined and are not counted."""
    valid = p_values.dropna().sort_values()
    adjusted = (valid * np.arange(len(valid), 0, -1)).cummax().clip(upper=1)
    return adjusted.reindex(p_values.index)

In [ ]:
def compare_models(monthly, country, window):
    """Paired tests of every model pair in one country and training history."""
    is_main = monthly["seed"] == monthly["strategy"].map(main_seed)
    part = monthly[is_main & (monthly["country"] == country) & (monthly["window"] == window)]
    indices = block_indices(part["predicted_month"].nunique(), np.random.default_rng(SEED))

    rows = []
    for metric, (column, sign, statistic) in TEST_METRICS.items():
        series = part.pivot(index="predicted_month", columns="strategy", values=column).sort_index()
        for a, b in combinations(MODEL_NAMES, 2):
            test = bootstrap_difference(series[a].to_numpy(), series[b].to_numpy(), sign, statistic, indices)
            family = "secondary" if "local" in (a, b) else "primary"
            rows.append({"country": country, "window": window, "metric": metric, "family": family, "a": a, "b": b, **test})
    return pd.DataFrame(rows)


def label_outcome(row):
    if pd.isna(row["p_adjusted"]):
        return "not available"
    if row["p_adjusted"] >= SIGNIFICANCE_LEVEL:
        return "no significant difference"
    return "A better" if row["improvement"] > 0 else "B better"

In [ ]:
if RUN_FINAL_EVALUATION:
    print("Running paired block-bootstrap tests...")
    comparisons = pd.concat(
        [compare_models(monthly, country, window) for country in LOCAL_COUNTRIES for window in TRAIN_WINDOWS[country]],
        ignore_index=True,
    )
    family_columns = ["country", "window", "metric", "family"]
    comparisons["p_adjusted"] = comparisons.groupby(family_columns)["p_value"].transform(holm_adjust)
    comparisons["significant"] = comparisons["p_adjusted"] < SIGNIFICANCE_LEVEL
    comparisons["outcome"] = comparisons.apply(label_outcome, axis=1)
    comparisons["model_a"] = comparisons["a"].map(MODEL_NAMES)
    comparisons["model_b"] = comparisons["b"].map(MODEL_NAMES)

    print("Primary comparisons by outcome:")
    primary = comparisons[comparisons["family"] == "primary"]
    display(primary.pivot_table(index="metric", columns="outcome", values="a", aggfunc="count", fill_value=0))

## 38. Cross-Market Model Comparisons

Each table reports the improvement of model A over model B. The matched local Transformer is the transfer benchmark.
Direct transfer is one set of U.S. models evaluated against each local history, not three separately trained models.

In [ ]:
def pair_table(pairs, metrics=("MSE", "Rank IC", "Net return")):
    """Test results of selected model pairs."""
    in_pairs = np.array([pair in pairs for pair in zip(comparisons["a"], comparisons["b"])])
    selected = comparisons[in_pairs & comparisons["metric"].isin(metrics).to_numpy()]
    return selected[[
        "country", "window", "metric", "model_a", "model_b", "improvement", "ci_low", "ci_high", "p_value",
        "p_adjusted", "significant", "outcome",
    ]]


if RUN_FINAL_EVALUATION:
    print("A. Cross-stock attention: local Transformer versus Elastic Net and XGBoost")
    display(pair_table([("localmatched", "elastic_net"), ("localmatched", "xgboost")]))
    print("B. U.S. pretraining: direct transfer and fine-tuning versus the local Transformer")
    display(pair_table([("direct", "localmatched"), ("finetuned", "localmatched")]))

In [ ]:
if RUN_FINAL_EVALUATION:
    print("C. Transfer benefit by length of local training history (improvement over the matched local Transformer)")
    transfer = pair_table([("direct", "localmatched"), ("finetuned", "localmatched"), ("joint", "localmatched")])
    display(transfer.pivot(index=["country", "metric", "model_a"], columns="window", values="improvement"))
    display(transfer.pivot(index=["country", "metric", "model_a"], columns="window", values="outcome"))

    print("D. Transfer versus pooling: fine-tuned versus joint Transformer")
    display(pair_table([("finetuned", "joint")]))

In [ ]:
if RUN_FINAL_EVALUATION:
    print("E. Ensemble versus each individual model")
    display(pair_table([("ensemble", model) for model in PRIMARY_MODELS if model != "ensemble"]))
    print("Separate comparison: matched versus freely tuned local Transformer")
    display(pair_table([("localmatched", "local")]))

## 39. Research Results and Visualizations

Essential tables, the monthly series and the figures are saved in `evaluation/` inside the results folder. Predictions
and model files are never changed.

In [ ]:
if RUN_FINAL_EVALUATION:
    EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    monthly.to_parquet(EVALUATION_DIR / "monthly_results.parquet", index=False)
    summary.to_csv(EVALUATION_DIR / "summary.csv", index=False)
    comparisons.to_csv(EVALUATION_DIR / "model_comparisons.csv", index=False)
    print(f"Saved evaluation tables in {EVALUATION_DIR}")

In [ ]:
WINDOW_NAMES = {"train_1y": "1 year", "train_3y": "3 years", "train_5y": "5 years"}


def finish_figure(figure, name):
    figure.tight_layout()
    figure.savefig(EVALUATION_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


def bar_figure(column, label, name, symlog=False):
    """Grouped bars by model and training history, one panel per country."""
    figure, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for axis, country in zip(axes, LOCAL_COUNTRIES):
        table = summary[summary["is_main"] & (summary["country"] == country)]
        table = table.pivot(index="strategy", columns="window", values=column).loc[PRIMARY_MODELS, list(WINDOW_NAMES)]
        table.rename(index=MODEL_NAMES, columns=WINDOW_NAMES).plot.barh(ax=axis)
        axis.axvline(0, color="black", linewidth=0.8)
        if symlog:
            axis.set_xscale("symlog", linthresh=0.01)  # large negative values remain visible
        axis.set(title=country, xlabel=label, ylabel="")
        axis.legend(title="Local history")
    finish_figure(figure, name)


if RUN_FINAL_EVALUATION:
    bar_figure("oos_r2", "Out-of-sample R² (symmetric log scale)", "oos_r2", symlog=True)
    bar_figure("mean_ic", "Mean monthly rank IC", "rank_ic")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Cumulative gross long-short performance (sum of monthly returns, constant $1 sides)
    figure, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
    main_monthly = monthly[monthly["seed"] == monthly["strategy"].map(main_seed)]
    for row, country in enumerate(LOCAL_COUNTRIES):
        for column, window in enumerate(WINDOW_NAMES):
            axis = axes[row, column]
            part = main_monthly[(main_monthly["country"] == country) & (main_monthly["window"] == window)]
            series = part.pivot(index="predicted_month", columns="strategy", values="gross_return").sort_index()
            for strategy in PRIMARY_MODELS:
                axis.plot(series.index, series[strategy].cumsum(), label=MODEL_NAMES[strategy], linewidth=1.3)
            axis.axhline(0, color="black", linewidth=0.8)
            axis.set(title=f"{country}, {WINDOW_NAMES[window]}", ylabel="Sum of monthly returns")
    axes[0, 2].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    finish_figure(figure, "cumulative_long_short")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Gross versus net annualized Sharpe ratios
    figure, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
    for row, country in enumerate(LOCAL_COUNTRIES):
        for column, window in enumerate(WINDOW_NAMES):
            table = summary[summary["is_main"] & (summary["country"] == country) & (summary["window"] == window)]
            table = table.set_index("strategy").loc[PRIMARY_MODELS, ["gross_sharpe", "net_sharpe"]]
            table.columns = ["Gross", f"Net ({COST_BPS} bps)"]
            table.rename(index=MODEL_NAMES).plot.barh(ax=axes[row, column], legend=(row == 0 and column == 0))
            axes[row, column].axvline(0, color="black", linewidth=0.8)
            axes[row, column].set(title=f"{country}, {WINDOW_NAMES[window]}", xlabel="Annualized Sharpe ratio", ylabel="")
    finish_figure(figure, "gross_net_sharpe")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Rank IC improvement over the matched local Transformer, with 95% bootstrap intervals
    figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
    for axis, country in zip(axes, LOCAL_COUNTRIES):
        for shift, strategy in zip([-0.1, 0, 0.1], ["direct", "finetuned", "joint"]):
            part = comparisons[
                (comparisons["country"] == country) & (comparisons["metric"] == "Rank IC")
                & (comparisons["a"] == strategy) & (comparisons["b"] == "localmatched")
            ].set_index("window").loc[list(WINDOW_NAMES)]
            errors = np.array([part["improvement"] - part["ci_low"], part["ci_high"] - part["improvement"]])
            errors = np.clip(errors, 0, None)  # a percentile interval need not contain the estimate
            positions = np.arange(len(part)) + shift
            axis.errorbar(positions, part["improvement"], yerr=errors, marker="o", capsize=4, label=MODEL_NAMES[strategy])
        axis.axhline(0, color="black", linewidth=0.8)
        axis.set_xticks(range(len(WINDOW_NAMES)), list(WINDOW_NAMES.values()))
        axis.set(title=country, xlabel="Local training history", ylabel="Rank IC improvement")
        axis.legend()
    finish_figure(figure, "transfer_benefit")

## 40. Evaluation Verification

Configuration checks always run. The small artificial checks run with `RUN_SMOKE_TESTS`, and the result checks run with
`RUN_FINAL_EVALUATION`.

In [ ]:
assert set(MODEL_NAMES) == set(ALL_STRATEGIES) | {"ensemble"} and "ensemble" not in PRIMARY_STRATEGIES
assert len(list(combinations(PRIMARY_MODELS, 2))) == 21
assert 0 < DECILE <= 0.5 and COST_BPS >= 0 and BLOCK_MONTHS == 6
assert PERIODS["validation"][1] < PERIODS["test"][0]  # selection ends before the test period
print("Evaluation configuration verified.")

In [ ]:
if RUN_SMOKE_TESTS:
    # Deciles, signs and stable ties
    example = pd.DataFrame({"id": np.arange(20), "prediction": np.arange(20) / 100, "actual_return": np.arange(20) / 10})
    weights = long_short_weights(example)
    assert set(weights[weights > 0].index) == {18, 19} and set(weights[weights < 0].index) == {0, 1}
    assert np.isclose(weights.sum(), 0) and np.isclose(weights.abs().sum(), 2)
    assert np.isclose(portfolio_returns(example, weights)["gross_return"], 1.85 - 0.05)
    assert long_short_weights(example.assign(prediction=0.0)).empty
    tied = long_short_weights(example.assign(prediction=[0.0] * 19 + [1.0]))
    assert len(tied) == 4 and tied.index.is_unique

    # Turnover: opening positions trades $2; replacing one of two longs trades 0.5 + 0.5
    nothing = weights.iloc[:0]
    changed = weights.rename(index={19: 17})
    assert np.isclose(traded_notional(weights, nothing), 2) and np.isclose(traded_notional(weights, weights), 0)
    assert np.isclose(traded_notional(changed, weights), 1) and np.isclose(traded_notional(nothing, weights), 2)

    # Rank IC is a rank correlation, and undefined without variation
    assert np.isclose(rank_ic(example.assign(actual_return=np.exp(example["actual_return"]))), 1)
    assert np.isnan(rank_ic(example.assign(prediction=0.0)))

    # Holm's procedure, and paired blocks: a constant difference has no sampling variation
    assert np.allclose(holm_adjust(pd.Series([0.01, 0.04, 0.03])), [0.03, 0.06, 0.06])
    indices = block_indices(72, np.random.default_rng(SEED))
    assert indices.shape == (BOOTSTRAP_SAMPLES, 72)
    assert (np.diff(indices.reshape(BOOTSTRAP_SAMPLES, -1, BLOCK_MONTHS), axis=2) == 1).all()
    series = np.random.default_rng(SEED).normal(size=72)
    test = bootstrap_difference(series + 0.5, series, 1, np.mean, indices)
    assert np.allclose([test["improvement"], test["ci_low"], test["ci_high"]], 0.5)
    print("Portfolio, turnover, rank IC, bootstrap and Holm calculations verified.")

In [ ]:
if RUN_FINAL_EVALUATION:
    # Dates and unique identifiers
    assert not monthly.duplicated(GROUP + ["predicted_month"]).any()
    assert monthly["predicted_month"].min() == PERIODS["test"][0] and monthly["predicted_month"].max() == PERIODS["test"][1]
    assert (monthly.groupby(GROUP).size() == 72).all()

    # Every model of an experiment is evaluated on the same stock-months and zero benchmark
    by_month = monthly.groupby(["country", "window", "predicted_month"])
    assert (by_month["n_evaluated"].nunique() == 1).all()
    assert np.allclose(by_month["zero_mse"].max(), by_month["zero_mse"].min())

    # R², portfolio signs, costs and seed labels
    average = monthly.groupby(GROUP)[["mse", "zero_mse"]].mean()
    saved_r2 = summary.set_index(GROUP)["oos_r2"].reindex(average.index)
    assert np.allclose(saved_r2, 1 - average["mse"] / average["zero_mse"])
    assert np.allclose(monthly["gross_return"], monthly["long_return"] - monthly["short_return"])
    assert np.allclose(monthly["net_return"], monthly["gross_return"] - monthly["turnover"] * COST_BPS / 10_000)
    assert monthly["turnover"].between(0, 4 + 1e-9).all()
    assert summary["is_main"].sum() == len(LOCAL_COUNTRIES) * 3 * len(MODEL_NAMES)
    assert not summary.loc[summary["is_main"], "seed"].isin([str(seed) for seed in TRANSFORMER_SEEDS]).any()

    # Multiple comparisons
    tested = comparisons.dropna(subset=["p_value"])
    assert (tested["p_adjusted"] >= tested["p_value"] - 1e-12).all() and tested["p_adjusted"].between(0, 1).all()
    assert (comparisons["significant"] == (comparisons["p_adjusted"] < SIGNIFICANCE_LEVEL)).all()
    print("Evaluation results verified.")

Phase 7 — End

Phase 8 — Start

## 41. Version 2: Purpose and Design

Version 1 (Git tag `v1.0`) is complete and frozen; its sections above define the shared data preparation, the control
model and the original metrics. Version 2 is an **exploratory follow-up designed after the Version 1 test results were
seen**. It reuses January 2020 – December 2025, so that period is **not a fresh, untouched holdout** for Version 2, and
Version 2 results can suggest but not confirm an improvement. All fitting and selection still use 2000–2019 only.

Every variant changes **one setting** of its reference variant, with the same seeds, search size and architecture size, so
a difference can be attributed to that setting. `combined` is the only variant with several changes.

| Variant | Reference | Single change |
|---|---|---|
| `control` | – | Version 1 Transformer, MSE and raw targets, with the small Version 2 search |
| `huber` | `control` | Huber loss |
| `clip` | `control` | Training targets clipped at training-period quantiles |
| `clip_month` | `control` | Training targets clipped within each training month |
| `robust` | `huber` | Adds quantile clipping (Huber + clipping) |
| `select_clip`, `select_ic` | `robust` | Validation selection by clipped MSE or by rank IC |
| `rank` | `robust` | Adds the ranking objective |
| `missing` | `robust` | Adds missing-value indicators |
| `mlp`, `gated` | `robust` | Stockwise MLP, or MLP with gated attention |
| `dropout`, `gradclip`, `schedule`, `earlystop` | `robust` | One regularization change each |
| `joint50`, `joint75` | `robust` | Joint training with 50% or 75% local months (joint strategy only) |
| `combined` | `robust` | Rank loss, indicators, gated model, gradient clipping, schedule, rank-IC selection, 50% local |

In [ ]:
BASE_SPEC = {
    "model": "transformer",     # "transformer", "mlp" or "gated"
    "target": "raw",            # training targets only: "raw", "clip" or "clip_month"
    "loss": "mse",              # "mse" or "huber"
    "huber_delta": 0.10,        # errors beyond a 10% monthly return count linearly
    "regression_weight": 1.0,
    "rank_weight": 0.0,         # above zero adds the ranking objective
    "missing_flags": False,     # True adds one missing-value indicator per predictor
    "local_share": None,        # share of local months in joint training; None keeps the original weighting
    "selection": "mse",         # validation score: "mse", "clipped_mse" or "rank_ic"
    "dropout": 0.10,
    "clip_norm": None,          # maximum gradient norm
    "schedule": "none",         # "plateau" halves the learning rate when validation stalls
    "early_stopping": "v1",     # "v1" or "patient", see EARLY_STOPPING
}
EARLY_STOPPING = {
    "v1": {"patience": PATIENCE, "min_epochs": 0, "min_delta": 0.0},
    "patient": {"patience": 15, "min_epochs": 10, "min_delta": 1e-4},  # min_delta is relative
}
V2_TRANSFORMER_STRATEGIES = ["localmatched", "direct", "finetuned", "joint"]
VARIANTS = {  # name: (reference, changes, strategies)
    "control": (None, {}, V2_TRANSFORMER_STRATEGIES),
    "huber": ("control", {"loss": "huber"}, V2_TRANSFORMER_STRATEGIES),
    "clip": ("control", {"target": "clip"}, V2_TRANSFORMER_STRATEGIES),
    "clip_month": ("control", {"target": "clip_month"}, V2_TRANSFORMER_STRATEGIES),
    "robust": ("huber", {"target": "clip"}, V2_TRANSFORMER_STRATEGIES),
    "select_clip": ("robust", {"selection": "clipped_mse"}, V2_TRANSFORMER_STRATEGIES),
    "select_ic": ("robust", {"selection": "rank_ic"}, V2_TRANSFORMER_STRATEGIES),
    "rank": ("robust", {"rank_weight": 1.0}, V2_TRANSFORMER_STRATEGIES),
    "missing": ("robust", {"missing_flags": True}, V2_TRANSFORMER_STRATEGIES),
    "mlp": ("robust", {"model": "mlp"}, V2_TRANSFORMER_STRATEGIES),
    "gated": ("robust", {"model": "gated"}, V2_TRANSFORMER_STRATEGIES),
    "dropout": ("robust", {"dropout": 0.30}, V2_TRANSFORMER_STRATEGIES),
    "gradclip": ("robust", {"clip_norm": 1.0}, V2_TRANSFORMER_STRATEGIES),
    "schedule": ("robust", {"schedule": "plateau"}, V2_TRANSFORMER_STRATEGIES),
    "earlystop": ("robust", {"early_stopping": "patient"}, V2_TRANSFORMER_STRATEGIES),
    "joint50": ("robust", {"local_share": 0.50}, ["joint"]),
    "joint75": ("robust", {"local_share": 0.75}, ["joint"]),
    "combined": ("robust", {
        "rank_weight": 1.0, "missing_flags": True, "model": "gated", "clip_norm": 1.0, "schedule": "plateau",
        "selection": "rank_ic", "local_share": 0.50,
    }, V2_TRANSFORMER_STRATEGIES),
}
BASE_BASELINE_SPEC = {
    "target": "raw", "grid": "v1", "objective": "squared_error", "selection": "mse", "missing_flags": False,
}
BASELINE_VARIANTS = {  # name: (reference, changes, model families)
    "b_control": (None, {}, ["elastic_net", "xgboost"]),
    "b_clip": ("b_control", {"target": "clip"}, ["elastic_net", "xgboost"]),
    "b_grid": ("b_clip", {"grid": "v2"}, ["elastic_net", "xgboost"]),
    "b_ic": ("b_grid", {"selection": "rank_ic"}, ["elastic_net", "xgboost"]),
    "b_missing": ("b_grid", {"missing_flags": True}, ["elastic_net", "xgboost"]),
    "b_huber": ("b_grid", {"objective": "pseudo_huber"}, ["xgboost"]),
}
ALL_VARIANTS = {**VARIANTS, **BASELINE_VARIANTS}
ENSEMBLE_BASELINES = "b_grid"  # baselines used in every Version 2 ensemble, so only the Transformers differ


def variant_spec(variant):
    """Complete settings of a variant: its reference with the changes applied."""
    reference, changes, _ = ALL_VARIANTS[variant]
    if reference is None:
        return dict(BASE_SPEC if variant in VARIANTS else BASE_BASELINE_SPEC)
    return {**variant_spec(reference), **changes}

In [ ]:
for name, (reference, changes, _) in ALL_VARIANTS.items():
    base = BASE_SPEC if name in VARIANTS else BASE_BASELINE_SPEC
    assert set(changes) <= set(base), f"{name}: unknown settings"
    assert name == "combined" or len(changes) == (0 if reference is None else 1), f"{name}: more than one change"
    if reference is not None:
        assert all(variant_spec(name)[key] != variant_spec(reference)[key] for key in changes), f"{name}: no change"

unknown = set(V2_VARIANTS) - set(ALL_VARIANTS)
if unknown:
    raise ValueError(f"Unknown V2_VARIANTS: {sorted(unknown)}. Choose from {list(ALL_VARIANTS)}.")
if not set(V2_STAGES) <= {"train", "predict", "evaluate", "compare"}:
    raise ValueError(f"Unknown V2_STAGES: {V2_STAGES}")
if not set(V2_STRATEGIES) <= set(V2_TRANSFORMER_STRATEGIES):
    raise ValueError(f"Unknown V2_STRATEGIES: {V2_STRATEGIES}")


def v2_stage(stage):
    """True if the stage is switched on for this run."""
    if RUN_V2 and stage in V2_STAGES:
        return True
    print(f"Version 2 {stage}: not run (RUN_V2 = {RUN_V2}, V2_STAGES = {V2_STAGES})")
    return False


print(f"Version 2 variants: {len(VARIANTS)} Transformer and {len(BASELINE_VARIANTS)} baseline configurations verified.")

## 42. Version 2 Outputs and Compatibility

Outputs go to `results_v2/<variant>/` (`results_v2_{N}epochs/` for shortened runs), never to the Version 1 folder. Each
saved object carries a fingerprint of all settings that affect it: data, predictors, periods, variant, architecture, search
grid, seeds and epochs. An object with another fingerprint, or from Version 1, raises an error instead of being loaded.
Each saved model also has a unique `run_id`; fine-tuned models and predictions record the `run_id` of their source.

In [ ]:
V2_CODE_VERSION = 1  # increase after any change to Version 2 training code, so older models are rejected
V2_DIR = PROJECT_ROOT / ("results_v2" if RESEARCH_EPOCHS == 100 else f"results_v2_{RESEARCH_EPOCHS}epochs")
V1_DIR = Path(os.environ.get("CMP_V1_RESULTS", PROJECT_ROOT / "results"))  # read only, for comparison
assert V2_DIR.resolve() not in [V1_DIR.resolve(), RESULTS_DIR.resolve()], "Version 2 must not use the Version 1 folder."

TARGET_QUANTILES = (0.005, 0.995)        # clip: pooled over the training window of one country
MONTH_QUANTILES = (0.01, 0.99)           # clip_month: within each training month
V2_EMBEDDING_DIM, V2_BLOCKS = 64, 2      # the architecture selected for the U.S. in Version 1
V2_GRID = [
    {"embedding_dim": V2_EMBEDDING_DIM, "n_blocks": V2_BLOCKS, "learning_rate": rate, "weight_decay": decay}
    for rate, decay in product([3e-4, 1e-3], [1e-4, 1e-2])
]
V2_FINE_TUNING = list(product([0.3, 0.1], [1e-4, 1e-2]))  # learning-rate factor and weight decay

V2_ELASTIC_NET = list(product([0.5, 0.2, 0.1, 0.03, 0.01], [0.1, 0.5, 0.9]))  # alpha as a share of alpha_max, l1_ratio
V2_XGBOOST_GRID = [
    {"max_depth": depth, "learning_rate": rate, "n_estimators": trees, "min_child_weight": weight,
     "subsample": 0.7, "colsample_bytree": 0.5, "reg_lambda": 10.0}
    for depth, rate, trees, weight in product([2, 4], [0.02, 0.05], [100, 300], [20, 200])
]
assert max(len(V2_GRID), len(V2_FINE_TUNING), len(V2_ELASTIC_NET), len(V2_XGBOOST_GRID)) <= MAX_SETTINGS

In [ ]:
def git_commit():
    """Commit of the running code, from the submission script or the .git folder ("unknown" if neither is available)."""
    commit = os.environ.get("CMP_GIT_COMMIT", "")
    if commit not in ["", "unknown"]:
        return commit
    git = PROJECT_ROOT / ".git"
    head = git / "HEAD"
    if not head.exists():
        return "unknown"
    reference = head.read_text().strip()
    if not reference.startswith("ref: "):
        return reference
    reference = reference[5:]
    if (git / reference).exists():
        return (git / reference).read_text().strip()
    packed = git / "packed-refs"
    for line in packed.read_text().splitlines() if packed.exists() else []:
        if line.endswith(f" {reference}"):
            return line.split()[0]
    return "unknown"


def feature_hash(names):
    """Hash of the ordered predictor names."""
    return hashlib.sha256("\n".join(names).encode()).hexdigest()


def dataset_fingerprint(path):
    """Hash of a Parquet footer (rows, row groups, statistics) and the file size; quick but not a full checksum."""
    size = path.stat().st_size
    with open(path, "rb") as file:
        file.seek(-8, os.SEEK_END)
        length = int.from_bytes(file.read(4), "little")
        file.seek(-8 - length, os.SEEK_END)
        footer = file.read(length)
    return hashlib.sha256(footer + str(size).encode()).hexdigest()


def environment_details():
    """Where and with what a result was produced; not part of the fingerprint."""
    packages = ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "xgboost", "torch"]
    cuda = torch.cuda.is_available()
    return {
        "python": sys.version.split()[0], "platform": sys.platform, "device": str(DEVICE),
        "git_commit": git_commit(), "git_dirty": os.environ.get("CMP_GIT_DIRTY", "unknown"),
        "notebook_sha256": os.environ.get("CMP_NOTEBOOK_SHA256", "unknown"),
        "slurm_job": os.environ.get("SLURM_JOB_ID", ""), "slurm_job_name": os.environ.get("SLURM_JOB_NAME", ""),
        "host": os.environ.get("SLURMD_NODENAME", ""), "recorded_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "cuda_build": str(torch.version.cuda), "cudnn": str(torch.backends.cudnn.version()) if cuda else "",
        "gpu": torch.cuda.get_device_name() if cuda else "", "n_features": len(selected_features),
        "features_sha256": feature_hash(selected_features),
        "data_fingerprints": {country: dataset_fingerprint(path) for country, path in FILES.items()},
        **{name: version(name) for name in packages},
    }


def variant_config(variant):
    """Everything that determines the results of a variant."""
    config = {
        "version": "v2", "code_version": V2_CODE_VERSION, "variant": variant, "spec": variant_spec(variant),
        "max_epochs": RESEARCH_EPOCHS, "features": selected_features, "data_rows": {c: len(data[c]) for c in COUNTRIES},
        "periods": {name: [str(date.date()) for date in dates] for name, dates in PERIODS.items()},
        "target_quantiles": TARGET_QUANTILES, "month_quantiles": MONTH_QUANTILES,
    }
    if variant in VARIANTS:
        config.update(
            seeds=TRANSFORMER_SEEDS, n_heads=N_HEADS, feedforward_dim=FEEDFORWARD_DIM, grid=V2_GRID,
            fine_tuning=V2_FINE_TUNING, early_stopping=EARLY_STOPPING[variant_spec(variant)["early_stopping"]],
        )
    else:
        grids = {
            "v1": [ELASTIC_NET_GRID, XGBOOST_GRID], "v2": [V2_ELASTIC_NET, V2_XGBOOST_GRID],
        }
        config.update(seed=SEED, grids=grids[variant_spec(variant)["grid"]])
    return config


def fingerprint(variant):
    """Short hash of the variant configuration."""
    text = json.dumps(variant_config(variant), sort_keys=True)
    return hashlib.sha256(text.encode()).hexdigest()[:16]


def variant_dir(variant):
    return V2_DIR / variant


def start_variant(variant):
    """Create the output folder, or confirm that an existing one has the same configuration."""
    folder = variant_dir(variant)
    (folder / "models").mkdir(parents=True, exist_ok=True)
    (folder / "predictions").mkdir(exist_ok=True)
    path = folder / "config.json"
    if path.exists():
        saved = json.loads(path.read_text())["fingerprint"]
        if saved != fingerprint(variant):
            raise ValueError(f"{folder} holds results of another configuration ({saved}). Move it before rerunning.")
    else:
        text = json.dumps(
            {"fingerprint": fingerprint(variant), **variant_config(variant), "environment": environment_details()}, indent=2
        )
        save_safely(path, lambda temporary: temporary.write_text(text))


def check_identity(saved, variant, name):
    """Reject objects from Version 1 or from another Version 2 configuration."""
    if saved.get("version") != "v2" or saved.get("variant") != variant:
        raise ValueError(f"{name}: not a Version 2 object of variant '{variant}'.")
    if saved.get("fingerprint") != fingerprint(variant):
        raise ValueError(f"{name}: saved with another configuration ({saved.get('fingerprint')}). Move it before rerunning.")

In [ ]:
def record_validation(variant, table, strategy, country, window):
    """Add one experiment to the variant's validation file, replacing its earlier rows."""
    path = variant_dir(variant) / "validation.csv"
    table = table.assign(variant=variant, strategy=strategy, country=country, window=window)
    if path.exists():
        saved = pd.read_csv(path)
        same = (saved["strategy"] == strategy) & (saved["country"] == country) & (saved["window"] == window)
        table = pd.concat([saved[~same], table], ignore_index=True)
    save_safely(path, lambda temporary: table.to_csv(temporary, index=False))


def is_recorded(variant, strategy, country, window):
    path = variant_dir(variant) / "validation.csv"
    if not path.exists():
        return False
    saved = pd.read_csv(path)
    return bool(((saved["strategy"] == strategy) & (saved["country"] == country) & (saved["window"] == window)).any())

## 43. Training Targets, Losses and Validation Scores

- **Clipping** changes training targets only. `clip` uses the 0.5% and 99.5% quantiles of the labeled returns in the
  training window of that country; `clip_month` uses the 1% and 99% quantiles of each training month. Validation and test
  returns are never modified for the reported metrics.
- **Huber loss** is scaled to equal the squared error for errors below `huber_delta`.
- **Ranking objective**: one minus the correlation, within a month, between predictions and the ranks of actual returns. It
  is differentiable in the predictions and unaffected by outliers. When it is used, the regression loss is divided by the
  training loss of a zero forecast, so both parts start near one.
- **Validation scores** (2018–2019) are recorded for every epoch: raw MSE, MSE with returns clipped at the *training*
  quantiles, and mean monthly rank IC. `selection` decides which one chooses epochs and settings.

In [ ]:
def clip_limits(country, window):
    """Training-window quantiles of the labeled returns of one country."""
    if window == "test" or window == "validation":
        raise ValueError("Clipping limits may use training periods only.")
    df = data[country]
    targets = df.loc[in_period(df["predicted_month"], window), TARGET].dropna()
    return tuple(float(value) for value in targets.quantile(list(TARGET_QUANTILES)))


def attach_targets(months, spec, limits):
    """Add training targets and standardized return ranks; the original returns stay in `y`."""
    prepared = []
    for month in months:
        labeled = month["labeled"]
        returns = month["y"][labeled]
        target = month["y"].clone()
        if len(returns) and spec["target"] == "clip":
            target = target.clamp(*limits[month["info"]["country"].iloc[0]])
        elif len(returns) and spec["target"] == "clip_month":
            low, high = np.quantile(returns.numpy(), MONTH_QUANTILES)
            target = target.clamp(float(low), float(high))
        ranks = pd.Series(returns.numpy()).rank().to_numpy()
        ranks = ranks - ranks.mean() if len(ranks) else ranks
        ranks = ranks / max(np.sqrt((ranks ** 2).sum()), 1e-12)  # centered, unit length
        prepared.append({**month, "target": target, "ranks": torch.tensor(ranks, dtype=torch.float32)})
    return prepared


def regression_loss(prediction, target, spec):
    if spec["loss"] == "huber":
        return 2 * F.huber_loss(prediction, target, delta=spec["huber_delta"])
    return F.mse_loss(prediction, target)


def ranking_loss(prediction, ranks):
    """One minus the correlation between predictions and (standardized) return ranks."""
    centered = prediction - prediction.mean()
    return 1 - (centered * ranks).sum() / torch.sqrt((centered ** 2).sum() + 1e-8)


def zero_forecast_loss(months, spec):
    """Average monthly regression loss of a zero forecast on the training months."""
    losses = [
        regression_loss(torch.zeros_like(m["target"][m["labeled"]]), m["target"][m["labeled"]], spec).item() for m in months
    ]
    return float(np.mean(losses))


def month_loss(model, month, spec, scale):
    """Training loss of one month, using labeled stocks only."""
    labeled = month["labeled"].to(DEVICE)
    prediction = model(month["X"].to(DEVICE))[labeled]
    loss = regression_loss(prediction, month["target"].to(DEVICE)[labeled], spec)
    if spec["rank_weight"] == 0:
        return loss
    ranking = ranking_loss(prediction, month["ranks"].to(DEVICE))
    return spec["regression_weight"] * loss / scale + spec["rank_weight"] * ranking

In [ ]:
def score_forecasts(predictions, returns, limits):
    """Monthly-equal validation scores from lists of arrays, one per month."""
    rows = []
    for prediction, actual in zip(predictions, returns):
        rows.append({
            "mse": np.mean((actual - prediction) ** 2),
            "clipped_mse": np.mean((np.clip(actual, *limits) - prediction) ** 2),
            "rank_ic": rank_ic(pd.DataFrame({"prediction": prediction, "actual_return": actual})),
        })
    rows = pd.DataFrame(rows)
    # skipna=False: a month without any ranking makes the rank IC undefined
    return {name: float(rows[name].mean(skipna=False)) for name in ["mse", "clipped_mse", "rank_ic"]}


def validation_scores(model, months, limits):
    """Scores of a neural model on validation months, without gradients."""
    model.eval()
    predictions, returns = [], []
    with torch.inference_mode():
        for month in months:
            if month["labeled"].any():
                try:
                    prediction = model(month["X"].to(DEVICE)).cpu()
                except RuntimeError as error:
                    raise device_error(month, error) from error
                predictions.append(prediction[month["labeled"]].double().numpy())
                returns.append(month["y"][month["labeled"]].double().numpy())
    if not returns:
        raise ValueError("No labeled months are available for validation.")
    return score_forecasts(predictions, returns, limits)


def selection_value(scores, selection):
    """Value to minimize; undefined scores are never selected."""
    value = -scores["rank_ic"] if selection == "rank_ic" else scores[selection]
    return value if np.isfinite(value) else np.inf

## 44. Version 2 Architectures

- **`transformer`**: the Version 1 model, unchanged (control).
- **`mlp`**: residual feed-forward blocks applied to each stock separately; no information from other stocks.
- **`gated`**: the same stockwise blocks, each followed by cross-stock attention multiplied by a learnable gate. Gates start
  at zero, so training begins with the MLP and adds cross-stock information only where it helps.

Missing-value indicators, when used, are appended to the ranked predictors: 1 if the original value is missing in that
country-month and 0 otherwise. All 126 indicators are kept in every country, and they use no information from other months.

In [ ]:
class ResidualMLPBlock(nn.Module):
    """Feed-forward layer with a residual connection, applied to each stock separately."""

    def __init__(self, dim, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.Sequential(
            nn.Linear(dim, FEEDFORWARD_DIM), nn.ReLU(), nn.Dropout(dropout), nn.Linear(FEEDFORWARD_DIM, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.layers(self.norm(x))


class GatedAttention(nn.Module):
    """Attention across the stocks of one month, scaled by a learnable gate that starts at zero."""

    def __init__(self, dim, dropout):
        super().__init__()
        self.dropout_rate = dropout
        self.norm = nn.LayerNorm(dim)
        self.query_key_value = nn.Linear(dim, 3 * dim)
        self.output = nn.Linear(dim, dim)
        self.gate = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        n_stocks, dim = x.shape
        query, key, value = [
            part.view(1, n_stocks, N_HEADS, dim // N_HEADS).transpose(1, 2)
            for part in self.query_key_value(self.norm(x)).chunk(3, dim=-1)
        ]
        attended = F.scaled_dot_product_attention(
            query, key, value, dropout_p=self.dropout_rate if self.training else 0.0
        )
        return x + torch.tanh(self.gate) * self.output(attended.transpose(1, 2).reshape(n_stocks, dim))


class StockwiseNetwork(nn.Module):
    """Residual MLP, optionally with gated attention after each block."""

    def __init__(self, n_features, dim, n_blocks, dropout, use_attention):
        super().__init__()
        self.input_layer = nn.Linear(n_features, dim)
        self.input_dropout = nn.Dropout(dropout)
        layers = []
        for _ in range(n_blocks):
            layers.append(ResidualMLPBlock(dim, dropout))
            if use_attention:
                layers.append(GatedAttention(dim, dropout))
        self.layers = nn.Sequential(*layers)
        self.output_norm = nn.LayerNorm(dim)
        self.output_layer = nn.Linear(dim, 1)

    def forward(self, x):
        x = self.layers(self.input_dropout(self.input_layer(x)))
        return self.output_layer(self.output_norm(x)).squeeze(-1)


def input_size(spec):
    return len(selected_features) * (2 if spec["missing_flags"] else 1)


def make_model(spec, config):
    """Build the variant's model on the selected device."""
    if spec["model"] == "transformer":
        model = CrossSectionalTransformer(input_size(spec), config["embedding_dim"], config["n_blocks"], spec["dropout"])
    else:
        model = StockwiseNetwork(
            input_size(spec), config["embedding_dim"], config["n_blocks"], spec["dropout"], spec["model"] == "gated"
        )
    return model.to(DEVICE)


def gate_size(model):
    """Average absolute gate of the gated model; NaN for other models."""
    gates = [torch.tanh(value).abs().mean().item() for name, value in model.named_parameters() if name.endswith("gate")]
    return float(np.mean(gates)) if gates else float("nan")

## 45. Version 2 Training

As in Version 1, one country-month is one step. With `local_share`, an epoch keeps the original number of steps, but local
months are drawn for that share of them (repeating local months, and using a random subset of U.S. months each epoch).
Training for at most `RESEARCH_EPOCHS` epochs (100 in the full run) with early stopping; the best validation state is kept.

In [ ]:
def epoch_order(countries, local_share, rng):
    """Training-month positions for one epoch."""
    countries = np.asarray(countries)
    if local_share is None:
        return rng.permutation(len(countries))
    if not 0 < local_share < 1:
        raise ValueError("local_share must be between 0 and 1.")

    def draw(positions, n):
        rounds = int(np.ceil(n / len(positions)))
        return np.concatenate([rng.permutation(positions) for _ in range(rounds)])[:n]

    n_local = int(round(local_share * len(countries)))
    local = draw(np.flatnonzero(countries != "USA"), n_local)
    us = draw(np.flatnonzero(countries == "USA"), len(countries) - n_local)
    return rng.permutation(np.concatenate([local, us]))


def train_epoch_v2(model, optimizer, months, order, spec, scale):
    """One optimizer step per month; returns the average training loss."""
    model.train()
    losses = []
    for i in order:
        month = months[i]
        optimizer.zero_grad(set_to_none=True)
        try:
            loss = month_loss(model, month, spec, scale)
            loss.backward()
        except RuntimeError as error:
            raise device_error(month, error) from error
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss in {describe_month(month)}.")
        if not all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None):
            raise FloatingPointError(f"Nonfinite gradient in {describe_month(month)}.")
        if spec["clip_norm"] is not None:
            nn.utils.clip_grad_norm_(model.parameters(), spec["clip_norm"])
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))

In [ ]:
def train_v2(train_months, valid_months, spec, config, seed, label, limits, initial_state=None, max_epochs=MAX_EPOCHS):
    """Train one model with early stopping; `limits` are the clipping limits used by the validation scores."""
    if not 1 <= max_epochs <= 100:
        raise ValueError("max_epochs must be between 1 and 100.")
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = make_model(spec, config)
    if initial_state is not None:
        model.load_state_dict(initial_state)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    scheduler = None
    if spec["schedule"] == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)
    stopping = EARLY_STOPPING[spec["early_stopping"]]
    train_months = [month for month in train_months if month["labeled"].any()]
    if not train_months:
        raise ValueError(f"{label}: no labeled training months.")
    countries = [month["info"]["country"].iloc[0] for month in train_months]
    local_share = spec["local_share"] if len(set(countries)) > 1 else None  # applies to joint training only
    scale = zero_forecast_loss(train_months, spec)
    print_run(label, train_months, valid_months, config, seed, max_epochs)

    started = time.time()
    best, best_epoch, best_state, best_scores = np.inf, 0, None, None
    anchor, anchor_epoch, history = np.inf, 0, []
    for epoch in range(1, max_epochs + 1):
        order = epoch_order(countries, local_share, rng)
        train_loss = train_epoch_v2(model, optimizer, train_months, order, spec, scale)
        scores = validation_scores(model, valid_months, limits)
        if not np.isfinite(scores["mse"]):
            raise FloatingPointError(f"{label}: nonfinite validation MSE at epoch {epoch}.")
        value = selection_value(scores, spec["selection"])
        rate = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch, "train_loss": train_loss, "learning_rate": float(rate), **scores})
        if value < best:
            best, best_epoch, best_scores = value, epoch, scores
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
        if np.isinf(anchor) or value < anchor - stopping["min_delta"] * abs(anchor):
            anchor, anchor_epoch = value, epoch
        if scheduler is not None and np.isfinite(value):
            scheduler.step(value)
        print(f"  epoch {epoch}/{max_epochs} | train loss {train_loss:.6f} | validation MSE {scores['mse']:.6f}, clipped "
              f"{scores['clipped_mse']:.6f}, IC {scores['rank_ic']:.4f} | best epoch {best_epoch} | "
              f"{time.time() - started:.0f}s", flush=True)
        if epoch >= stopping["min_epochs"] and epoch - anchor_epoch >= stopping["patience"]:
            print("  stopped early", flush=True)
            break

    if best_state is None:
        raise FloatingPointError(f"{label}: the validation score was never defined.")
    model.load_state_dict(best_state)
    return model, {
        "config": dict(config), "seed": seed, "best_epoch": best_epoch, "scores": best_scores, "selection_value": float(best),
        "n_parameters": count_parameters(model), "gate": gate_size(model), "history": history, "max_epochs": max_epochs,
        "training_seconds": time.time() - started, "state": best_state,
    }

In [ ]:
def search_v2(train_months, valid_months, spec, grid, label, limits, initial_state=None):
    """Evaluate every setting with the first seed; returns the best run and a table of all settings."""
    best_run, trials = None, []
    for number, config in enumerate(grid, start=1):
        trial = {**config, "seed": SEED, "failure": ""}
        try:
            _, run = train_v2(
                train_months, valid_months, spec, config, SEED, f"{label} | setting {number}/{len(grid)}", limits, initial_state
            )
            trial.update(run["scores"], best_epoch=run["best_epoch"], selection_value=run["selection_value"])
            if best_run is None or run["selection_value"] < best_run["selection_value"]:
                best_run = run
        except FloatingPointError as error:  # unstable setting: record it and continue
            trial["failure"] = str(error)
            print(f"FAILED {label}: {error}", flush=True)
        trials.append(trial)
    if best_run is None:
        raise RuntimeError(f"{label}: every setting failed.")
    return best_run, pd.DataFrame(trials)


def v2_checkpoint_path(variant, strategy, country, window, seed):
    return variant_dir(variant) / "models" / f"{country}_{window}_{strategy}_seed{seed}.pt"


def save_v2_checkpoint(run, variant, strategy, country, window, source_run_id=""):
    checkpoint = {
        **run, "version": "v2", "variant": variant, "fingerprint": fingerprint(variant), "strategy": strategy,
        "country": country, "window": window, "run_id": f"{variant}-{strategy}-{country}-{window}-{run['seed']}-{time.time_ns()}",
        "source_run_id": source_run_id, "spec": variant_spec(variant), "features": list(selected_features),
        "environment": environment_details(),
    }
    path = v2_checkpoint_path(variant, strategy, country, window, run["seed"])
    save_safely(path, lambda temporary: torch.save(checkpoint, temporary))


def load_v2_checkpoint(variant, strategy, country, window, seed):
    """Rebuild a Version 2 model after checking its identity; returns (model, checkpoint)."""
    path = v2_checkpoint_path(variant, strategy, country, window, seed)
    if not path.exists():
        raise FileNotFoundError(f"Missing Version 2 checkpoint: {path}")
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    check_identity(checkpoint, variant, path.name)
    identity = tuple(checkpoint[name] for name in ["strategy", "country", "window", "seed"])
    if identity != (strategy, country, window, seed) or checkpoint["features"] != selected_features:
        raise ValueError(f"{path.name}: belongs to another experiment or predictor set.")
    if checkpoint["max_epochs"] != MAX_EPOCHS:
        raise ValueError(f"{path.name}: trained with another epoch limit.")
    model = make_model(variant_spec(variant), checkpoint["config"])
    model.load_state_dict(checkpoint["state"])
    return model.eval(), checkpoint

In [ ]:
def v2_complete(variant, strategy, country, window, sources=None):
    """True if all seeds are saved, compatible and recorded; incomplete experiments are trained again."""
    paths = [v2_checkpoint_path(variant, strategy, country, window, seed) for seed in TRANSFORMER_SEEDS]
    if not all(path.exists() for path in paths) or not is_recorded(variant, strategy, country, window):
        return False
    for seed in TRANSFORMER_SEEDS:
        _, checkpoint = load_v2_checkpoint(variant, strategy, country, window, seed)
        if sources is not None and checkpoint["source_run_id"] != sources[seed]["run_id"]:
            print(f"{variant} | {strategy} | {country} | {window}: trained again (its U.S. source changed)")
            return False
    print(f"{variant} | {strategy} | {country} | {window}: skipped (verified checkpoints and results exist)")
    return True


def train_v2_experiment(variant, strategy, country, window, train_months, valid_months, grid, limits, sources=None):
    """Search, train three seeds, record and save one experiment."""
    spec, label = variant_spec(variant), f"{variant} | {strategy} | {country} | {window}"
    start = {seed: None if sources is None else sources[seed]["state"] for seed in TRANSFORMER_SEEDS}
    best_run, trials = search_v2(train_months, valid_months, spec, grid, label, limits, start[SEED])
    runs = [best_run]
    for seed in TRANSFORMER_SEEDS:
        if seed != SEED:
            _, run = train_v2(train_months, valid_months, spec, best_run["config"], seed, label, limits, start[seed])
            runs.append(run)
    seed_rows = [{
        **run["config"], "seed": run["seed"], **run["scores"], "best_epoch": run["best_epoch"], "gate": run["gate"],
        "selection_value": run["selection_value"], "n_parameters": run["n_parameters"],
        "training_seconds": run["training_seconds"], "failure": "",
    } for run in runs]
    table = pd.concat([trials.assign(stage="search"), pd.DataFrame(seed_rows).assign(stage="seed")])
    record_validation(variant, table, strategy, country, window)  # results first, models last
    for run in runs:
        save_v2_checkpoint(
            run, variant, strategy, country, window, "" if sources is None else sources[run["seed"]]["run_id"]
        )


def v2_sources(variant):
    """The U.S. models of the same variant, one per seed."""
    return {seed: load_v2_checkpoint(variant, "pretrained", "USA", "train_full", seed)[1] for seed in TRANSFORMER_SEEDS}

In [ ]:
def load_v2_months(country, period, spec):
    """Training or validation months, with missing-value indicators if the variant uses them."""
    if period == "test":
        raise ValueError("Test data cannot be used for training or model selection.")
    return make_months(*prepare_data(country, period, selected_features, spec["missing_flags"]))


def train_variant(variant):
    """Train every strategy of one Transformer variant; completed and compatible experiments are skipped."""
    spec = variant_spec(variant)
    strategies = [s for s in ALL_VARIANTS[variant][2] if not V2_STRATEGIES or s in V2_STRATEGIES]
    start_variant(variant)
    us_limits = {"USA": clip_limits("USA", "train_full")}
    sources, us_months = None, None
    needs_source = bool({"direct", "finetuned"} & set(strategies))

    if needs_source:
        if not v2_complete(variant, "pretrained", "USA", "train_full"):
            us_valid = load_v2_months("USA", "validation", spec)
            us_months = attach_targets(load_v2_months("USA", "train_full", spec), spec, us_limits)
            check_training_periods(us_months, us_valid, "train_full")
            train_v2_experiment(
                variant, "pretrained", "USA", "train_full", us_months, us_valid, V2_GRID, us_limits["USA"]
            )
            del us_valid
        sources = v2_sources(variant)
        rate = sources[SEED]["config"]["learning_rate"]
        fine_grid_v2 = [
            {**sources[SEED]["config"], "learning_rate": factor * rate, "weight_decay": decay}
            for factor, decay in V2_FINE_TUNING
        ]

    for country in LOCAL_COUNTRIES:
        valid_months = load_v2_months(country, "validation", spec)
        if "direct" in strategies:
            rows = []
            for seed in TRANSFORMER_SEEDS:
                model, source = load_v2_checkpoint(variant, "pretrained", "USA", "train_full", seed)
                scores = validation_scores(model, valid_months, us_limits["USA"])
                rows.append({"seed": seed, **scores, "best_epoch": source["best_epoch"], "stage": "seed", "failure": ""})
            record_validation(variant, pd.DataFrame(rows), "direct", country, "all")

        for window in TRAIN_WINDOWS[country]:
            limits = {**us_limits, country: clip_limits(country, window)}
            for strategy in ["finetuned", "joint", "localmatched"]:
                if strategy not in strategies:
                    continue
                used_sources = sources if strategy == "finetuned" else None
                if v2_complete(variant, strategy, country, window, used_sources):
                    continue
                local_months = attach_targets(load_v2_months(country, window, spec), spec, limits)
                check_training_periods(local_months, valid_months, window)
                train_months = local_months
                if strategy == "joint":
                    if us_months is None:
                        us_months = attach_targets(load_v2_months("USA", "train_full", spec), spec, us_limits)
                    check_training_periods(us_months, valid_months, "train_full")
                    train_months = join_months(us_months, local_months)
                grid = fine_grid_v2 if strategy == "finetuned" else V2_GRID
                train_v2_experiment(
                    variant, strategy, country, window, train_months, valid_months, grid, limits[country], used_sources
                )

In [ ]:
if v2_stage("train"):
    for variant in V2_VARIANTS:
        if variant in VARIANTS:
            train_variant(variant)

## 46. Version 2 Baselines

In Version 1, Elastic Net often chose so large a penalty that all coefficients were zero: with a few extreme returns, the
raw validation MSE favors a constant forecast. The changes are tested separately: clipped training targets (`b_clip`), a
search scaled to the data (`b_grid`), rank-IC selection (`b_ic`), missing-value indicators (`b_missing`) and a
pseudo-Huber XGBoost objective (`b_huber`).

In `b_grid`, the Elastic Net penalty is a share of `alpha_max`, the smallest penalty with no active coefficient, so every
setting keeps some predictors. The XGBoost search uses shallow, slow and regularized trees, with 16 settings instead of 18.
A setting with a constant forecast has no rank IC and is marked in the diagnostics.

In [ ]:
BASELINE_NAMES = {"elastic_net": "Elastic Net", "xgboost": "XGBoost"}


def alpha_max(train, l1_ratio):
    """Smallest Elastic Net penalty for which all coefficients are zero, with monthly-equal weights."""
    weights = month_weights(train["info"])
    y = train["y"] - np.average(train["y"], weights=weights)
    gradient = train["X"].T.astype("float64") @ (weights * y) / len(y)
    gradient -= np.average(train["X"], axis=0, weights=weights) * (weights * y).mean()  # zero up to rounding
    return float(np.abs(gradient).max() / l1_ratio)


def baseline_grid(family, spec, train):
    """Settings of one family for this variant."""
    if spec["grid"] == "v1":
        return ELASTIC_NET_GRID if family == "elastic_net" else XGBOOST_GRID
    if family == "elastic_net":
        largest = {ratio: alpha_max(train, ratio) for _, ratio in V2_ELASTIC_NET}
        return [{"alpha": share * largest[ratio], "l1_ratio": ratio} for share, ratio in V2_ELASTIC_NET]
    return V2_XGBOOST_GRID


def make_baseline(family, spec, params, train):
    if family == "elastic_net":
        return make_elastic_net(params)
    if spec["objective"] == "pseudo_huber":
        center = float(np.average(train["y"], weights=month_weights(train["info"])))
        return XGBRegressor(
            **params, objective="reg:pseudohubererror", huber_slope=BASE_SPEC["huber_delta"], base_score=center,
            tree_method="hist", n_jobs=N_JOBS, random_state=SEED,
        )
    return make_xgboost(params)


def load_v2_labeled(country, period, spec):
    if period == "test":
        raise ValueError("Test data cannot be used for training or model selection.")
    return keep_labeled(*prepare_data(country, period, selected_features, spec["missing_flags"]))


def monthly_lists(info, *columns):
    """Split aligned arrays into lists with one array per month."""
    positions = list(info.reset_index(drop=True).groupby("eom").indices.values())
    return [[np.asarray(column)[rows] for rows in positions] for column in columns]

In [ ]:
def v2_baseline_path(variant, family, country, window):
    return variant_dir(variant) / "models" / f"{country}_{window}_{family}.joblib"


def load_v2_baseline(variant, family, country, window):
    path = v2_baseline_path(variant, family, country, window)
    if not path.exists():
        raise FileNotFoundError(f"Missing Version 2 model: {path}")
    saved = joblib.load(path)
    check_identity(saved, variant, path.name)
    if (saved["family"], saved["country"], saved["window"]) != (family, country, window):
        raise ValueError(f"{path.name}: belongs to another experiment.")
    if saved["features"] != selected_features:
        raise ValueError(f"{path.name}: saved predictors differ from selected_features.")
    return saved


def train_baseline_variant(variant):
    """Search and save the baselines of one variant; compatible completed models are skipped."""
    spec = variant_spec(variant)
    start_variant(variant)
    for country in LOCAL_COUNTRIES:
        valid = load_v2_labeled(country, "validation", spec)
        for window in TRAIN_WINDOWS[country]:
            limits = clip_limits(country, window)
            train = load_v2_labeled(country, window, spec)
            assert train["info"]["predicted_month"].max() < valid["info"]["predicted_month"].min()
            if spec["target"] == "clip":
                train["y"] = np.clip(train["y"], *limits)  # training targets only
            for family in ALL_VARIANTS[variant][2]:
                path = v2_baseline_path(variant, family, country, window)
                if path.exists() and is_recorded(variant, family, country, window):
                    load_v2_baseline(variant, family, country, window)  # raises an error if incompatible
                    print(f"{variant} | {family} | {country} | {window}: skipped (verified model and results exist)")
                    continue
                print(f"\n{variant} | {family} | {country} | {window}", flush=True)
                best_model, best_value, trials = None, np.inf, []
                for params in baseline_grid(family, spec, train):
                    model = make_baseline(family, spec, params, train)
                    converged = fit_model(model, train)
                    prediction = model.predict(valid["X"]).astype("float64")
                    scores = score_forecasts(*monthly_lists(valid["info"], prediction, valid["y"]), limits)
                    value = selection_value(scores, spec["selection"])
                    trials.append({
                        **params, **scores, "selection_value": value, "converged": converged,
                        "prediction_std": prediction.std(), "max_abs_prediction": np.abs(prediction).max(),
                        "nonzero_coefficients": int((model.coef_ != 0).sum()) if family == "elastic_net" else np.nan,
                    })
                    if value < best_value:
                        best_model, best_value = model, value
                if best_model is None:
                    raise ValueError(f"{variant} {family} {country} {window}: no setting has a defined validation score.")
                trials = pd.DataFrame(trials)
                trials["selected"] = trials.index == trials["selection_value"].idxmin()
                trials["zero_mse"] = monthly_mse(valid["info"], valid["y"], 0.0)
                record_validation(variant, trials.assign(stage="search"), family, country, window)
                saved = {
                    "version": "v2", "variant": variant, "fingerprint": fingerprint(variant), "family": family,
                    "country": country, "window": window, "model": best_model, "features": selected_features,
                    "run_id": f"{variant}-{family}-{country}-{window}-{time.time_ns()}", "spec": spec,
                    "environment": environment_details(),
                }
                save_safely(path, lambda temporary: joblib.dump(saved, temporary))


if v2_stage("train"):
    for variant in V2_VARIANTS:
        if variant in BASELINE_VARIANTS:
            train_baseline_variant(variant)

In [ ]:
def baseline_diagnostics(table, groups):
    """How often a search produced a constant forecast, and what was selected."""
    selected = table[table["selected"].astype(bool)].set_index(groups)
    result = table.assign(constant=table["prediction_std"] < 1e-8).groupby(groups).agg(
        settings=("constant", "size"), constant_settings=("constant", "sum"),
    )
    return result.assign(
        selected_std=selected["prediction_std"], selected_max=selected["max_abs_prediction"], zero_mse=selected["zero_mse"],
        selected_mse=selected["mse"] if "mse" in selected else selected["validation_mse"],
    )


# Version 1 is read here only as a record of its search; nothing is loaded or changed
v1_baselines = V1_DIR / "baseline_validation.csv"
if v1_baselines.exists():
    print("Version 1 baseline searches:")
    display(baseline_diagnostics(pd.read_csv(v1_baselines), ["family", "country", "window"]))
if v2_stage("compare"):
    for variant in BASELINE_VARIANTS:
        path = variant_dir(variant) / "validation.csv"
        if path.exists():
            print(f"{variant}:")
            display(baseline_diagnostics(pd.read_csv(path), ["strategy", "country", "window"]))

## 47. Version 2 Forecasts

Every trained model predicts the validation months (used only to estimate ensemble weights) and the test months. Each
forecast file records the `run_id` values of its models, so forecasts of replaced models are generated again.

In [ ]:
def forecast_path(variant, strategy, country, window):
    return variant_dir(variant) / "predictions" / f"{country}_{window}_{strategy}.parquet"


def load_for_inference(country, period, spec):
    """All stocks of the validation or test period, including those without a return."""
    if period not in ["validation", "test"]:
        raise ValueError("Forecasts are made for validation and test periods only.")
    info, X = prepare_data(country, period, selected_features, spec["missing_flags"])
    assert in_period(info["predicted_month"], period).all()
    assert info["predicted_month"].nunique() == expected_months[period]
    return info, X


def member_models(variant, strategy, country, window):
    """The saved models of one forecast: {column name: (model, run_id)}."""
    if variant in BASELINE_VARIANTS:
        saved = load_v2_baseline(variant, strategy, country, window)
        return {"prediction": (saved["model"], saved["run_id"])}
    source = ("pretrained", "USA", "train_full") if strategy == "direct" else (strategy, country, window)
    models = {}
    for seed in TRANSFORMER_SEEDS:
        model, checkpoint = load_v2_checkpoint(variant, *source, seed)
        models[f"seed_{seed}"] = (model, checkpoint["run_id"])
    return models


def predict_variant(variant):
    """Save the forecasts of every strategy of one variant; files of unchanged models are kept."""
    spec = variant_spec(variant)
    start_variant(variant)
    strategies = ALL_VARIANTS[variant][2]
    if variant in VARIANTS and V2_STRATEGIES:
        strategies = [s for s in strategies if s in V2_STRATEGIES]
    for country in LOCAL_COUNTRIES:
        samples = {}
        for strategy in strategies:
            for window in ["all"] if strategy == "direct" else TRAIN_WINDOWS[country]:
                models = member_models(variant, strategy, country, window)
                sources = "|".join(run_id for _, run_id in models.values())
                path = forecast_path(variant, strategy, country, window)
                if path.exists() and set(pd.read_parquet(path, columns=["sources"])["sources"]) == {sources}:
                    print(f"{variant} | {strategy} | {country} | {window}: forecasts skipped (already current)")
                    continue
                print(f"Predicting: {variant} | {strategy} | {country} | {window}", flush=True)
                tables = []
                for period in ["validation", "test"]:
                    if period not in samples:
                        info, X = load_for_inference(country, period, spec)
                        samples[period] = (info, X, make_months(info, X) if variant in VARIANTS else None)
                    info, X, months = samples[period]
                    table = info[KEYS].copy()
                    for column, (model, _) in models.items():
                        if months is None:
                            predictions = predict_baseline(model, info, X)
                        else:
                            predictions = predict_transformer(model, months)
                        check_predictions(predictions, info, f"{variant} {strategy} {country} {window} {period}")
                        table = table.merge(
                            predictions.rename(columns={"prediction": column}), on=KEYS, how="left", validate="one_to_one"
                        )
                    if "prediction" not in table:
                        table["prediction"] = table[list(models)].mean(axis=1, skipna=False)
                    tables.append(table.assign(period=period, sources=sources))
                forecasts = pd.concat(tables, ignore_index=True)
                assert np.isfinite(forecasts["prediction"]).all()
                save_safely(path, lambda temporary: forecasts.to_parquet(temporary, index=False))


if v2_stage("predict"):
    for variant in V2_VARIANTS:
        predict_variant(variant)

## 48. Version 2 Ensembles

Each Transformer variant is combined with the same baselines (`b_grid`). Weights use validation months 2018–2019 only:

- **equal**: the Version 1 rule, one sixth per strategy;
- **inverse_mse**: proportional to one over the clipped validation MSE;
- **rank_ic**: proportional to the positive part of the validation rank IC (equal if none is positive);
- **stacked**: nonnegative least squares on clipped validation returns, scaled to sum to one, then averaged with equal
  weights to limit overfitting.

The validation months were also used for early stopping and model selection, so these weights are somewhat optimistic
about the members. An ensemble is built only if all six members exist.

In [ ]:
ENSEMBLE_METHODS = ["equal", "inverse_mse", "rank_ic", "stacked"]


def read_forecast(variant, strategy, country, window, period):
    """Saved forecast of one period, or None if the experiment is not available."""
    path = forecast_path(variant, strategy, country, "all" if strategy == "direct" else window)
    if not path.exists():
        return None
    # The recorded model versions must be the current ones
    models = member_models(variant, strategy, country, "all" if strategy == "direct" else window)
    current = "|".join(run_id for _, run_id in models.values())
    table = pd.read_parquet(path)
    if set(table["sources"]) != {current}:
        raise ValueError(f"{path.name} in {variant} comes from replaced models. Run the predict stage again.")
    return table[table["period"] == period].sort_values(["eom", "id"], ignore_index=True)


def add_actuals(table, country):
    """Unmodified returns and size groups for saved forecasts, in the format of the Version 1 evaluation."""
    source = data[country][KEYS[1:] + [TARGET, "size_grp"]].rename(columns={TARGET: "actual_return"})
    stocks = table.merge(source, on=KEYS[1:], how="left", validate="one_to_one", indicator=True)
    assert (stocks["_merge"] == "both").all(), "A forecast does not match any stock-month."
    stocks["has_actual"] = stocks["actual_return"].notna()
    stocks["in_common"] = True  # every model predicts every stock-month
    return stocks.drop(columns="_merge")


def ensemble_weights(forecasts, actual, months, limits):
    """Weights of every method, from validation forecasts of shape (stock-months, members)."""
    n = forecasts.shape[1]
    labeled = np.isfinite(actual)
    forecasts, actual, months = forecasts[labeled], actual[labeled], months[labeled]
    info = pd.DataFrame({"eom": months})
    scores = []
    for j in range(n):
        split = monthly_lists(info, forecasts[:, j], actual)
        scores.append(score_forecasts(*split, limits))
    inverse = np.array([1 / score["clipped_mse"] for score in scores])
    ic = np.nan_to_num(np.array([score["rank_ic"] for score in scores]), nan=0.0).clip(min=0)

    month_size = info.groupby("eom")["eom"].transform("size").to_numpy()
    stack = LinearRegression(positive=True, fit_intercept=False)
    stack.fit(forecasts, np.clip(actual, *limits), sample_weight=1 / month_size)
    equal = np.full(n, 1 / n)

    def scaled(values):
        return values / values.sum() if values.sum() > 0 else equal

    return {
        "equal": equal, "inverse_mse": scaled(inverse), "rank_ic": scaled(ic),
        "stacked": 0.5 * equal + 0.5 * scaled(stack.coef_),
    }


def build_ensembles(variant, country, window):
    """Test forecasts of every ensemble method and their weights; (None, None) if a member is missing."""
    members = {(variant, s): None for s in V2_TRANSFORMER_STRATEGIES}
    members.update({(ENSEMBLE_BASELINES, family): None for family in ["elastic_net", "xgboost"]})
    tables = {period: {} for period in ["validation", "test"]}
    for period in tables:
        for owner, strategy in members:
            table = read_forecast(owner, strategy, country, window, period)
            if table is None:
                return None, None
            tables[period][strategy] = table
    for period in tables:  # identical stock-months in the same order
        reference = tables[period]["localmatched"][KEYS]
        assert all(table[KEYS].equals(reference) for table in tables[period].values())

    valid = add_actuals(tables["validation"]["localmatched"][KEYS], country)
    matrix = np.column_stack([table["prediction"] for table in tables["validation"].values()])
    weights = ensemble_weights(
        matrix, valid["actual_return"].to_numpy(), valid["eom"].to_numpy(), clip_limits(country, window)
    )
    test_matrix = np.column_stack([table["prediction"] for table in tables["test"].values()])
    forecasts = {
        f"ensemble_{name}": tables["test"]["localmatched"][KEYS].assign(prediction=test_matrix @ value)
        for name, value in weights.items()
    }
    rows = pd.DataFrame(weights, index=list(tables["test"])).T.rename_axis("method").reset_index()
    return forecasts, rows.assign(country=country, window=window)

## 49. Version 2 Evaluation and Sensitivity

The original metrics are calculated by the Version 1 functions on all 72 test months and unmodified returns. Further checks
are reported **next to** them:

- **Scenarios** (main forecasts): deciles replaced by quintiles, costs of 25 bps, returns limited to ±100%, and the universe without
  micro and nano stocks. No scenario removes a month.
- **Drawdowns**: `max_drawdown` is additive (Version 1: the largest fall of the sum of monthly returns, which can exceed
  100%). `drawdown_compounded` compounds one unit of capital per $1 long and $1 short, and `drawdown_half` holds half
  that exposure. Both are percentages of the previous peak, between −100% and 0.
- **Portfolio check**: gross returns are recalculated independently for every main forecast and must agree.

In [ ]:
SCENARIOS = {
    "base": {}, "quintiles": {"decile": 0.20}, "cost25": {"cost_bps": 25}, "returns_capped": {"cap": 1.0},
    "no_micro": {"exclude": ["micro", "nano"]},
}


def evaluate_scenario(stocks, decile=DECILE, cost_bps=COST_BPS, cap=None, exclude=()):
    """Monthly results of one forecast under one scenario."""
    stocks = stocks.copy()
    if cap is not None:
        stocks["actual_return"] = stocks["actual_return"].clip(-cap, cap)
    if exclude:
        stocks = stocks[~stocks["size_grp"].isin(exclude).to_numpy()]
    return evaluate_forecast(stocks, decile, cost_bps)


def compounded_drawdown(returns, exposure):
    """Largest percentage fall of compounded wealth; wealth stays at zero after a complete loss."""
    wealth = (1 + exposure * returns).clip(lower=0).cumprod()
    peak = wealth.cummax().clip(lower=1)
    return float((wealth / peak - 1).min())


def summarize_v2(months):
    months = months.sort_values("predicted_month")
    extra = {
        f"{name}_{label}": compounded_drawdown(months[f"{name}_return"], exposure)
        for name in ["gross", "net"] for label, exposure in [("drawdown_compounded", 1.0), ("drawdown_half", 0.5)]
    }
    return {**summarize(months).to_dict(), **extra}


def independent_gross_returns(stocks):
    """Recalculate decile long-short returns without the functions of Section 35."""
    results = {}
    for month, part in stocks.groupby("predicted_month"):
        prediction, ids = part["prediction"].to_numpy(), part["id"].to_numpy()
        actual = part["actual_return"].to_numpy()
        order = np.lexsort((ids, prediction))
        n = max(1, int(np.floor(DECILE * len(order))))
        constant = len(np.unique(prediction)) < 2
        results[month] = 0.0 if constant else np.nanmean(actual[order[-n:]]) - np.nanmean(actual[order[:n]])
    return pd.Series(results).sort_index()

In [ ]:
def evaluation_path(variant, name):
    return variant_dir(variant) / "evaluation" / name


def evaluate_variant(variant):
    """Evaluate every available test forecast of one variant; returns False if nothing is available."""
    monthly_rows, weight_rows = [], []
    strategies = ALL_VARIANTS[variant][2]
    for country in LOCAL_COUNTRIES:
        n_expected = in_period(data[country]["predicted_month"], "test").sum()
        for window in TRAIN_WINDOWS[country]:
            print(f"Evaluating: {variant} | {country} | {window}", flush=True)
            forecasts = {}
            for strategy in strategies:
                table = read_forecast(variant, strategy, country, window, "test")
                if table is not None:
                    forecasts[strategy] = table
            if variant in VARIANTS:
                ensembles, weights = build_ensembles(variant, country, window)
                if ensembles is not None:
                    forecasts.update(ensembles)
                    weight_rows.append(weights)
            for strategy, table in forecasts.items():
                label = {"country": country, "window": window, "strategy": strategy}
                stocks = add_actuals(table, country)
                assert len(stocks) == n_expected and stocks["predicted_month"].nunique() == expected_months["test"]
                for scenario, options in SCENARIOS.items():
                    months = evaluate_scenario(stocks, **options)
                    monthly_rows.append(months.assign(**label, scenario=scenario, seed="main"))
                    if scenario == "base":
                        check = independent_gross_returns(stocks)
                        np.testing.assert_allclose(check.to_numpy(), months["gross_return"].to_numpy(), atol=1e-12)
                for column in [c for c in table.columns if c.startswith("seed_")]:  # individual seeds, base scenario
                    months = evaluate_scenario(stocks.assign(prediction=stocks[column]))
                    monthly_rows.append(months.assign(**label, scenario="base", seed=column[5:]))
    if not monthly_rows:
        print(f"{variant}: no forecasts to evaluate")
        return False

    results = pd.concat(monthly_rows, ignore_index=True).assign(variant=variant)
    groups = ["variant", "country", "window", "strategy", "scenario", "seed"]
    table = pd.DataFrame([{**dict(zip(groups, names)), **summarize_v2(part)} for names, part in results.groupby(groups)])
    evaluation_path(variant, "").mkdir(parents=True, exist_ok=True)
    if weight_rows:
        weights = pd.concat(weight_rows, ignore_index=True)
        save_safely(evaluation_path(variant, "ensemble_weights.csv"), lambda path: weights.to_csv(path, index=False))
    save_safely(evaluation_path(variant, "summary.csv"), lambda path: table.to_csv(path, index=False))
    save_safely(evaluation_path(variant, "monthly.parquet"), lambda path: results.to_parquet(path, index=False))
    return True


if v2_stage("evaluate"):
    for variant in V2_VARIANTS:
        if (variant_dir(variant) / "config.json").exists():
            start_variant(variant)  # rejects results of another configuration
            evaluate_variant(variant)
        else:
            print(f"{variant}: not trained yet")

## 50. Ablation Comparison

Each variant is compared with its reference, and with Version 1, on the same months by the paired block bootstrap of
Section 37. Holm's adjustment covers all variants of one country, history, strategy and metric. These are exploratory tests
on a reused test period. The **validation** comparison is the one that may guide a choice of variant; the test comparison
reports what happened.

In [ ]:
V1_STRATEGY = {"ensemble_equal": "ensemble"}  # Version 2 name: Version 1 name, if different


def read_all(name, reader):
    """Combine one evaluation file over all evaluated variants, checking their configuration."""
    tables = []
    for variant in ALL_VARIANTS:
        path = evaluation_path(variant, name)
        if path.exists():
            start_variant(variant)
            tables.append(reader(path))
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


def paired_tests(new, old, label):
    """Bootstrap tests of one monthly result series against another, for every metric."""
    new, old = new.sort_values("predicted_month"), old.sort_values("predicted_month")
    assert new["predicted_month"].tolist() == old["predicted_month"].tolist()
    assert np.allclose(new["zero_mse"], old["zero_mse"]), "The two results use different stock-months."
    indices = block_indices(len(new), np.random.default_rng(SEED))
    return [
        {**label, "metric": metric,
         **bootstrap_difference(new[column].to_numpy(), old[column].to_numpy(), sign, statistic, indices)}
        for metric, (column, sign, statistic) in TEST_METRICS.items()
    ]

In [ ]:
def run_comparison():
    """Compare every evaluated variant with its reference and with Version 1; returns (tests, monthly, summary)."""
    monthly_v2 = read_all("monthly.parquet", pd.read_parquet)
    if monthly_v2.empty:
        raise FileNotFoundError("No evaluated Version 2 variants. Run the evaluate stage first.")
    missing = [name for name in ALL_VARIANTS if name not in set(monthly_v2["variant"])]
    if missing:
        print(f"WARNING: not evaluated and therefore not compared: {', '.join(missing)}")

    base = monthly_v2[(monthly_v2["scenario"] == "base") & (monthly_v2["seed"] == "main")]
    series = {names: part for names, part in base.groupby(["variant", "country", "window", "strategy"])}
    v1_path = V1_DIR / "evaluation" / "monthly_results.parquet"
    v1 = {}
    if v1_path.exists():
        v1_monthly = pd.read_parquet(v1_path)
        v1_monthly = v1_monthly[v1_monthly["seed"] == v1_monthly["strategy"].map(main_seed)]
        v1 = {names: part for names, part in v1_monthly.groupby(["country", "window", "strategy"])}
    else:
        print(f"WARNING: {v1_path} not found; Version 1 is not compared. Set CMP_V1_RESULTS to its results folder.")

    tests = []
    for (variant, country, window, strategy), part in series.items():
        label = {"country": country, "window": window, "strategy": strategy, "variant": variant}
        reference = ALL_VARIANTS[variant][0]
        if reference is not None and (reference, country, window, strategy) in series:
            other = series[reference, country, window, strategy]
            tests += paired_tests(part, other, {**label, "family": "ablation", "against": reference})
        other = v1.get((country, window, V1_STRATEGY.get(strategy, strategy)))
        if other is not None:
            tests += paired_tests(part, other, {**label, "family": "version 1", "against": "Version 1"})
        # Ensemble weighting methods against the equal-weight ensemble of the same variant
        if strategy.startswith("ensemble_") and strategy != "ensemble_equal":
            other = series[variant, country, window, "ensemble_equal"]
            tests += paired_tests(part, other, {**label, "family": "ensemble weights", "against": "ensemble_equal"})
    tests = pd.DataFrame(tests)
    if not tests.empty:
        family = ["family", "country", "window", "strategy", "metric"]
        tests["p_adjusted"] = tests.groupby(family)["p_value"].transform(holm_adjust)
    return tests, monthly_v2, read_all("summary.csv", pd.read_csv)


def validation_comparison():
    """Validation scores of every variant next to its reference (average of the three seeds)."""
    tables = [pd.read_csv(p) for p in [variant_dir(v) / "validation.csv" for v in ALL_VARIANTS] if p.exists()]
    table = pd.concat(tables, ignore_index=True)
    is_final = (table["stage"] == "seed") | (table["selected"].eq(True) if "selected" in table else False)
    groups, scores = ["variant", "strategy", "country", "window"], ["mse", "clipped_mse", "rank_ic"]
    result = table[is_final].groupby(groups)[scores].mean().reset_index()
    result["reference"] = result["variant"].map(lambda name: ALL_VARIANTS[name][0])
    other = result[groups + scores].rename(columns={"variant": "reference"})
    return result.merge(other, on=["reference"] + groups[1:], how="left", suffixes=("", "_reference"))

In [ ]:
def month_influence(monthly_v2):
    """Leave one test month out at a time: range of each metric and the most influential month."""
    base = monthly_v2[(monthly_v2["scenario"] == "base") & (monthly_v2["seed"] == "main")]
    rows = []
    for names, part in base.groupby(["variant", "country", "window", "strategy"]):
        part = part.sort_values("predicted_month")
        mse, zero, ic, gross = [part[c].to_numpy() for c in ["mse", "zero_mse", "rank_ic", "gross_return"]]
        n = len(part)
        others = ~np.eye(n, dtype=bool)
        values = {
            "oos_r2": np.array([1 - mse[keep].mean() / zero[keep].mean() for keep in others]),
            "mean_ic": np.array([ic[keep].mean() for keep in others]),
            "gross_sharpe": np.array([sharpe_ratio(gross[keep]) for keep in others]),
        }
        full = {"oos_r2": 1 - mse.mean() / zero.mean(), "mean_ic": ic.mean(), "gross_sharpe": sharpe_ratio(gross)}
        for metric, loo in values.items():
            if np.isfinite(loo).all() and np.isfinite(full[metric]):
                rows.append({
                    **dict(zip(["variant", "country", "window", "strategy"], names)), "metric": metric,
                    "full": full[metric], "lowest": loo.min(), "highest": loo.max(),
                    "most_influential": part["predicted_month"].iloc[int(np.abs(loo - full[metric]).argmax())],
                })
    return pd.DataFrame(rows)


def outlier_diagnostics():
    """Concentration of squared returns by country and period; no observation is removed."""
    rows = []
    for country in COUNTRIES:
        for period in TRAIN_WINDOWS[country] + ["validation", "test"]:
            df = data[country]
            part = df[in_period(df["predicted_month"], period)].dropna(subset=[TARGET])
            squared = part[TARGET] ** 2
            by_month = squared.groupby(part["predicted_month"]).mean()
            rows.append({
                "country": country, "period": period, "labeled": len(part), "largest": part[TARGET].max(),
                "above_100pct": int((part[TARGET] > 1).sum()), "top10_share": squared.nlargest(10).sum() / squared.sum(),
                "top_month": by_month.idxmax(), "top_month_share": by_month.max() / by_month.sum(),
            })
    return pd.DataFrame(rows)

In [ ]:
if v2_stage("compare"):
    tests, monthly_v2, summary_v2 = run_comparison()
    validation_table = validation_comparison()
    influence, outliers = month_influence(monthly_v2), outlier_diagnostics()
    comparison_dir = V2_DIR / "comparison"
    comparison_dir.mkdir(parents=True, exist_ok=True)
    for name, table in {
        "tests": tests, "summary": summary_v2, "validation": validation_table, "month_influence": influence,
        "outliers": outliers,
    }.items():
        save_safely(comparison_dir / f"{name}.csv", lambda path, table=table: table.to_csv(path, index=False))
    print("Exploratory results: the test period was already examined in Version 1.")
    print("Validation scores against the reference (these may guide a choice):")
    display(validation_table)
    if not tests.empty:
        print("Test rank IC: improvement over the reference variant")
        display(tests[(tests["family"] == "ablation") & (tests["metric"] == "Rank IC")].round(4))
        print("Test rank IC: improvement over Version 1")
        display(tests[(tests["family"] == "version 1") & (tests["metric"] == "Rank IC")].round(4))
    print("Months with the largest influence, and concentration of squared returns:")
    display(influence.head(20))
    display(outliers)

In [ ]:
if v2_stage("compare") and not tests.empty:
    # Rank IC improvement over the reference, averaged over countries and histories, by strategy
    shown = tests[(tests["family"] == "ablation") & (tests["metric"] == "Rank IC")
                  & tests["strategy"].isin(V2_TRANSFORMER_STRATEGIES)]
    if not shown.empty:
        chart = shown.pivot_table(index="variant", columns="strategy", values="improvement")
        chart = chart.reindex([name for name in VARIANTS if name in chart.index])
        axis = chart.plot.barh(figsize=(10, 9))
        axis.axvline(0, color="black", linewidth=0.8)
        axis.set(xlabel="Change in test rank IC against the reference variant (average, exploratory)", ylabel="")
        axis.figure.tight_layout()
        axis.figure.savefig(V2_DIR / "comparison" / "ablation_rank_ic.png", dpi=200, bbox_inches="tight")
        plt.show()

## 51. Version 2 Verification

The configuration checks always run; the code checks use small artificial months with `RUN_SMOKE_TESTS` and save nothing.

In [ ]:
assert variant_spec("control") == BASE_SPEC and not variant_spec("robust")["rank_weight"]
assert len({fingerprint(name) for name in ALL_VARIANTS}) == len(ALL_VARIANTS)
assert ENSEMBLE_BASELINES in BASELINE_VARIANTS and 1 <= RESEARCH_EPOCHS <= 100
for loader in [load_v2_months, load_v2_labeled]:
    try:
        loader("GBR", "test", BASE_SPEC)
        refused = False
    except ValueError:
        refused = True
    assert refused
try:
    check_identity({"experiment": "local"}, "control", "a Version 1 checkpoint")
    refused = False
except ValueError:
    refused = True
assert refused
print("Version 2 configuration, test-period protection and checkpoint protection verified.")

In [ ]:
if RUN_SMOKE_TESTS:
    small_info, small_X = make_smoke_months([20, 35, 27], "2017-09-30")
    limits = {"GBR": (-0.1, 0.1), "USA": (-0.1, 0.1)}
    raw_months = make_months(small_info, small_X)
    small_valid = make_months(*make_smoke_months([25, 30], "2017-12-31"))
    spec = {**BASE_SPEC, "target": "clip"}
    months = attach_targets(raw_months, spec, limits)

    # Clipping changes the training target only; unlabeled stocks do not affect the loss
    labeled = months[0]["labeled"]
    assert torch.equal(months[0]["y"][labeled], raw_months[0]["y"][labeled])
    assert months[0]["target"][labeled].abs().max() <= 0.1 + 1e-6
    small = torch.tensor([0.01, -0.02])
    assert torch.isclose(regression_loss(small, torch.zeros(2), {**spec, "loss": "huber"}), (small ** 2).mean())

    # The ranking loss is zero for perfectly linear ranks, two for reversed ranks, and differentiable
    ranks = months[0]["ranks"]
    assert abs(ranking_loss(ranks * 3 + 1, ranks).item()) < 1e-3 and abs(ranking_loss(-ranks, ranks).item() - 2) < 1e-3

    # Joint sampling: number of steps, shares and full coverage in the original weighting
    countries = ["USA"] * 80 + ["GBR"] * 20
    for share, expected in [(0.5, 50), (0.75, 75)]:
        order = epoch_order(countries, share, np.random.default_rng(SEED))
        assert len(order) == 100 and (order >= 80).sum() == expected
    assert sorted(epoch_order(countries, None, np.random.default_rng(SEED))) == list(range(100))
    print("Targets, losses and sampling verified.")

In [ ]:
if RUN_SMOKE_TESTS:
    config = V2_GRID[0]
    for architecture_name in ["transformer", "mlp", "gated"]:
        for flags in [False, True]:
            spec = {**BASE_SPEC, "model": architecture_name, "missing_flags": flags, "target": "clip", "loss": "huber",
                    "rank_weight": 1.0, "clip_norm": 1.0, "schedule": "plateau", "early_stopping": "patient",
                    "selection": "rank_ic"}
            wide = [
                {**m, "X": torch.cat([m["X"], torch.zeros_like(m["X"])], dim=1)} if flags else m for m in raw_months
            ]
            wide_valid = [
                {**m, "X": torch.cat([m["X"], torch.zeros_like(m["X"])], dim=1)} if flags else m for m in small_valid
            ]
            model, run = train_v2(
                attach_targets(wide, spec, limits), wide_valid, spec, config, SEED, f"Smoke {architecture_name}", (-0.1, 0.1),
                max_epochs=2,
            )
            rebuilt = make_model(spec, config)
            rebuilt.load_state_dict(run["state"])
            again = validation_scores(rebuilt, wide_valid, (-0.1, 0.1))
            assert np.isclose(again["mse"], run["scores"]["mse"], rtol=1e-4)

            # Reordering stocks reorders predictions; only models with attention use other stocks
            month = wide[0]
            shuffled = torch.randperm(len(month["X"]))
            rebuilt.eval()
            with torch.inference_mode():
                whole = rebuilt(month["X"].to(DEVICE)).cpu()
                assert torch.allclose(whole[shuffled], rebuilt(month["X"][shuffled].to(DEVICE)).cpu(), atol=1e-4)
                alone = rebuilt(month["X"][:1].to(DEVICE)).cpu()
            if architecture_name == "mlp":
                assert torch.allclose(alone, whole[:1], atol=1e-4)
    print("Version 2 architectures, training and saving format verified.")

In [ ]:
if RUN_SMOKE_TESTS:
    # Missing-value indicators: same ranked predictors, indicator equal to the raw missing pattern
    end = PERIODS["train_1y"][1] - pd.offsets.MonthEnd(1)
    plain_info, plain_X = prepare_months("GBR", end, end, selected_features)
    flag_info, flag_X = prepare_months("GBR", end, end, selected_features, True)
    raw = read_months("GBR", end, end, selected_features)
    n = len(selected_features)
    assert flag_info.equals(plain_info) and np.array_equal(flag_X[:, :n], plain_X) and flag_X.shape[1] == 2 * n
    assert np.array_equal(flag_X[:, n:], (~np.isfinite(raw[selected_features].to_numpy(dtype=float))).astype("float32"))

    # Compounded drawdowns, and ensemble weights that sum to one
    assert np.isclose(compounded_drawdown(pd.Series([0.1, -0.5, 0.2]), 1.0), -0.5)
    assert np.isclose(compounded_drawdown(pd.Series([-2.0, 0.5]), 1.0), -1.0)
    generator = np.random.default_rng(SEED)
    example_actual = generator.normal(0, 0.1, 400)
    example_forecasts = np.column_stack([example_actual + generator.normal(0, 0.1, 400), generator.normal(0, 0.1, 400)])
    weights = ensemble_weights(example_forecasts, example_actual, np.repeat([1, 2, 3, 4], 100), (-0.3, 0.3))
    assert set(weights) == set(ENSEMBLE_METHODS)
    assert all(np.isclose(value.sum(), 1) and (value >= 0).all() for value in weights.values())
    assert weights["stacked"][0] > weights["stacked"][1]

    # The independent portfolio calculation agrees with Section 35
    example = pd.DataFrame({
        "id": np.arange(50), "prediction": generator.normal(size=50), "actual_return": generator.normal(0, 0.1, 50),
        "predicted_month": pd.Timestamp("2020-01-31"), "in_common": True, "has_actual": True,
    })
    assert np.allclose(independent_gross_returns(example), evaluate_forecast(example)["gross_return"])
    print("Indicators, drawdowns, ensemble weights and portfolio calculation verified.")

Phase 8 — End

Phase 9 — Start

## 52. Finalization Audit: Purpose and Safeguards

Version 2 was trained and evaluated at commit `69a628c` (tag `v2.0-experiment`). This phase changes no model. It reads the saved
Version 2 results and the data, and writes **only** to a new folder (`audit_v2/`). The Version 2 tables keep their values and
names; every corrected or additional quantity has a new name.

- `audit_v2/private/` holds stock-level records (licensed; never commit or share). `audit_v2/aggregate/` holds tables without
  stock identifiers.
- `CMP_V2_RESULTS` names the completed Version 2 results. Their SHA-256 checksums are compared before and after the audit.
- **Stages:** `reports` uses saved tables only; `data`, `validation` and `portfolio` also need the data and saved forecasts.
- All 2020–2025 results are **exploratory**: Version 1 already used this test period.

In [ ]:
AUDIT_CODE_VERSION = 1  # evaluation-code revision; separate from V2_CODE_VERSION, which identifies the trained models
AUDIT_DIR = Path(os.environ.get("CMP_AUDIT_DIR", PROJECT_ROOT / "audit_v2"))
V2_SOURCE = Path(os.environ.get("CMP_V2_RESULTS", V2_DIR))  # completed Version 2 results, read only
AUDIT_BOOTSTRAP = 20_000  # smallest possible p-value: 1 / 20,001
AUDIT_CANDIDATES = [  # (variant, strategy) used by the portfolio stage; from the validation shortlist and its comparisons
    ("rank", "finetuned"), ("rank", "joint"), ("select_ic", "joint"), ("combined", "joint"), ("rank", "localmatched"),
    ("robust", "localmatched"), ("b_huber", "xgboost"), ("b_clip", "elastic_net"),
]

if RUN_AUDIT:
    if RUN_V2:
        raise ValueError("The audit is read-only; run it separately from Version 2 training and evaluation.")
    for protected in [V2_SOURCE, V1_DIR]:
        inside = AUDIT_DIR.resolve() == protected.resolve() or protected.resolve() in AUDIT_DIR.resolve().parents
        if inside:
            raise ValueError(f"AUDIT_DIR must be outside {protected}.")
    if not (V2_SOURCE / "comparison" / "tests.csv").exists():
        raise FileNotFoundError(f"{V2_SOURCE} holds no completed Version 2 comparison. Set CMP_V2_RESULTS.")


def audit_stage(stage):
    """True if this audit stage is switched on."""
    if RUN_AUDIT and stage in AUDIT_STAGES:
        return True
    print(f"Audit {stage}: not run (RUN_AUDIT = {RUN_AUDIT}, AUDIT_STAGES = {AUDIT_STAGES})")
    return False


def save_audit(table, name, private=False):
    """Save one audit table as CSV."""
    folder = AUDIT_DIR / ("private" if private else "aggregate")
    folder.mkdir(parents=True, exist_ok=True)
    save_safely(folder / f"{name}.csv", lambda path: table.to_csv(path, index=False))


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def tree_manifest(folder):
    """Checksum of every file below a folder."""
    files = sorted(path for path in Path(folder).rglob("*") if path.is_file() and path.name != ".DS_Store")
    return pd.DataFrame([{"file": str(p.relative_to(folder)), "bytes": p.stat().st_size, "sha256": file_sha256(p)} for p in files])


def report_missing(missing, stage):
    """Record forecasts that were expected but not found; an incomplete audit must be visible."""
    save_audit(pd.DataFrame({"forecast": missing}), f"missing_forecasts_{stage}")
    if missing:
        print(f"WARNING: {len(missing)} expected forecasts were not found, so the {stage} stage is incomplete: {missing[:5]} ...")

In [ ]:
audit_manifest = tree_manifest(V2_SOURCE) if RUN_AUDIT else None
if RUN_AUDIT:
    save_audit(audit_manifest, "v2_results_manifest")
    print(f"Version 2 results: {len(audit_manifest)} files recorded before the audit.")

## 53. Validation-Only Model Selection

Candidates are compared on 2018–2019 only. The primary score is the mean monthly **rank IC**, averaged over seeds within each
country and history, then equally over histories and countries. Raw MSE is shown as a gain over the zero forecast **within**
a country, because the return scales of countries differ too much to average raw MSE.

- **Hyperparameters** were chosen with seed 42 only; seeds 43 and 44 reuse that choice. **Epochs** were chosen separately for every
  seed. Both use the same 2018–2019 months, so every score below is optimistic, most of all for seed 42 and for the variants that
  select by rank IC (`select_ic`, `combined`, `b_ic`).
- The original `clipped_mse` is kept under its original name with its clipping bounds stated: direct transfer used *U.S. training*
  bounds, so it is **not comparable** with the other strategies. Section 58 rescores it with common bounds.
- An undefined score (a constant forecast) stays undefined and is listed, not replaced.

In [ ]:
def load_validation_rows(source):
    """Final validation rows of every variant, one per seed; direct transfer appears once per history."""
    tables = [pd.read_csv(path) for path in sorted(Path(source).glob("*/validation.csv"))
              if path.parent.name != "comparison"]
    table = pd.concat(tables, ignore_index=True)
    # Baselines have no seed stage: their selected setting is the final model
    final = (table["stage"] == "seed") | table["selected"].eq(True)
    table = table[final & (table["strategy"] != "pretrained")].copy()
    direct = table[table["window"] == "all"]
    expanded = [direct.assign(window=window) for window in ["train_1y", "train_3y", "train_5y"]]
    table = pd.concat([table[table["window"] != "all"]] + expanded, ignore_index=True)
    table["clipping_bounds"] = np.where(table["strategy"] == "direct", "USA train_full", "destination training window")
    return table


def validation_zero_mse(source):
    """Raw MSE of the zero forecast in the validation months of each country (the same for every model)."""
    table = pd.read_csv(Path(source) / "b_grid" / "validation.csv")
    return table.groupby("country")["zero_mse"].first()


def validation_cells(rows, zero_mse):
    """One row per variant, strategy, country and history: averages over seeds and seed variability."""
    groups = ["variant", "strategy", "country", "window"]
    grouped = rows.groupby(groups)
    cells = grouped.agg(
        seeds=("rank_ic", "size"), defined=("rank_ic", "count"), rank_ic=("rank_ic", "mean"), rank_ic_seed_std=("rank_ic", "std"),
        mse=("mse", "mean"), clipped_mse_original=("clipped_mse", "mean"), clipping_bounds=("clipping_bounds", "first"),
    ).reset_index()
    cells.loc[cells["defined"] < cells["seeds"], "rank_ic"] = np.nan  # undefined, never an average over fewer seeds
    tuned = rows[rows["seed"] == 42].set_index(groups)["rank_ic"].rename("rank_ic_tuned_seed")
    others = rows[rows["seed"].isin([43, 44])].groupby(groups)["rank_ic"].mean().rename("rank_ic_other_seeds")
    cells = cells.join(tuned, on=groups).join(others, on=groups)
    cells["mse_gain_pct"] = 100 * (1 - cells["mse"] / cells["country"].map(zero_mse))
    return cells


def selection_table(cells):
    """One row per candidate with the primary score under several ways of aggregating countries and histories."""
    rows = []
    for (variant, strategy), part in cells.groupby(["variant", "strategy"]):
        ic = part.set_index(["country", "window"])["rank_ic"]
        row = {"variant": variant, "strategy": strategy, "cells": len(ic), "undefined_cells": int(ic.isna().sum())}
        complete = len(ic) == 6 and ic.notna().all()
        row["mean_ic"] = ic.mean() if complete else np.nan  # equal country and history weights
        row["median_ic"] = ic.median() if complete else np.nan
        row["worst_cell_ic"] = ic.min() if complete else np.nan
        for country in ["GBR", "JPN"]:
            row[f"ic_{country}"] = ic.xs(country, level="country").mean(skipna=False)
            row[f"mse_gain_pct_{country}"] = part.loc[part["country"] == country, "mse_gain_pct"].mean()
        for window in ["train_1y", "train_3y", "train_5y"]:
            row[f"ic_{window}"] = ic.xs(window, level="window").mean(skipna=False)
        row["mean_seed_std"] = part["rank_ic_seed_std"].mean()
        row["mean_ic_tuned_seed"] = part["rank_ic_tuned_seed"].mean(skipna=False)
        row["mean_ic_other_seeds"] = part["rank_ic_other_seeds"].mean(skipna=False)
        rows.append(row)
    table = pd.DataFrame(rows)
    # Rank of every candidate within each cell, averaged: unaffected by the scale of the IC in a country
    complete = cells.merge(table.loc[table["mean_ic"].notna(), ["variant", "strategy"]], on=["variant", "strategy"])
    complete["cell_rank"] = complete.groupby(["country", "window"])["rank_ic"].rank(ascending=False)
    mean_rank = complete.groupby(["variant", "strategy"])["cell_rank"].mean().rename("mean_cell_rank")
    return table.join(mean_rank, on=["variant", "strategy"]).sort_values("mean_ic", ascending=False, ignore_index=True)


def seed_aggregates(rows):
    """Equal-country, equal-history rank IC of every candidate, separately for each seed."""
    by_seed = rows.groupby(["variant", "strategy", "seed"])["rank_ic"].agg(["mean", "size", "count"]).reset_index()
    by_seed["mean"] = by_seed["mean"].where((by_seed["size"] == 6) & (by_seed["count"] == 6))
    return by_seed.pivot(index=["variant", "strategy"], columns="seed", values="mean").add_prefix("seed_").reset_index()


def aggregation_sensitivity(table, top=5):
    """The leading candidates under each aggregation."""
    criteria = {name: False for name in ["mean_ic", "median_ic", "worst_cell_ic", "ic_GBR", "ic_JPN", "ic_train_1y",
                                         "ic_train_3y", "ic_train_5y", "mean_ic_other_seeds"]}
    criteria["mean_cell_rank"] = True  # smaller is better
    rows = []
    for name, ascending in criteria.items():
        best = table.dropna(subset=[name, "mean_ic"]).sort_values(name, ascending=ascending).head(top)
        rows += [{"criterion": name, "place": i, "variant": r["variant"], "strategy": r["strategy"], "value": r[name]}
                 for i, (_, r) in enumerate(best.iterrows(), start=1)]
    return pd.DataFrame(rows)

## 54. Statistical Testing Hierarchy

1. **Version 2 tests (unchanged).** The 9,792 saved comparisons have Holm adjustment only within one test family, country, history,
   strategy and metric. They do not control the error rate over the whole collection. `existing_test_summary` reports how many
   remain under adjustment over all of them; with 2,000 resamples no p-value is below 1/2,001, so such an adjustment cannot reject.
2. **Transfer family (new).** Direct, fine-tuned and joint models against the local model of the *same variant*, by rank IC. Holm's
   adjustment covers the variants, strategies, countries and histories listed together (column `family`).
3. **Baseline family (new).** Candidates against the baselines, also by rank IC.

The new tests were designed after the test results were known. No claim of global error control is made: adjustment covers
the stated family of each row and nothing else. Confidence intervals are pointwise.

In [ ]:
def audit_block_indices(n_months, rng, samples=AUDIT_BOOTSTRAP):
    """Month positions of the moving-block bootstrap of Section 37, with more samples."""
    n_blocks = int(np.ceil(n_months / BLOCK_MONTHS))
    starts = rng.integers(0, n_months - BLOCK_MONTHS + 1, size=(samples, n_blocks))
    return (starts[:, :, None] + np.arange(BLOCK_MONTHS)).reshape(samples, -1)[:, :n_months]


def load_monthly(source):
    """Saved monthly test results of every variant."""
    paths = sorted(Path(source).glob("*/evaluation/monthly.parquet"))
    return pd.concat([pd.read_parquet(path) for path in paths], ignore_index=True)


def existing_test_summary(tests):
    """How the saved comparisons change with the scope of the adjustment; nothing is replaced."""
    valid = tests["p_value"].dropna()
    return pd.DataFrame([
        {"quantity": "comparisons saved", "value": len(tests)},
        {"quantity": "comparisons with a defined p-value", "value": len(valid)},
        {"quantity": "smallest p-value", "value": valid.min()},
        {"quantity": "unadjusted p < 0.05", "value": int((valid < 0.05).sum())},
        {"quantity": "Holm within the original groups, p < 0.05", "value": int((tests["p_adjusted"] < 0.05).sum())},
        {"quantity": "Holm over all comparisons, p < 0.05", "value": int((holm_adjust(tests["p_value"]) < 0.05).sum())},
    ])


def paired_ic_tests(monthly, pairs, family):
    """Rank-IC tests of (variant, strategy) pairs in every country and history, with Holm over the whole family."""
    base = monthly[(monthly["scenario"] == "base") & (monthly["seed"] == "main")]
    series = {names: part.sort_values("predicted_month")
              for names, part in base.groupby(["variant", "strategy", "country", "window"])}
    months = base["predicted_month"].nunique()
    indices = audit_block_indices(months, np.random.default_rng(SEED))
    rows = []
    for new, old in pairs:
        for country, window in product(LOCAL_COUNTRIES, TRAIN_WINDOWS["GBR"]):
            a, b = series.get((*new, country, window)), series.get((*old, country, window))
            if a is None or b is None:
                continue
            assert a["predicted_month"].tolist() == b["predicted_month"].tolist() and len(a) == months
            test = bootstrap_difference(a["rank_ic"].to_numpy(), b["rank_ic"].to_numpy(), 1, np.mean, indices)
            rows.append({"family": family, "variant": new[0], "strategy": new[1], "against_variant": old[0],
                         "against_strategy": old[1], "country": country, "window": window, "metric": "Rank IC", **test})
    table = pd.DataFrame(rows)
    table["p_holm_family"] = holm_adjust(table["p_value"])
    table["family_size"] = int(table["p_value"].notna().sum())
    return table


def transfer_pairs(variants):
    return [((v, s), (v, "localmatched")) for v in variants for s in ["direct", "finetuned", "joint"]]

## 55. Portfolio Sensitivity from Saved Monthly Results

These tables restate saved Version 2 results. **Capped returns are a diagnostic of outlier influence, not realized returns** of any
investable strategy. The influence table drops the month with the largest absolute return of each series to show how much of the
result depends on it; the full series remains the result.

Cumulative P&L is the **sum** of monthly returns per $1 long and $1 short. A compounded drawdown needs a capital base: the
table states wealth per $1 of capital with exposure of $1 (`capital 1`) or $0.50 (`capital 2`) per side, constant exposure
rebalanced monthly, no interest on capital, and bankruptcy (wealth stays zero) once losses reach the capital.

In [ ]:
def portfolio_sensitivity(monthly, candidates):
    """Sharpe ratios of the saved scenarios, kept separate by country and history."""
    rows = []
    main = monthly[monthly["seed"] == "main"]
    for (variant, strategy, country, window, scenario), part in main.groupby(
            ["variant", "strategy", "country", "window", "scenario"]):
        if (variant, strategy) not in candidates:
            continue
        part = part.sort_values("predicted_month")
        net, gross = part["net_return"].to_numpy(), part["gross_return"].to_numpy()
        worst = int(np.abs(gross).argmax())
        keep = np.arange(len(net)) != worst
        rows.append({
            "variant": variant, "strategy": strategy, "country": country, "window": window, "scenario": scenario,
            "type": "ex-post diagnostic, not investable" if scenario == "returns_capped" else "sensitivity",
            "gross_sharpe": sharpe_ratio(gross), "net_sharpe": sharpe_ratio(net), "net_mean_monthly": net.mean(),
            "additive_net_pnl": net.sum(), "mean_turnover": part["turnover"].mean(), "stocks_per_side": part["n_side"].mean(),
            "largest_abs_month": part["predicted_month"].iloc[worst], "largest_abs_gross": gross[worst],
            "net_sharpe_without_that_month": sharpe_ratio(net[keep]),
            "compounded_drawdown_capital_1": compounded_drawdown(part["net_return"], 1.0),
            "compounded_drawdown_capital_2": compounded_drawdown(part["net_return"], 0.5),
            "bankrupt_capital_1": bool((net <= -1).any()), "bankrupt_capital_2": bool((0.5 * net <= -1).any()),
        })
    return pd.DataFrame(rows)


def run_reports(source):
    """All tables that need only saved Version 2 results."""
    source = Path(source)
    rows = load_validation_rows(source)
    cells = validation_cells(rows, validation_zero_mse(source))
    selection = selection_table(cells)
    save_audit(cells, "selection_cells")
    save_audit(selection, "selection_candidates")
    save_audit(seed_aggregates(rows), "selection_by_seed")
    save_audit(aggregation_sensitivity(selection), "selection_sensitivity")

    monthly = load_monthly(source)
    save_audit(existing_test_summary(pd.read_csv(source / "comparison" / "tests.csv")), "existing_tests_scope")
    save_audit(paired_ic_tests(monthly, transfer_pairs(["rank"]), "transfer, rank variant"), "transfer_tests_rank")
    save_audit(paired_ic_tests(
        monthly, transfer_pairs([v for v in VARIANTS if v not in ["joint50", "joint75"]]), "transfer, all variants"),
        "transfer_tests_all")
    baselines = [("b_huber", "xgboost"), ("b_clip", "elastic_net")]
    chosen = [c for c in AUDIT_CANDIDATES if c not in baselines and c[0] != "robust"]
    save_audit(paired_ic_tests(
        monthly, list(product(chosen + [("rank", "ensemble_equal")], baselines)), "candidates against baselines"),
        "baseline_tests")
    sensitivity = portfolio_sensitivity(monthly, AUDIT_CANDIDATES + [("rank", "ensemble_equal")])
    save_audit(sensitivity, "portfolio_sensitivity")
    return selection, sensitivity

In [ ]:
if audit_stage("reports"):
    selection, sensitivity = run_reports(V2_SOURCE)
    display(selection.head(15))
    print("Net Sharpe by scenario (exploratory; averages over histories):")
    display(sensitivity.pivot_table(index=["country", "variant", "strategy"], columns="scenario", values="net_sharpe"))

## 56. Data-Quality Audit of Returns

Nothing is removed or changed. **Concentration of squared returns** (how much a few returns dominate the loss of a zero forecast) is
separate from **concentration of squared errors** of a model; both are reported, pooled and with the equal month weights used
by the reported MSE. The Version 2 column `largest` in `outliers.csv` is the largest *positive* return only; here
the largest positive, negative and absolute returns are named separately.

The files contain no delisting return, local-currency return, exchange rate or price-adjustment factor, so these
causes cannot be confirmed here. The private table of extremes gives evidence instead: the target against the next `ret_1_0`,
the change in market equity and price over the same month, the largest daily return of that month, and the returns before and after.

In [ ]:
def return_profile(returns):
    """Description of one set of returns; all named quantities are explicit about sign."""
    values = returns.dropna()
    squared = values ** 2
    quantiles = values.quantile([0.001, 0.01, 0.5, 0.99, 0.999]).to_numpy()
    return {
        "stock_months": len(returns), "missing": int(returns.isna().sum()), "largest_positive": values.max(),
        "largest_negative": values.min(), "largest_absolute": values.abs().max(), "above_plus_100pct": int((values > 1).sum()),
        "below_minus_100pct": int((values < -1).sum()), "at_or_below_minus_99pct": int((values <= -0.99).sum()),
        "above_plus_300pct": int((values > 3).sum()),
        **dict(zip(["q001", "q01", "median", "q99", "q999"], quantiles)), "mean": values.mean(), "std": values.std(),
        "top10_share_of_squared_returns": squared.nlargest(10).sum() / squared.sum(),
    }


def return_profiles(frames, by_size=False):
    """Profiles by country and period, optionally by size group at portfolio formation."""
    rows = []
    for country, df in frames.items():
        for period in TRAIN_WINDOWS[country] + ["validation", "test"]:
            part = df[in_period(df["predicted_month"], period)]
            groups = part.groupby(part["size_grp"].astype("string").fillna("unknown")) if by_size else [("all", part)]
            total = (part[TARGET] ** 2).sum()
            for name, group in groups:
                rows.append({"country": country, "period": period, "size_group": name, **return_profile(group[TARGET]),
                             "share_of_squared_returns": (group[TARGET] ** 2).sum() / total})
    return pd.DataFrame(rows)


def squared_return_concentration(df):
    """Share of squared returns by month and by stock, pooled and with equal month weights."""
    part = df.dropna(subset=[TARGET])
    squared = part[TARGET] ** 2
    n_month = part.groupby("predicted_month")["id"].transform("size")
    equal = squared / (n_month * part["predicted_month"].nunique())  # adds up to the zero-forecast MSE
    by_month = pd.DataFrame({
        "stock_months": squared.groupby(part["predicted_month"]).size(),
        "pooled_share": squared.groupby(part["predicted_month"]).sum() / squared.sum(),
        "share_of_zero_forecast_mse": equal.groupby(part["predicted_month"]).sum() / equal.sum(),
    }).reset_index()
    by_stock = pd.DataFrame({
        "stock_months": squared.groupby(part["id"]).size(),
        "pooled_share": squared.groupby(part["id"]).sum() / squared.sum(),
        "share_of_zero_forecast_mse": equal.groupby(part["id"]).sum() / equal.sum(),
    }).reset_index()
    return by_month, by_stock.sort_values("pooled_share", ascending=False)


def error_concentration(stocks, label):
    """How much of one model's MSE comes from few stock-months; the MSE itself is unchanged."""
    part = stocks.dropna(subset=["actual_return"])
    error = (part["actual_return"] - part["prediction"]) ** 2
    zero = part["actual_return"] ** 2
    n_month = part.groupby("predicted_month")["id"].transform("size")
    weight = 1 / (n_month * part["predicted_month"].nunique())
    contribution = error * weight
    by_month = contribution.groupby(part["predicted_month"]).sum()
    largest = contribution.nlargest(10).index
    others = ~part.index.isin(largest)

    def remaining_r2():
        kept = part[others]
        model = error[others].groupby(kept["predicted_month"]).mean().mean()
        return 1 - model / zero[others].groupby(kept["predicted_month"]).mean().mean()

    return {
        **label, "mse": contribution.sum(), "oos_r2": 1 - contribution.sum() / (zero * weight).sum(),
        "top10_share_of_squared_errors_pooled": error.nlargest(10).sum() / error.sum(),
        "top10_share_of_mse": contribution.loc[largest].sum() / contribution.sum(),
        "top_month": by_month.idxmax(), "top_month_share_of_mse": by_month.max() / contribution.sum(),
        "oos_r2_without_top10_diagnostic": remaining_r2(),
    }

In [ ]:
def add_neighbors(df):
    """Previous and next values of each stock, only where the months are consecutive."""
    df = df.sort_values(["id", "eom"], ignore_index=True)
    by_stock = df.groupby("id")
    month = pd.offsets.MonthEnd(1)
    has_next = by_stock["eom"].shift(-1) == df["eom"] + month
    has_previous = by_stock["eom"].shift(1) == df["eom"] - month
    df["next_month_observed"] = has_next
    for name in ["ret_1_0", "me", "prc", "rmax1_21d", TARGET]:
        df[f"{name}_next"] = by_stock[name].shift(-1).where(has_next)
    df["previous_month_observed"] = has_previous
    for name in ["gvkey", "permno"]:
        same = by_stock[name].shift(-1).astype("string") == df[name].astype("string")
        df[f"{name}_same_next"] = same.where(has_next & df[name].notna())
    return df


def alignment_checks(frames):
    """Aggregate consistency of the target with the next month's return, and of gaps and missing targets."""
    rows = []
    for country, df in frames.items():
        difference = df["ret_1_0_next"] - df[TARGET]  # should equal a small T-bill rate
        both = difference.notna()
        last_month = df["eom"] == df["eom"].max()
        last_row = ~df["id"].duplicated(keep="last")
        rows.append({
            "country": country, "pairs": int(both.sum()), "target_is_next_return_minus_small_rate":
                int(difference[both].between(-0.0002, 0.01).sum()),
            "mismatches": int((~difference[both].between(-0.0002, 0.01)).sum()),
            "extreme_target_pairs": int((both & (df[TARGET] > 1)).sum()),
            "extreme_target_mismatches": int((both & (df[TARGET] > 1) & ~difference.between(-0.0002, 0.01)).sum()),
            "target_without_next_month_row": int((df[TARGET].notna() & ~df["next_month_observed"]).sum()),
            "missing_target": int(df[TARGET].isna().sum()),
            "missing_target_final_data_month": int((df[TARGET].isna() & last_month).sum()),
            "missing_target_last_row_of_stock": int((df[TARGET].isna() & last_row & ~last_month).sum()),
            "missing_target_but_next_return_observed": int((df[TARGET].isna() & df["ret_1_0_next"].notna()).sum()),
            "last_rows_before_final_month": int((last_row & ~last_month).sum()),
            "last_rows_with_target": int((last_row & ~last_month & df[TARGET].notna()).sum()),
        })
    return pd.DataFrame(rows)


def extreme_records(df, top=100):
    """PRIVATE stock-level evidence for the largest absolute targets of one country."""
    part = df.loc[df[TARGET].abs().nlargest(top).index].copy()
    growth = 1 + part["ret_1_0_next"]
    part["me_growth"] = part["me_next"] / part["me"]
    part["price_growth"] = part["prc_next"].abs() / part["prc"].abs()
    part["me_growth_matches_return"] = np.where(part["me_growth"].isna(), pd.NA, np.abs(np.log(part["me_growth"] / growth)) < 0.5)
    part["price_growth_matches_return"] = np.where(
        part["price_growth"].isna(), pd.NA, np.abs(np.log(part["price_growth"] / growth)) < 0.5)
    part["mostly_one_day"] = part["rmax1_21d_next"] > 0.5 * part["ret_1_0_next"]
    part["reversal_next_month"] = part[f"{TARGET}_next"] < -0.5
    part["penny_stock_before"] = part["prc"].abs() < 1
    return part


def extreme_summary(records):
    """Counts of the evidence, without identifiers or stock-level values."""
    rows = []
    for (country, period), part in records.groupby(["country", "period"]):
        evidence = {name: int(part[name].eq(True).sum()) for name in [
            "next_month_observed", "previous_month_observed", "me_growth_matches_return", "price_growth_matches_return",
            "mostly_one_day", "reversal_next_month", "penny_stock_before", "gvkey_same_next"]}
        known = {f"{name}_known": int(part[name].notna().sum())
                 for name in ["me_growth_matches_return", "price_growth_matches_return", "gvkey_same_next"]}
        sizes = part["size_grp"].astype("string").fillna("unknown").value_counts().add_prefix("size_").to_dict()
        rows.append({"country": country, "period": period, "records": len(part), **evidence, **known, **sizes})
    return pd.DataFrame(rows).fillna(0)


def load_audit_data():
    """Returns with the price fields needed for the audit, from the same files as the models."""
    frames = {}
    for country, path in FILES.items():
        df = pd.read_parquet(path, columns=["id", "eom", "gvkey", "permno", "size_grp", "me", "prc", "rmax1_21d", "ret_1_0", TARGET])
        df["eom"] = pd.to_datetime(df["eom"])
        df["predicted_month"] = df["eom"] + pd.offsets.MonthEnd(1)
        frames[country] = add_neighbors(df)
    return frames


def run_data_audit():
    frames = load_audit_data()
    save_audit(return_profiles(frames), "return_profiles")
    save_audit(return_profiles(frames, by_size=True), "return_profiles_by_size")
    save_audit(alignment_checks(frames), "target_alignment")
    records, months = [], []
    for country, df in frames.items():
        for period in TRAIN_WINDOWS[country] + ["validation", "test"]:
            part = df[in_period(df["predicted_month"], period)]
            by_month, by_stock = squared_return_concentration(part)
            months.append(by_month.assign(country=country, period=period))
            save_audit(by_stock.head(100), f"stock_concentration_{country}_{period}", private=True)
            records.append(extreme_records(part).assign(country=country, period=period))
    records = pd.concat(records, ignore_index=True)
    save_audit(pd.concat(months, ignore_index=True), "month_concentration")
    save_audit(records, "extreme_records", private=True)
    save_audit(extreme_summary(records), "extreme_evidence_summary")

    rows, missing = [], []
    for variant, strategy in AUDIT_CANDIDATES:
        for country, window in product(LOCAL_COUNTRIES, TRAIN_WINDOWS["GBR"]):
            table = read_source_forecast(variant, strategy, country, window, "test")
            if table is None:
                missing.append(f"{variant}/{strategy}/{country}/{window}")
                continue
            rows.append(error_concentration(
                add_actuals(table, country), {"variant": variant, "strategy": strategy, "country": country, "window": window}))
    report_missing(missing, "data")
    save_audit(pd.DataFrame(rows), "squared_error_concentration")

In [ ]:
def read_source_forecast(variant, strategy, country, window, period):
    """Saved forecast of the completed Version 2 run, opened read-only; None if it does not exist."""
    name = f"{country}_{'all' if strategy == 'direct' else window}_{strategy}.parquet"
    path = V2_SOURCE / variant / "predictions" / name
    if not path.exists():
        return None
    table = pd.read_parquet(path)
    if table["sources"].nunique() != 1:
        raise ValueError(f"{path}: forecasts of several model versions are mixed.")
    if table.duplicated(KEYS).any() or not np.isfinite(table["prediction"]).all():
        raise ValueError(f"{path}: duplicate keys or undefined forecasts.")
    return table[table["period"] == period].sort_values(["eom", "id"], ignore_index=True)


if audit_stage("data"):
    run_data_audit()
    print("Data-quality audit saved. Stock-level records are in the private folder.")

## 57. Portfolio Accounting with Missing Returns

The Version 2 portfolio chooses stocks without knowing future returns (no look-ahead in selection), but a side's return is the
mean over the stocks whose return was later **observed**. That silently reassigns the weight of the others after the fact. These
results are kept as `legacy`. The intended weights are now fixed at formation, and each month is reported as:

| Accounting | Return of a selected stock without an observed return |
|---|---|
| `legacy` | weight redistributed to observed stocks (Version 2) |
| `exact` | none needed; the month is undefined if any exposure is unobserved |
| `assume_zero` | 0% excess return (assumption) |
| `assume_minus30` | −30%, a common imputation for performance-related delistings, applied to all (assumption) |
| `assume_adverse` | −30% on longs and +30% on shorts: a pessimistic bound (assumption) |

No stock is dropped at formation because of its later label. Delisting returns are not in the files, so the last three are
assumptions, not measurements. Turnover is reported without drift (`static`, the Version 2 rule) and from drifted positions
(`drift`), where last month's weights grow with the return assumed by that accounting; costs follow. For `exact`, a position
without an observed return keeps its weight before the next trade, so its drift turnover is approximate.

In [ ]:
ACCOUNTING = {"assume_zero": (0.0, 0.0), "assume_minus30": (-0.30, -0.30), "assume_adverse": (-0.30, 0.30)}  # (long, short)


def weighted_sides(universe, decile, weighting):
    """Intended signed weights of one month: each side adds up to one, known at formation."""
    weights = long_short_weights(universe, decile)  # the Version 2 selection
    if weights.empty or weighting == "equal":
        return weights
    size = universe.set_index("id")["me"].reindex(weights.index).astype("float64")
    size = size.where(size > 0, 0.0).fillna(0.0)
    if weighting == "value_capped":  # cap at the 80th percentile of the formation universe
        size = size.clip(upper=universe["me"].quantile(0.80))
    elif weighting != "value":
        raise ValueError(f"Unknown weighting: {weighting}")
    result = weights.copy()
    for side in [weights > 0, weights < 0]:
        if size[side].sum() > 0:
            result[side] = np.sign(weights[side]) * size[side] / size[side].sum()
    return result


def month_accounting(universe, weights):
    """Returns of intended weights under every accounting, and the exposure without observed returns."""
    returns = universe.set_index("id")["actual_return"].reindex(weights.index)
    observed, longs = returns.notna(), weights > 0
    row = {
        "n_side": int(longs.sum()), "long_unobserved": weights[longs & ~observed].sum(),
        "short_unobserved": -weights[~longs & ~observed].sum(), "exactly_measured": bool(observed.all()),
    }
    if weights.empty:
        return {**row, "legacy": 0.0, "exact": 0.0, **dict.fromkeys(ACCOUNTING, 0.0)}, dict.fromkeys(["legacy", "exact", *ACCOUNTING], returns)
    contribution = (weights * returns).where(observed)
    sides = {}
    for name, side in [("long", longs), ("short", ~longs)]:
        exposure = weights[side & observed].abs().sum()
        sides[name] = contribution[side].sum() / exposure if exposure > 0 else np.nan
    row["legacy"] = sides["long"] + sides["short"]
    row["exact"] = contribution.sum() if row["exactly_measured"] else np.nan
    filled = {"exact": returns.fillna(0.0)}
    for name, (long_value, short_value) in ACCOUNTING.items():
        filled[name] = returns.where(observed, pd.Series(np.where(longs, long_value, short_value), index=weights.index))
        row[name] = (weights * filled[name]).sum()
    filled["legacy"] = returns.fillna(0.0)
    return row, filled


def account_portfolio(stocks, decile=DECILE, weighting="equal", exclude=()):
    """Monthly accounting of one forecast; turnover is the traded notional per $1 long and $1 short."""
    if exclude:
        stocks = stocks[~stocks["size_grp"].isin(exclude).to_numpy()]
    empty = pd.Series(dtype="float64", index=pd.Index([], dtype="int64"))
    names = ["legacy", "exact", *ACCOUNTING]
    previous, drifted = empty, dict.fromkeys(names, empty)
    rows = []
    for month, universe in stocks.groupby("predicted_month"):
        weights = weighted_sides(universe, decile, weighting)
        row, filled = month_accounting(universe, weights)
        row = {"predicted_month": month, **row, "turnover_static": traded_notional(weights, previous)}
        for name in names:
            row[f"turnover_drift_{name}"] = traded_notional(weights, drifted[name])
            drifted[name] = weights * (1 + filled[name])
        rows.append(row)
        previous = weights
    return pd.DataFrame(rows)


def accounting_summary(months, cost_bps):
    """Net results under every accounting and both turnover rules; undefined months are counted, not filled."""
    rows = []
    for name, rule in product(["legacy", "exact", *ACCOUNTING], ["static", "drift"]):
        turnover = months["turnover_static" if rule == "static" else f"turnover_drift_{name}"]
        net = (months[name] - turnover * cost_bps / 10_000).to_numpy()
        valid = np.isfinite(net)
        rows.append({
            "accounting": name, "turnover_rule": rule, "cost_bps": cost_bps, "months": len(net), "months_measured": int(valid.sum()),
            "mean_turnover": turnover.mean(), "net_mean_monthly": net[valid].mean() if valid.any() else np.nan,
            "net_sharpe": sharpe_ratio(net[valid]) if valid.sum() > 1 else np.nan,
            "additive_net_pnl": net[valid].sum(), "worst_month": net[valid].min() if valid.any() else np.nan,
            "note": "undefined months left out; not a 72-month result" if not valid.all() else "",
        })
    table = pd.DataFrame(rows)
    return table.assign(
        months_with_unobserved=int((~months["exactly_measured"]).sum()), mean_long_unobserved=months["long_unobserved"].mean(),
        mean_short_unobserved=months["short_unobserved"].mean(), max_long_unobserved=months["long_unobserved"].max(),
        max_short_unobserved=months["short_unobserved"].max(),
    )


AUDIT_UNIVERSES = {"full": (), "no_micro_nano": ("micro", "nano")}


def run_portfolio_audit():
    tables, monthly_rows, missing = [], [], []
    for country in LOCAL_COUNTRIES:
        size = data[country][KEYS[1:] + ["me"]]  # market equity at formation
        for (variant, strategy), window in product(AUDIT_CANDIDATES, TRAIN_WINDOWS[country]):
            label = {"variant": variant, "strategy": strategy, "country": country, "window": window}
            forecast = read_source_forecast(variant, strategy, country, window, "test")
            if forecast is None:
                missing.append("/".join(label.values()))
                continue
            stocks = add_actuals(forecast, country).merge(size, on=KEYS[1:], how="left", validate="one_to_one")
            assert stocks["predicted_month"].nunique() == expected_months["test"]
            print(f"Accounting: {' | '.join(label.values())}", flush=True)
            for (universe, exclude), quantile, weighting in product(
                    AUDIT_UNIVERSES.items(), [0.10, 0.20, 0.30], ["equal", "value", "value_capped"]):
                months = account_portfolio(stocks, quantile, weighting, exclude)
                setting = {**label, "universe": universe, "quantile": quantile, "weighting": weighting}
                monthly_rows.append(months.assign(**setting))
                tables += [accounting_summary(months, cost).assign(**setting) for cost in [10, 25]]
                if weighting == "equal" and universe == "full":  # the legacy numbers must be reproduced exactly
                    saved = evaluate_forecast(stocks, quantile).set_index("predicted_month")
                    np.testing.assert_allclose(months["legacy"], saved["gross_return"], atol=1e-12)
                    np.testing.assert_allclose(months["turnover_static"], saved["turnover"], atol=1e-12)
    report_missing(missing, "portfolio")
    save_audit(pd.concat(tables, ignore_index=True), "portfolio_accounting")
    save_audit(pd.concat(monthly_rows, ignore_index=True), "portfolio_accounting_monthly")

In [ ]:
if audit_stage("portfolio"):
    run_portfolio_audit()

## 58. Validation Scores with Common Clipping Bounds

Policy `destination`: for one country and local history, every strategy (including direct transfer) is scored with the bounds of
**that country's local training window**. Actual returns are unmodified for raw MSE and rank IC. Scores are recomputed from the
saved validation forecasts without retraining. The original scores stay in separate columns, and the table records how closely
raw MSE and rank IC are reproduced (small differences come from GPU and 32-bit arithmetic).

`seed` rows score single models; the `average` row scores the seed-average forecast that the test evaluation uses. Monthly
rank ICs (24 validation months) are saved for paired comparisons of the candidates.

In [ ]:
def rescore(table, limits):
    """Scores of every forecast column of one validation table with the given clipping bounds."""
    labeled = table[table["actual_return"].notna()]
    columns = [c for c in labeled.columns if c.startswith("seed_")] + ["prediction"]
    rows = []
    for column in columns:
        lists = monthly_lists(labeled, labeled[column].to_numpy(dtype=float), labeled["actual_return"].to_numpy(dtype=float))
        rows.append({"model": "average" if column == "prediction" else column, **score_forecasts(*lists, limits),
                     "lower_bound": limits[0], "upper_bound": limits[1]})
    return pd.DataFrame(rows)


def monthly_rank_ic(table):
    """Monthly validation rank IC of the final forecast."""
    labeled = table[table["actual_return"].notna()]
    return pd.DataFrame([{"predicted_month": month, "rank_ic": rank_ic(part)} for month, part in labeled.groupby("predicted_month")])


def validation_strategies(variant):
    return ALL_VARIANTS[variant][2]


def run_validation_audit():
    original = load_validation_rows(V2_SOURCE)
    rows, series, missing = [], [], []
    for variant in ALL_VARIANTS:
        for strategy, country, window in product(validation_strategies(variant), LOCAL_COUNTRIES, TRAIN_WINDOWS["GBR"]):
            table = read_source_forecast(variant, strategy, country, window, "validation")
            label = {"variant": variant, "strategy": strategy, "country": country, "window": window}
            if table is None:
                missing.append("/".join(label.values()))
                continue
            stocks = add_actuals(table, country)
            rows.append(rescore(stocks, clip_limits(country, window)).assign(**label))
            series.append(monthly_rank_ic(stocks).assign(**label))
    report_missing(missing, "validation")
    scores = pd.concat(rows, ignore_index=True).rename(columns={"clipped_mse": "clipped_mse_destination_bounds"})
    names = ["variant", "strategy", "country", "window", "model"]
    original["model"] = np.where(original["stage"] == "seed", "seed_" + original["seed"].astype("Int64").astype("string"), "average")
    original = original[names + ["mse", "clipped_mse", "rank_ic", "clipping_bounds"]].rename(columns={
        "mse": "mse_original", "clipped_mse": "clipped_mse_original", "rank_ic": "rank_ic_original",
        "clipping_bounds": "clipping_bounds_original"})
    scores = scores.merge(original, on=names, how="left", validate="one_to_one")
    scores["mse_relative_difference"] = scores["mse"] / scores["mse_original"] - 1
    scores["rank_ic_difference"] = scores["rank_ic"] - scores["rank_ic_original"]
    save_audit(scores, "validation_rescored")
    save_audit(pd.concat(series, ignore_index=True), "validation_monthly_ic")

    reproduced = scores.dropna(subset=["mse_original"])
    print(f"Rescored {len(scores)} forecasts; {len(reproduced)} have an original score. Largest relative MSE difference "
          f"{reproduced['mse_relative_difference'].abs().max():.2e}, rank IC {reproduced['rank_ic_difference'].abs().max():.2e}")
    save_audit(validation_shortlist_tests(pd.concat(series, ignore_index=True)), "validation_paired_tests")


def validation_shortlist_tests(series):
    """Paired tests of every complete candidate against the best one, on the average IC of the six cells by month."""
    wide = series.pivot_table(index="predicted_month", columns=["variant", "strategy", "country", "window"], values="rank_ic",
                              dropna=False)
    candidates = {}
    for name in dict.fromkeys((v, s) for v, s, _, _ in wide.columns):
        part = wide[name]
        if part.shape[1] == 6 and part.notna().all().all():
            candidates[name] = part.mean(axis=1).to_numpy()
    best = max(candidates, key=lambda name: candidates[name].mean())
    indices = audit_block_indices(len(wide), np.random.default_rng(SEED))
    rows = [{"variant": name[0], "strategy": name[1], "mean_ic": values.mean(), "best_variant": best[0], "best_strategy": best[1],
             **({} if name == best else bootstrap_difference(candidates[best], values, 1, np.mean, indices))}
            for name, values in candidates.items()]
    table = pd.DataFrame(rows).rename(columns={"improvement": "best_minus_candidate"})
    table["p_holm"] = holm_adjust(table["p_value"])
    return table.sort_values("mean_ic", ascending=False, ignore_index=True)

In [ ]:
if audit_stage("validation"):
    run_validation_audit()

## 59. Provenance Record

The record separates the **evaluation-code revision** (this branch) from the **trained-model revision**. The 24 saved configurations
say `git_commit: unknown`; they are left as they are. The verified facts are that the notebook in the review package equals commit
`69a628c`, and that the settings in every configuration reproduce its fingerprint.

In [ ]:
def verify_fingerprints(source):
    """Recompute the fingerprint of every saved configuration from its own contents."""
    rows = []
    for path in sorted(Path(source).glob("*/config.json")):
        config = json.loads(path.read_text())
        environment, saved = config.pop("environment"), config.pop("fingerprint")
        text = json.dumps(config, sort_keys=True)
        rows.append({
            "variant": config["variant"], "fingerprint": saved,
            "reproduced": hashlib.sha256(text.encode()).hexdigest()[:16] == saved,
            "code_version": config["code_version"], "recorded_git_commit": environment["git_commit"],
            "slurm_job": environment["slurm_job"], "features": len(config["features"]),
            "features_sha256": feature_hash(config["features"]), "selection": config["spec"]["selection"],
            "seeds": "|".join(map(str, config.get("seeds", [config.get("seed")]))),
        })
    return pd.DataFrame(rows)


if RUN_AUDIT:
    fingerprints = verify_fingerprints(V2_SOURCE)
    save_audit(fingerprints, "v2_fingerprints")
    assert fingerprints["reproduced"].all(), "A saved configuration does not match its fingerprint."
    after = tree_manifest(V2_SOURCE)
    assert after.equals(audit_manifest), "The Version 2 results changed during the audit."
    record = {
        "audit_code_version": AUDIT_CODE_VERSION, "evaluation_environment": environment_details(),
        "stages": AUDIT_STAGES, "trained_model_code_version": V2_CODE_VERSION,
        "trained_model_commit_recorded_by_jobs": sorted(set(fingerprints["recorded_git_commit"])),
        "trained_model_commit_verified_from_review_package": "69a628c08888b375621b02aca3c10e2070d20b9a",
        "data_sha256": {country: file_sha256(path) for country, path in FILES.items()},
        "periods": {name: [str(date.date()) for date in dates] for name, dates in PERIODS.items()},
        "v2_results_files": len(audit_manifest), "v2_results_unchanged": True,
    }
    save_safely(AUDIT_DIR / "provenance.json", lambda path: path.write_text(json.dumps(record, indent=2)))
    print("Version 2 results unchanged; provenance recorded.")

## 60. Audit Verification

The calculations of this phase are tested on artificial data by `tests/test_audit.py`, which loads these functions from the notebook
without running it (no licensed data, training or GPU): `python tests/test_audit.py`.

Phase 9 — End